In [ ]:
!pip install transformers datasets accelerate
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from google.colab import userdata
import os

# Retrieve the HF_TOKEN from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Set it as an environment variable
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN loaded from Colab secrets and set as environment variable.")
# Now, when load_dataset is called, it will automatically use this token for authentication.

HF_TOKEN loaded from Colab secrets and set as environment variable.


In [ ]:
# ============================================================
# Phase 1: CinePile Text-Only Evaluation — Final Fix
# answer_key 是答案全文，需与 choices 匹配转为 A~E
# ============================================================

import gc
import torch
import pandas as pd
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModel,
    BitsAndBytesConfig
)
from tqdm import tqdm


# ============================================================
# 1. Configure Class
# ============================================================
@dataclass
class ExperimentConfig:
    dataset_name: str           = 'tomg-group-umd/cinepile'
    dataset_split: str          = 'test'
    max_samples: Optional[int]  = None      # 调试时设 200

    max_scene_length: int       = 1500
    max_input_tokens: int       = 2048
    deberta_max_tokens: int     = 512

    max_new_tokens: int         = 5
    do_sample: bool             = False

    load_in_4bit: bool          = True
    bnb_4bit_quant_type: str    = "nf4"
    bnb_4bit_compute_dtype: torch.dtype = torch.bfloat16

    models: list = field(default_factory=lambda: [
        ('DeBERTa-v3-base', 'microsoft/deberta-v3-base',          'deberta'),
        ('Llama-3.1-8B',    'meta-llama/Llama-3.1-8B-Instruct',  'llm'),
        ('Mistral-7B',      'mistralai/Mistral-7B-Instruct-v0.3', 'llm'),
    ])

    output_csv: str = 'phase1_results.csv'

    category_map: dict = field(default_factory=lambda: {
        'TEMP': 'Temporal',
        'CRD':  'Character and',        # map 'Character and\nRelationship Dynamics'
        'NPA':  'Narrative and',        # map 'Narrative and\nPlot Analysis'
        'STA':  'Setting and',          # map 'Setting and\nTechnical Analysis'
        'TH':   'Theme Exploration',    # map 'Theme Exploration'
    })



# ============================================================
# 2. Prepare dataset
# ============================================================
class CinePileDataset:
    """
    CinePile 的 answer_key 是答案全文字符串。
    通过与 choices 列表匹配，转换为字母索引 'A'~'E'。
    """
    LETTERS = ['A', 'B', 'C', 'D', 'E']

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.data   = self._load()

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        """
        将答案全文映射到对应字母。
        先做精确匹配，再做 strip 后的模糊匹配，
        均失败时返回 'A' 并打印警告。
        """
        # 精确匹配
        for i, choice in enumerate(choices):
            if answer_text == choice:
                return cls.LETTERS[i]

        # strip 后匹配（防止空格差异）
        answer_stripped = answer_text.strip()
        for i, choice in enumerate(choices):
            if answer_stripped == choice.strip():
                return cls.LETTERS[i]

        # 包含匹配（防止部分截断）
        for i, choice in enumerate(choices):
            if answer_stripped in choice or choice.strip() in answer_stripped:
                return cls.LETTERS[i]

        print(f"  [Warning] Cannot match answer_key to choices.")
        print(f"    answer_key : {repr(answer_text)}")
        print(f"    choices    : {choices}")
        return 'A'  # fallback

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == 'true'

    def _load(self):
        raw = load_dataset(
            self.config.dataset_name,
            split=self.config.dataset_split
        )
        samples = []
        unmatched = 0

        for ex in raw:
            choices    = ex['choices']
            answer_key = ex['answer_key']

            # 核心修复：answer_key 全文 → 字母
            letter = self._answer_text_to_letter(answer_key, choices)
            if letter == 'A' and answer_key != choices[0]:
                unmatched += 1

            samples.append({
                'movie_scene':       ex['movie_scene'],
                'question':          ex['question'],
                'choices':           choices,
                'answer_key':        letter,           # 统一为 'A'~'E'
                'answer_text':       answer_key,       # 保留原始全文（调试用）
                'question_category': ex['question_category'],
                'hard_split':        self._normalize_hard_split(ex['hard_split']),
            })

        if self.config.max_samples:
            samples = samples[:self.config.max_samples]

        print(f"[Dataset] Loaded {len(samples)} samples")
        print(f"          answer_key sample : "
              f"{repr(samples[0]['answer_text'])} → {samples[0]['answer_key']}")
        print(f"          hard_split sample : {samples[0]['hard_split']}")
        if unmatched > 0:
            print(f"  [Warning] {unmatched} samples fell back to 'A' "
                  f"(answer not matched in choices)")
        return samples

    def run_sanity_check(self, n: int = 5):
        """快速验证前 n 个样本的 answer_key 转换是否正确"""
        print("\n=== Sanity Check ===")
        for i, s in enumerate(self.data[:n]):
            idx = ord(s['answer_key']) - 65
            matched_text = s['choices'][idx]
            match_ok = matched_text.strip() == s['answer_text'].strip()
            print(f"  [{i}] {s['answer_key']} → \"{matched_text}\"")
            print(f"       original: \"{s['answer_text']}\"  match={match_ok}")
        print()


# ============================================================
# 3. Base Evaluator
# ============================================================
class BaseEvaluator(ABC):
    def __init__(self, model_id: str, config: ExperimentConfig):
        self.model_id  = model_id
        self.config    = config
        self.model     = None
        self.tokenizer = None

    @abstractmethod
    def load_model(self): pass

    @abstractmethod
    def predict(self, sample: dict) -> str: pass

    def evaluate(self, data: list) -> pd.DataFrame:
        records = []
        for sample in tqdm(data, desc=f"  {self.model_id}"):
            pred = self.predict(sample)
            records.append({
                'pred':              pred,
                'label':             sample['answer_key'],
                'question_category': sample['question_category'],
                'hard_split':        sample['hard_split'],
            })
        return pd.DataFrame(records)

    def free_memory(self):
        if self.model is not None:
            del self.model
            self.model = None
        if self.tokenizer is not None:
            del self.tokenizer
            self.tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  [Memory freed] {self.model_id}")


# ============================================================
# 4. LLM Evaluator
# ============================================================
class LLMEvaluator(BaseEvaluator):
    def load_model(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = self.config.load_in_4bit,
            bnb_4bit_use_double_quant = True,
            bnb_4bit_quant_type       = self.config.bnb_4bit_quant_type,
            bnb_4bit_compute_dtype    = self.config.bnb_4bit_compute_dtype,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
        self.model.eval()
        print(f"  [Loaded] {self.model_id} (4-bit)")

    def _build_prompt(self, sample: dict) -> str:
        scene   = sample['movie_scene'][:self.config.max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            "Answer with only the letter (A, B, C, D, or E):"
        )

    def predict(self, sample: dict) -> str:
        prompt = self._build_prompt(sample)
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.config.max_input_tokens,
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.config.max_new_tokens,
                do_sample=self.config.do_sample,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        decoded = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()

        for char in decoded:
            if char in 'ABCDE':
                return char
        return 'A'  # fallback


# ============================================================
# 5. DeBERTa-v3 Evaluator
# ============================================================
class DeBERTaEvaluator(BaseEvaluator):
    """
    用 AutoModel + [CLS] embedding norm 对每个 choice 打分，
    选分数最高的 choice 作为预测答案。
    """
    def load_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model     = AutoModel.from_pretrained(self.model_id).to(
            'cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.model.eval()
        print(f"  [Loaded] {self.model_id}")

    def _score(self, text_a: str, text_b: str) -> float:
        inputs = self.tokenizer(
            text_a, text_b,
            return_tensors='pt',
            truncation=True,
            max_length=self.config.deberta_max_tokens,
            padding=True,
        ).to(self.model.device)
        with torch.no_grad():
            cls_emb = self.model(**inputs).last_hidden_state[:, 0, :]
        return cls_emb.norm(dim=-1).item()

    def predict(self, sample: dict) -> str:
        scene_q = (
            f"{sample['question']} "
            f"{sample['movie_scene'][:self.config.deberta_max_tokens // 2]}"
        )
        scores = [self._score(scene_q, c) for c in sample['choices']]
        return chr(65 + scores.index(max(scores))) # 'A'~'E'


# ============================================================
# 6. Metrics calculation
# ============================================================
class MetricsCalculator:
    def __init__(self, config: ExperimentConfig):
        self.config = config

    def compute(self, df: pd.DataFrame) -> dict:
        metrics = {}
        correct = df['pred'] == df['label']

        metrics['Overall']   = correct.mean()
        metrics['Overall_n'] = len(df)

        for abbr, cat_name in self.config.category_map.items():
            mask = df['question_category'].str.contains(
                cat_name, case=False, na=False
            )
            n = int(mask.sum())
            metrics[abbr]        = correct[mask].mean() if n > 0 else None
            metrics[f'{abbr}_n'] = n

        hard_mask = df['hard_split'] == True
        n_hard    = int(hard_mask.sum())
        metrics['Hard']   = correct[hard_mask].mean() if n_hard > 0 else None
        metrics['Hard_n'] = n_hard

        return metrics


# ============================================================
# 7. Phase1 Runner
# ============================================================
class Phase1Runner:
    EVALUATOR_REGISTRY = {
        'llm':     LLMEvaluator,
        'deberta': DeBERTaEvaluator,
    }

    def __init__(self, config: ExperimentConfig):
        self.config       = config
        self.dataset      = CinePileDataset(config)
        self.metrics_calc = MetricsCalculator(config)
        self.all_metrics  = {}

        # 运行 sanity check，确认 answer_key 转换正确
        self.dataset.run_sanity_check(n=3)

    def run(self):
        print("="*60)
        print("PHASE 1: CinePile Text-Only Evaluation")
        print("="*60)

        for display_name, model_id, eval_type in self.config.models:
            print(f"\n>>> [{eval_type.upper()}] {display_name}")

            evaluator_cls = self.EVALUATOR_REGISTRY[eval_type]
            evaluator     = evaluator_cls(model_id, self.config)
            evaluator.load_model()

            results_df = evaluator.evaluate(self.dataset.data)

            metrics = self.metrics_calc.compute(results_df)
            self.all_metrics[display_name] = metrics

            hard_str = (f"{metrics['Hard']:.2%}"
                        if metrics.get('Hard') is not None else "N/A")
            print(f"  Overall: {metrics['Overall']:.2%}  "
                  f"| Hard: {hard_str}  "
                  f"| n={metrics['Overall_n']}")

            # Free memory after each model run
            evaluator.free_memory()

        self._display_and_save()

    def _display_and_save(self):
        display_cols = ['Overall', 'TEMP', 'CRD', 'NPA', 'STA', 'TH', 'Hard']
        rows = []
        for model_name, metrics in self.all_metrics.items():
            row = {'Model': model_name}
            for col in display_cols:
                val = metrics.get(col)
                n   = metrics.get(f'{col}_n', metrics.get('Overall_n', ''))
                row[col] = f"{val:.1%}(n={n})" if val is not None else 'N/A'
            rows.append(row)

        df = pd.DataFrame(rows)
        print("\n" + "="*60)
        print("FINAL RESULTS — CinePile Text-Only")
        print("="*60)
        print(df.to_string(index=False))
        df.to_csv(self.config.output_csv, index=False)
        print(f"\nSaved → {self.config.output_csv}")


if __name__ == '__main__':

    # config = ExperimentConfig(
    #     max_samples     = None,     # using None for all test datasets
    #     max_scene_length= 1500,
    #     load_in_4bit    = True,
    #     output_csv      = 'phase1_mistral_results.csv',
    #     models=[
    #         # ('DeBERTa-v3-base', 'microsoft/deberta-v3-base',          'deberta'),
    #         # ('Llama-3.1-8B',    'meta-llama/Llama-3.1-8B-Instruct',  'llm'),
    #         ('Mistral-7B',      'mistralai/Mistral-7B-Instruct-v0.3', 'llm'),
    #     ]
    # )

    config = ExperimentConfig(
        max_samples     = None,     # using None for all test datasets
        max_scene_length= 1500,
        load_in_4bit    = True,
        output_csv      = 'phase1_llama_results.csv',
        models=[
            ('DeBERTa-v3-base', 'microsoft/deberta-v3-base',          'deberta'),
            # ('Llama-3.1-8B',    'meta-llama/Llama-3.1-8B-Instruct',  'llm'),
            # ('Mistral-7B',      'mistralai/Mistral-7B-Instruct-v0.3', 'llm'),
        ]
    )

    runner = Phase1Runner(config)
    runner.run()

[Dataset] Loaded 4941 samples
          answer_key sample : 'From anxiety to excitement' → E
          hard_split sample : True

=== Sanity Check ===
  [0] E → "From anxiety to excitement"
       original: "From anxiety to excitement"  match=True
  [1] E → "Suggests next steps"
       original: "Suggests next steps"  match=True
  [2] D → "The rough foliage"
       original: "The rough foliage"  match=True

PHASE 1: CinePile Text-Only Evaluation

>>> [DEBERTA] DeBERTa-v3-base


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [Loaded] microsoft/deberta-v3-base


  microsoft/deberta-v3-base: 100%|██████████| 4941/4941 [11:20<00:00,  7.26it/s]


  Overall: 21.72%  | Hard: 20.63%  | n=4941
  [Memory freed] microsoft/deberta-v3-base

FINAL RESULTS — CinePile Text-Only
          Model       Overall         TEMP           CRD          NPA           STA           TH         Hard
DeBERTa-v3-base 21.7%(n=4941) 20.8%(n=688) 21.9%(n=2068) 20.3%(n=463) 22.8%(n=1532) 17.9%(n=190) 20.6%(n=887)

Saved → phase1_llama_results.csv


In [ ]:
## Let's copy and paste phase 1 results here：
'''
============================================================
FINAL RESULTS — CinePile Text-Only
============================================================
          Model       Overall         TEMP           CRD          NPA           STA           TH         Hard
DeBERTa-v3-base 21.7%(n=4941) 20.8%(n=688) 21.9%(n=2068) 20.3%(n=463) 22.8%(n=1532) 17.9%(n=190) 20.6%(n=887)

     Model       Overall         TEMP           CRD          NPA           STA           TH         Hard
Mistral-7B 55.2%(n=4941) 34.9%(n=688) 57.4%(n=2068) 55.7%(n=463) 61.6%(n=1532) 52.6%(n=190) 32.7%(n=887)


Model	           Overall	       TEMP	          CRD	         NPA	         STA	         TH	        Hard
Llama-3.1-8B	60.0%(n=4941)	40.3%(n=688)	62.3%(n=2068)	56.4%(n=463)	67.8%(n=1532)	53.2%(n=190)	38.8%(n=887)
'''

In [ ]:
## Phase 2

## Phase 2

In [ ]:
!pip install transformers datasets accelerate bitsandbytes peft trl pyreft

INFO: pip is looking at multiple versions of pyreft to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.3/293.3 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 k

In [ ]:
from google.colab import userdata
import os

# Retrieve the HF_TOKEN from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Set it as an environment variable
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN loaded from Colab secrets and set as environment variable.")
# Now, when load_dataset is called, it will automatically use this token for authentication.

HF_TOKEN loaded from Colab secrets and set as environment variable.


In [ ]:
# ============================================================
# Phase 2: PEFT Fine-tuning on CinePile (Text-Only)
# Methods: QLoRA, Prefix Tuning, ReFT (LoReFT)
# !pip install transformers datasets accelerate bitsandbytes peft trl pyreft
# ============================================================

import gc
import os
import torch
import pandas as pd
import numpy as np
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
)
from peft import (
    LoraConfig,
    PrefixTuningConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm


# ============================================================
# 1. 配置类
# ============================================================
@dataclass
class Phase2Config:
    # --- 数据集 ---
    dataset_name: str          = 'tomg-group-umd/cinepile'
    train_split: str           = 'train'
    test_split: str            = 'test'
    max_train_samples: Optional[int] = 2000   # 调试用；None = 完整训练集
    max_test_samples: Optional[int]  = 500    # 调试用；None = 完整测试集

    # --- 基础模型（只选一个，节省显存）---
    base_model_id: str         = 'meta-llama/Llama-3.1-8B-Instruct'

    # --- 输入格式 ---
    max_scene_length: int      = 1000   # 训练时截短以节省显存
    max_seq_length: int        = 1024   # SFTTrainer 最大序列长度

    # --- 训练通用参数 ---
    num_train_epochs: int      = 2
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4   # 等效 batch size = 8
    learning_rate: float       = 2e-4
    warmup_ratio: float        = 0.05
    lr_scheduler_type: str     = 'cosine'
    fp16: bool                 = False
    bf16: bool                 = True
    logging_steps: int         = 20
    save_steps: int            = 200
    output_base_dir: str       = './phase2_outputs'

    # --- QLoRA 参数 ---
    lora_r: int                = 16
    lora_alpha: int            = 32
    lora_dropout: float        = 0.05
    lora_target_modules: list  = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ])

    # --- Prefix Tuning 参数 ---
    prefix_num_virtual_tokens: int = 20   # 也可以试 50

    # --- ReFT (LoReFT) 参数 ---
    reft_rank: int             = 4
    reft_layers: list          = field(default_factory=lambda: [8, 16, 24])

    # --- 评估 ---
    max_new_tokens: int        = 5
    output_csv: str            = 'phase2_results.csv'

    # --- 类别映射（与 Phase 1 一致）---
    category_map: dict = field(default_factory=lambda: {
        'TEMP': 'Temporal',
        'CRD':  'Character and',
        'NPA':  'Narrative and',
        'STA':  'Setting and',
        'TH':   'Theme Exploration',
    })


# ============================================================
# 2. 数据集（复用 Phase 1 的 answer_key 修复逻辑）
# ============================================================
class CinePileDataset:
    LETTERS = ['A', 'B', 'C', 'D', 'E']

    def __init__(self, config: Phase2Config):
        self.config = config
        self.train_data = self._load(config.train_split, config.max_train_samples)
        self.test_data  = self._load(config.test_split,  config.max_test_samples)
        print(f"[Dataset] Train: {len(self.train_data)} | Test: {len(self.test_data)}")

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        for i, c in enumerate(choices):
            if answer_text.strip() == c.strip():
                return cls.LETTERS[i]
        for i, c in enumerate(choices):
            if answer_text.strip() in c or c.strip() in answer_text:
                return cls.LETTERS[i]
        return 'A'

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == 'true'

    def _load(self, split: str, max_samples: Optional[int]) -> list:
        raw = load_dataset(self.config.dataset_name, split=split)
        samples = []
        for ex in raw:
            letter = self._answer_text_to_letter(ex['answer_key'], ex['choices'])
            samples.append({
                'movie_scene':       ex['movie_scene'],
                'question':          ex['question'],
                'choices':           ex['choices'],
                'answer_key':        letter,
                'question_category': ex['question_category'],
                'hard_split':        self._normalize_hard_split(ex['hard_split']),
            })
        if max_samples:
            samples = samples[:max_samples]
        return samples


# ============================================================
# 3. Prompt 工具
# ============================================================
class PromptBuilder:
    """
    训练用：构造带答案的完整 prompt（供 SFTTrainer 做 causal LM 训练）
    推理用：构造不含答案的 prompt（供 generate 输出）
    """
    @staticmethod
    def build_train_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample['movie_scene'][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
        )
        answer  = sample['answer_key']
        return (
            f"You are watching a movie. Based on the scene description, "
            f"answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            f"Answer: {answer}"   # 训练时包含答案
        )

    @staticmethod
    def build_inference_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample['movie_scene'][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
        )
        return (
            f"You are watching a movie. Based on the scene description, "
            f"answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            f"Answer:"   # 推理时不含答案
        )


# ============================================================
# 4. Evaluator（复用 Phase 1 逻辑）
# ============================================================
class Evaluator:
    def __init__(self, config: Phase2Config):
        self.config = config

    def evaluate(self, model, tokenizer, test_data: list,
                 method_name: str) -> dict:
        model.eval()
        records = []
        for sample in tqdm(test_data, desc=f"  Evaluating {method_name}"):
            pred = self._predict(model, tokenizer, sample)
            records.append({
                'pred':              pred,
                'label':             sample['answer_key'],
                'question_category': sample['question_category'],
                'hard_split':        sample['hard_split'],
            })
        df = pd.DataFrame(records)
        return self._compute_metrics(df)

    def _predict(self, model, tokenizer, sample: dict) -> str:
        prompt  = PromptBuilder.build_inference_prompt(
            sample, self.config.max_scene_length
        )
        inputs  = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.config.max_seq_length,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.config.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        decoded = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()

        for char in decoded:
            if char in 'ABCDE':
                return char
        return 'A'

    def _compute_metrics(self, df: pd.DataFrame) -> dict:
        correct  = df['pred'] == df['label']
        metrics  = {
            'Overall':   correct.mean(),
            'Overall_n': len(df),
        }
        for abbr, keyword in self.config.category_map.items():
            mask = df['question_category'].str.contains(
                keyword, case=False, na=False
            )
            n = int(mask.sum())
            metrics[abbr]        = correct[mask].mean() if n > 0 else None
            metrics[f'{abbr}_n'] = n

        hard_mask    = df['hard_split'] == True
        n_hard       = int(hard_mask.sum())
        metrics['Hard']   = correct[hard_mask].mean() if n_hard > 0 else None
        metrics['Hard_n'] = n_hard
        return metrics


# ============================================================
# 5. 基类 Trainer
# ============================================================
class BasePEFTTrainer(ABC):
    def __init__(self, config: Phase2Config):
        self.config    = config
        self.model     = None
        self.tokenizer = None
        self.method_name = "base"

    def _load_base_model_4bit(self):
        """加载 4-bit 量化基础模型（QLoRA 用）"""
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_use_double_quant = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = torch.bfloat16,
        )
        tokenizer = AutoTokenizer.from_pretrained(self.config.base_model_id)
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.padding_side = "right"

        model = AutoModelForCausalLM.from_pretrained(
            self.config.base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
        return model, tokenizer

    def _load_base_model_bf16(self):
        """加载 bf16 基础模型（Prefix Tuning / ReFT 用）"""
        tokenizer = AutoTokenizer.from_pretrained(self.config.base_model_id)
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.padding_side = "right"

        model = AutoModelForCausalLM.from_pretrained(
            self.config.base_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        return model, tokenizer

    def _make_hf_dataset(self, data: list):
        """将样本列表转为 HuggingFace Dataset（SFTTrainer 需要）"""
        from datasets import Dataset
        return Dataset.from_list([
            {'text': PromptBuilder.build_train_prompt(
                s, self.config.max_scene_length
            )}
            for s in data
        ])

    def _output_dir(self) -> str:
        path = os.path.join(self.config.output_base_dir, self.method_name)
        os.makedirs(path, exist_ok=True)
        return path

    def free_memory(self):
        if self.model is not None:
            del self.model
            self.model = None
        if self.tokenizer is not None:
            del self.tokenizer
            self.tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  [Memory freed] {self.method_name}")

    @abstractmethod
    def train(self, train_data: list): pass

    @abstractmethod
    def load_for_inference(self): pass


# ============================================================
# 6. QLoRA Trainer
# ============================================================
class QLoRATrainer(BasePEFTTrainer):
    """
    QLoRA: 4-bit 量化基础模型 + LoRA adapter
    最成熟稳定，推理类问题（Causal / NPA）预期表现最好
    """
    def __init__(self, config: Phase2Config):
        super().__init__(config)
        self.method_name = f"qlora_r{config.lora_r}"

    def train(self, train_data: list):
        print(f"\n[QLoRA] Loading base model (4-bit)...")
        model, tokenizer = self._load_base_model_4bit()

        lora_config = LoraConfig(
            r                = self.config.lora_r,
            lora_alpha       = self.config.lora_alpha,
            lora_dropout     = self.config.lora_dropout,
            target_modules   = self.config.lora_target_modules,
            bias             = "none",
            task_type        = TaskType.CAUSAL_LM,
        )

        train_args = SFTConfig(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.config.num_train_epochs,
            per_device_train_batch_size = self.config.per_device_train_batch_size,
            gradient_accumulation_steps = self.config.gradient_accumulation_steps,
            learning_rate               = self.config.learning_rate,
            warmup_ratio                = self.config.warmup_ratio,
            lr_scheduler_type           = self.config.lr_scheduler_type,
            bf16                        = self.config.bf16,
            logging_steps               = self.config.logging_steps,
            save_steps                  = self.config.save_steps,
            max_seq_length              = self.config.max_seq_length,
            dataset_text_field          = "text",
            report_to                   = "none",
            gradient_checkpointing      = True,
            optim                       = "paged_adamw_8bit",
        )

        hf_dataset = self._make_hf_dataset(train_data)

        trainer = SFTTrainer(
            model         = model,
            args          = train_args,
            train_dataset = hf_dataset,
            peft_config   = lora_config,
            tokenizer     = tokenizer,
        )

        print(f"[QLoRA] Training... (samples={len(train_data)})")
        trainer.train()
        trainer.save_model(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[QLoRA] Saved → {self._output_dir()}")

        self.model     = trainer.model
        self.tokenizer = tokenizer

    def load_for_inference(self):
        """从保存的 checkpoint 加载用于推理"""
        print(f"[QLoRA] Loading from {self._output_dir()}")
        model, tokenizer = self._load_base_model_4bit()
        self.model     = PeftModel.from_pretrained(model, self._output_dir())
        self.tokenizer = tokenizer


# ============================================================
# 7. Prefix Tuning Trainer
# ============================================================
class PrefixTuningTrainer(BasePEFTTrainer):
    """
    Prefix Tuning: 在每层添加可学习 virtual tokens
    对长序列上下文任务（Temporal / CRD）预期有优势
    注意：Prefix Tuning 不兼容 4-bit，需要 bf16 全精度基础模型
    """
    def __init__(self, config: Phase2Config):
        super().__init__(config)
        self.method_name = f"prefix_vt{config.prefix_num_virtual_tokens}"

    def train(self, train_data: list):
        print(f"\n[Prefix] Loading base model (bf16)...")
        model, tokenizer = self._load_base_model_bf16()

        prefix_config = PrefixTuningConfig(
            task_type          = TaskType.CAUSAL_LM,
            num_virtual_tokens = self.config.prefix_num_virtual_tokens,
            prefix_projection  = True,   # MLP 投影，比直接优化更稳定
        )

        model = get_peft_model(model, prefix_config)
        model.print_trainable_parameters()

        train_args = SFTConfig(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.config.num_train_epochs,
            per_device_train_batch_size = self.config.per_device_train_batch_size,
            gradient_accumulation_steps = self.config.gradient_accumulation_steps,
            learning_rate               = 1e-3,   # Prefix Tuning 需要更大 lr
            warmup_ratio                = self.config.warmup_ratio,
            lr_scheduler_type           = self.config.lr_scheduler_type,
            bf16                        = self.config.bf16,
            logging_steps               = self.config.logging_steps,
            save_steps                  = self.config.save_steps,
            max_seq_length              = self.config.max_seq_length,
            dataset_text_field          = "text",
            report_to                   = "none",
            gradient_checkpointing      = False,  # Prefix Tuning 不兼容 grad ckpt
        )

        hf_dataset = self._make_hf_dataset(train_data)

        trainer = SFTTrainer(
            model         = model,
            args          = train_args,
            train_dataset = hf_dataset,
            tokenizer     = tokenizer,
        )

        print(f"[Prefix] Training... (samples={len(train_data)})")
        trainer.train()
        trainer.save_model(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[Prefix] Saved → {self._output_dir()}")

        self.model     = model
        self.tokenizer = tokenizer

    def load_for_inference(self):
        print(f"[Prefix] Loading from {self._output_dir()}")
        model, tokenizer = self._load_base_model_bf16()
        self.model     = PeftModel.from_pretrained(model, self._output_dir())
        self.tokenizer = tokenizer


# ============================================================
# 8. ReFT (LoReFT) Trainer
# ============================================================
class ReFTTrainer(BasePEFTTrainer):
    """
    ReFT / LoReFT: 在 representation 空间做低秩线性干预
    参数极少 (<0.1%)，新方法，作为探索性对比
    依赖: pip install pyreft
    """
    def __init__(self, config: Phase2Config):
        super().__init__(config)
        self.method_name = f"reft_r{config.reft_rank}"

    def train(self, train_data: list):
        try:
            import pyreft
        except ImportError:
            print("[ReFT] pyreft not installed. Run: pip install pyreft")
            return

        print(f"\n[ReFT] Loading base model (bf16)...")
        model, tokenizer = self._load_base_model_bf16()

        # 在指定层应用 LoReFT 干预
        reft_config = pyreft.ReftConfig(
            representations=[
                {
                    "layer"           : l,
                    "component"       : "block_output",
                    "low_rank_dimension": self.config.reft_rank,
                    "intervention"    : pyreft.LoreftIntervention(
                        embed_dim          = model.config.hidden_size,
                        low_rank_dimension = self.config.reft_rank,
                    )
                }
                for l in self.config.reft_layers
            ]
        )
        reft_model = pyreft.get_reft_model(model, reft_config)
        reft_model.print_trainable_parameters()

        # 构造训练数据（ReFT 用自己的 DataCollator）
        train_dataset = self._make_reft_dataset(
            train_data, tokenizer, pyreft
        )

        training_args = TrainingArguments(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.config.num_train_epochs,
            per_device_train_batch_size = self.config.per_device_train_batch_size,
            gradient_accumulation_steps = self.config.gradient_accumulation_steps,
            learning_rate               = self.config.learning_rate,
            warmup_ratio                = self.config.warmup_ratio,
            bf16                        = self.config.bf16,
            logging_steps               = self.config.logging_steps,
            save_steps                  = self.config.save_steps,
            report_to                   = "none",
        )

        reft_trainer = pyreft.ReftTrainerForCausalLM(
            model         = reft_model,
            args          = training_args,
            train_dataset = train_dataset,
            tokenizer     = tokenizer,
            data_collator = pyreft.ReftDataCollator(
                data_collator=DataCollatorWithPadding(tokenizer)
            ),
        )

        print(f"[ReFT] Training... (samples={len(train_data)})")
        reft_trainer.train()
        reft_model.save(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[ReFT] Saved → {self._output_dir()}")

        self.model     = reft_model
        self.tokenizer = tokenizer

    def _make_reft_dataset(self, data, tokenizer, pyreft):
        """将样本转换为 ReFT 所需的 dataset 格式"""
        from datasets import Dataset
        prompts, outputs = [], []
        for s in data:
            prompt = PromptBuilder.build_inference_prompt(
                s, self.config.max_scene_length
            )
            prompts.append(prompt)
            outputs.append(s['answer_key'])

        return pyreft.make_last_position_supervised_data_module(
            tokenizer     = tokenizer,
            model         = self.model,
            inputs        = prompts,
            outputs       = outputs,
            num_interventions = len(self.config.reft_layers),
        )

    def load_for_inference(self):
        try:
            import pyreft
        except ImportError:
            print("[ReFT] pyreft not installed.")
            return

        print(f"[ReFT] Loading from {self._output_dir()}")
        model, tokenizer = self._load_base_model_bf16()
        self.model     = pyreft.ReftModel.load(self._output_dir(), model)
        self.tokenizer = tokenizer


# ============================================================
# 9. Phase2 主运行器
# ============================================================
class Phase2Runner:
    def __init__(self, config: Phase2Config):
        self.config      = config
        self.dataset     = CinePileDataset(config)
        self.evaluator   = Evaluator(config)
        self.all_metrics = {}

        # 注册要运行的 PEFT 方法
        self.trainers = [
            # QLoRATrainer(config),
            # PrefixTuningTrainer(config),
            ReFTTrainer(config),
        ]

    def run(self):
        print("\n" + "="*60)
        print("PHASE 2: PEFT Fine-tuning on CinePile")
        print(f"Base model : {self.config.base_model_id}")
        print(f"Train size : {len(self.dataset.train_data)}")
        print(f"Test size  : {len(self.dataset.test_data)}")
        print("="*60)

        for trainer in self.trainers:
            print(f"\n{'='*60}")
            print(f"Method: {trainer.method_name.upper()}")
            print(f"{'='*60}")

            # 1. 训练
            trainer.train(self.dataset.train_data)

            # 2. 评估
            if trainer.model is not None:
                metrics = self.evaluator.evaluate(
                    trainer.model,
                    trainer.tokenizer,
                    self.dataset.test_data,
                    trainer.method_name,
                )
                self.all_metrics[trainer.method_name] = metrics
                hard_str = (f"{metrics['Hard']:.2%}"
                            if metrics.get('Hard') is not None else "N/A")
                print(f"  Overall: {metrics['Overall']:.2%} "
                      f"| Hard: {hard_str}")

            # 3. 释放显存
            trainer.free_memory()

        self._display_and_save()

    def _display_and_save(self):
        display_cols = ['Overall', 'TEMP', 'CRD', 'NPA', 'STA', 'TH', 'Hard']
        rows = []
        for method_name, metrics in self.all_metrics.items():
            row = {'Method': method_name}
            for col in display_cols:
                val = metrics.get(col)
                n   = metrics.get(f'{col}_n', metrics.get('Overall_n', ''))
                row[col] = f"{val:.1%}(n={n})" if val is not None else 'N/A'
            rows.append(row)

        df = pd.DataFrame(rows)
        print("\n" + "="*60)
        print("PHASE 2 FINAL RESULTS — CinePile PEFT Fine-tuning")
        print("="*60)
        print(df.to_string(index=False))
        df.to_csv(self.config.output_csv, index=False)
        print(f"\nSaved → {self.config.output_csv}")


# ============================================================
# 10. 入口
# ============================================================
if __name__ == '__main__':

    config = Phase2Config(
        base_model_id        = 'meta-llama/Llama-3.1-8B-Instruct',
        max_train_samples    = 20,   # 调试：2000；完整：None
        max_test_samples     = 500,    # 调试：500； 完整：None
        num_train_epochs     = 2,
        lora_r               = 16,
        prefix_num_virtual_tokens = 20,
        reft_rank            = 4,
        reft_layers          = [8, 16, 24],
        output_base_dir      = './phase2_outputs',
        output_csv           = 'phase2_results.csv',
    )

    runner = Phase2Runner(config)
    runner.run()


README.md: 0.00B [00:00, ?B/s]

v2/train-00000-of-00003.parquet:   0%|          | 0.00/21.9M [00:00<?, ?B/s]

v2/train-00001-of-00003.parquet:   0%|          | 0.00/23.2M [00:00<?, ?B/s]

v2/train-00002-of-00003.parquet:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

v2/test-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

TypeError: 'LoreftIntervention' object is not subscriptable

In [ ]:
### Reft only

### 2.2 Reft only

In [ ]:
!pip install "transformers==4.39.3" "accelerate==0.27.2" -U
!pip install "pyreft==0.0.7" -U
!pip install datasets bitsandbytes


  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
Using cached huggingface_hub-1.5.0-py3-none-any.whl (596 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: accelerate
    Found exist

In [ ]:
import transformers, pyreft
import transformers.utils
print("transformers:", transformers.__version__)

ModuleNotFoundError: No module named 'pyreft'

In [ ]:
# ============================================================
# ReFT (LoReFT) on CinePile with pure transformers.Trainer
#   - 兼容 transformers 5.3.0
#   - 关键：ReFTModelWrapper 把 IntervenableModel 包成标准 forward 接口
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional
from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from transformers.modeling_outputs import CausalLMOutputWithPast

import pyreft  # pip install pyreft


@dataclass
class ReFTConfigCinePile:
    dataset_name: str          = "tomg-group-umd/cinepile"
    train_split: str           = "train"
    test_split: str            = "test"
    max_train_samples: Optional[int] = 200
    max_test_samples: Optional[int]  = 100

    base_model_id: str         = "meta-llama/Llama-3.1-8B-Instruct"

    max_scene_length: int      = 1000
    max_seq_length: int        = 1024

    num_train_epochs: float    = 1.0
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float       = 4e-3
    warmup_ratio: float        = 0.05

    bf16: bool                 = True
    logging_steps: int         = 20
    output_dir: str            = "./reft_cinepile_trainer"

    reft_rank: int             = 4
    reft_layer: int            = 15
    component: str             = "block_output"

    max_new_tokens: int        = 5

    category_map: dict = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


# ===================== 关键：ReFT 模型包装类 =====================
class ReFTModelWrapper(nn.Module):
    """
    把 pyreft.IntervenableModel 包装成标准 HuggingFace forward 接口：
        forward(input_ids, attention_mask, labels) -> CausalLMOutputWithPast
    让 transformers.Trainer 可以直接调用。
    """

    def __init__(self, reft_model):
        super().__init__()
        self.reft_model = reft_model
        # config 是 Trainer 检测 model 类型用的，从内部基础模型取
        self.config = reft_model.model.config

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs,
    ):
        # pyreft IntervenableModel.forward 需要:
        #   base = {"input_ids": ..., "attention_mask": ...}
        #   unit_locations = None（对 last-position 干预）
        base = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        _, counterfactual_outputs = self.reft_model(
            base,
            unit_locations=None,
            output_original_output=False,
        )
        # counterfactual_outputs 是 (logits,) 或 BaseModelOutput
        logits = counterfactual_outputs[0]  # shape: (batch, seq_len, vocab)

        loss = None
        if labels is not None:
            # shift: 标准 causal LM loss 计算方式
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutputWithPast(
            loss=loss,
            logits=logits,
        )

    def generate(self, **kwargs):
        # 推理阶段直接用内部基础模型生成，不走 intervention
        return self.reft_model.model.generate(**kwargs)

    def save(self, path: str):
        self.reft_model.save(path)


# ===================== 数据与 prompt =====================
LETTERS = ["A", "B", "C", "D", "E"]

def answer_text_to_letter(answer_text: str, choices: list) -> str:
    for i, c in enumerate(choices):
        if answer_text.strip() == c.strip():
            return LETTERS[i]
    for i, c in enumerate(choices):
        if answer_text.strip() in c or c.strip() in answer_text:
            return LETTERS[i]
    return "A"

def normalize_hard_split(raw) -> bool:
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() == "true"

def load_cinepile(cfg: ReFTConfigCinePile):
    raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
    raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

    def _convert(ex):
        return {
            "movie_scene":       ex["movie_scene"],
            "question":          ex["question"],
            "choices":           ex["choices"],
            "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
            "question_category": ex["question_category"],
            "hard_split":        normalize_hard_split(ex["hard_split"]),
        }

    train_data = [_convert(ex) for ex in raw_train]
    test_data  = [_convert(ex) for ex in raw_test]

    if cfg.max_train_samples:
        train_data = train_data[:cfg.max_train_samples]
    if cfg.max_test_samples:
        test_data  = test_data[:cfg.max_test_samples]

    print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
    return train_data, test_data

def build_inference_prompt(sample, max_scene_length: int) -> str:
    scene   = sample["movie_scene"][:max_scene_length]
    options = "\n".join(
        f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
    )
    return (
        "You are watching a movie. Based on the scene description, "
        "answer the multiple choice question.\n\n"
        f"Scene: {scene}\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )

def make_supervised_dataset(train_data, tokenizer, cfg: ReFTConfigCinePile) -> Dataset:
    texts = []
    for s in train_data:
        prompt = build_inference_prompt(s, cfg.max_scene_length)
        texts.append(prompt + " " + s["answer_key"])

    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=cfg.max_seq_length,
        padding=False,
    )

    max_len = max(len(ids) for ids in encodings["input_ids"])
    input_ids_list, attention_mask_list, labels_list = [], [], []

    for ids, mask in zip(encodings["input_ids"], encodings["attention_mask"]):
        pad_len = max_len - len(ids)
        input_ids_list.append(ids + [tokenizer.pad_token_id] * pad_len)
        attention_mask_list.append(mask + [0] * pad_len)
        labels_list.append(ids + [-100] * pad_len)

    return Dataset.from_dict(
        {
            "input_ids": input_ids_list,
            "attention_mask": attention_mask_list,
            "labels": labels_list,
        }
    )


# ===================== 评估 =====================
def predict_choice(model_wrapper: ReFTModelWrapper, tokenizer, sample,
                   cfg: ReFTConfigCinePile, device="cuda"):
    model_wrapper.eval()
    prompt = build_inference_prompt(sample, cfg.max_scene_length)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(device)

    with torch.no_grad():
        outputs = model_wrapper.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().upper()

    for ch in decoded:
        if ch in "ABCDE":
            return ch
    return "A"

def evaluate_reft(model_wrapper, tokenizer, test_data, cfg: ReFTConfigCinePile):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    records = []
    for s in tqdm(test_data, desc="Eval ReFT"):
        pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
        records.append(
            {
                "pred":              pred,
                "label":             s["answer_key"],
                "question_category": s["question_category"],
                "hard_split":        s["hard_split"],
            }
        )
    df = pd.DataFrame(records)
    correct = df["pred"] == df["label"]
    metrics = {"Overall": correct.mean(), "Overall_n": len(df)}
    for abbr, keyword in cfg.category_map.items():
        mask = df["question_category"].str.contains(keyword, case=False, na=False)
        n = int(mask.sum())
        metrics[abbr]        = correct[mask].mean() if n > 0 else None
        metrics[f"{abbr}_n"] = n
    hard_mask = df["hard_split"] == True
    n_hard    = int(hard_mask.sum())
    metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
    metrics["Hard_n"] = n_hard
    return metrics


# ===================== 训练 =====================
def train_reft_with_trainer(cfg: ReFTConfigCinePile):
    os.makedirs(cfg.output_dir, exist_ok=True)
    train_data, test_data = load_cinepile(cfg)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("[ReFT] Loading base model...")
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    reft_cfg = pyreft.ReftConfig(
        representations={
            "layer": cfg.reft_layer,
            "component": cfg.component,
            "low_rank_dimension": cfg.reft_rank,
            "intervention": pyreft.LoreftIntervention(
                embed_dim=base_model.config.hidden_size,
                low_rank_dimension=cfg.reft_rank,
            ),
        }
    )

    reft_model   = pyreft.get_reft_model(base_model, reft_cfg)
    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    train_ds = make_supervised_dataset(train_data, tokenizer, cfg)

    training_args = TrainingArguments(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_train_epochs,
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        learning_rate               = cfg.learning_rate,
        warmup_ratio                = cfg.warmup_ratio,
        bf16                        = cfg.bf16,
        logging_steps               = cfg.logging_steps,
        report_to                   = [],
        save_strategy               = "no",
    )

    trainer = Trainer(
        model         = model_wrapper,
        args          = training_args,
        train_dataset = train_ds,
    )

    print("[ReFT] Training with Trainer...")
    trainer.train()

    model_wrapper.save(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"[ReFT] Saved to: {cfg.output_dir}")

    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)
    print("\n[ReFT] Eval metrics:", metrics)
    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "reft_metrics.csv"), index=False
    )
    print(f"[ReFT] Metrics saved.")
    return model_wrapper, tokenizer, metrics


# ===================== 入口 =====================
if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    cfg = ReFTConfigCinePile(
        base_model_id     = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples = 200,
        max_test_samples  = 100,
        num_train_epochs  = 1.0,
        reft_rank         = 8, # tested 4
        reft_layer        = 15,
        output_dir        = "./reft_cinepile_llama3_8B_trainer",
    )

    model, tokenizer, metrics = train_reft_with_trainer(cfg)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


transformers: 5.3.0
[Dataset] Train: 200 | Test: 100
[ReFT] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[ReFT] Training with Trainer...


Step,Training Loss
20,7.342104


Directory './reft_cinepile_llama3_8B_trainer' already exists.
[ReFT] Saved to: ./reft_cinepile_llama3_8B_trainer


Eval ReFT: 100%|██████████| 100/100 [00:20<00:00,  4.98it/s]



[ReFT] Eval metrics: {'Overall': np.float64(0.5), 'Overall_n': 100, 'TEMP': np.float64(0.3076923076923077), 'TEMP_n': 13, 'CRD': np.float64(0.5238095238095238), 'CRD_n': 42, 'NPA': np.float64(0.6), 'NPA_n': 5, 'STA': np.float64(0.5757575757575758), 'STA_n': 33, 'TH': np.float64(0.2857142857142857), 'TH_n': 7, 'Hard': np.float64(0.5384615384615384), 'Hard_n': 26}
[ReFT] Metrics saved.


In [ ]:
### advanced version

In [ ]:
# ============================================================
# ReFT (LoReFT) on CinePile — Full Script
#   - 兼容 transformers 5.3.0
#   - 支持多层 LoReFT 干预
#   - 使用 ReFTModelWrapper 包装 IntervenableModel
# pip install pyreft datasets accelerate
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional
from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from transformers.modeling_outputs import CausalLMOutputWithPast

import pyreft


# ============================================================
# 1. 配置
# ============================================================
@dataclass
class ReFTConfigCinePile:
    # 数据集
    dataset_name: str          = "tomg-group-umd/cinepile"
    train_split: str           = "train"
    test_split: str            = "test"
    max_train_samples: Optional[int] = 2000
    max_test_samples: Optional[int]  = 500

    # 模型
    base_model_id: str         = "meta-llama/Llama-3.1-8B-Instruct"

    # 输入
    max_scene_length: int      = 1000
    max_seq_length: int        = 1024

    # 训练
    num_train_epochs: float    = 2.0
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float       = 2e-3
    warmup_ratio: float        = 0.05
    bf16: bool                 = True
    logging_steps: int         = 20
    output_dir: str            = "./reft_cinepile"

    # LoReFT 配置
    reft_rank: int             = 8
    reft_layers: list          = field(default_factory=lambda: [8, 15, 24])
    component: str             = "block_output"

    # 推理
    max_new_tokens: int        = 5

    # 评估类别
    category_map: dict = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


# ============================================================
# 2. ReFT 模型包装类（核心）
# ============================================================
class ReFTModelWrapper(nn.Module):
    def __init__(self, reft_model):
        super().__init__()
        self.reft_model = reft_model
        self.config     = reft_model.model.config

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        base = {"input_ids": input_ids, "attention_mask": attention_mask}
        _, counterfactual_outputs = self.reft_model(
            base,
            unit_locations=None,
            output_original_output=False,
        )
        logits = counterfactual_outputs[0]

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )
        return CausalLMOutputWithPast(loss=loss, logits=logits)

    def generate(self, input_ids=None, attention_mask=None, **kwargs):
        """推理时也走干预层，保证训练和推理一致"""
        device         = input_ids.device
        generated      = input_ids.clone()
        max_new_tokens = kwargs.get("max_new_tokens", 5)
        eos_id         = kwargs.get("pad_token_id", None)

        for _ in range(max_new_tokens):
            attn = torch.ones_like(generated)
            base = {"input_ids": generated, "attention_mask": attn}

            with torch.no_grad():
                _, outputs = self.reft_model(
                    base,
                    unit_locations=None,
                    output_original_output=False,
                )
            logits   = outputs[0]
            next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tok], dim=1)

            if eos_id is not None and (next_tok == eos_id).all():
                break

        return generated

    def save(self, path: str):
        self.reft_model.save(path)


# ============================================================
# 3. 数据集加载
# ============================================================
LETTERS = ["A", "B", "C", "D", "E"]

def answer_text_to_letter(answer_text: str, choices: list) -> str:
    for i, c in enumerate(choices):
        if answer_text.strip() == c.strip():
            return LETTERS[i]
    for i, c in enumerate(choices):
        if answer_text.strip() in c or c.strip() in answer_text:
            return LETTERS[i]
    return "A"

def normalize_hard_split(raw) -> bool:
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() == "true"

def load_cinepile(cfg: ReFTConfigCinePile):
    raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
    raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

    def _convert(ex):
        return {
            "movie_scene":       ex["movie_scene"],
            "question":          ex["question"],
            "choices":           ex["choices"],
            "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
            "question_category": ex["question_category"],
            "hard_split":        normalize_hard_split(ex["hard_split"]),
        }

    train_data = [_convert(ex) for ex in raw_train]
    test_data  = [_convert(ex) for ex in raw_test]

    if cfg.max_train_samples:
        train_data = train_data[:cfg.max_train_samples]
    if cfg.max_test_samples:
        test_data  = test_data[:cfg.max_test_samples]

    print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
    return train_data, test_data


# ============================================================
# 4. Prompt 构建
# ============================================================
def build_inference_prompt(sample, max_scene_length: int) -> str:
    scene   = sample["movie_scene"][:max_scene_length]
    options = "\n".join(
        f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
    )
    return (
        "You are watching a movie. Based on the scene description, "
        "answer the multiple choice question.\n\n"
        f"Scene: {scene}\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


# ============================================================
# 5. 训练数据集（统一 padding，避免 collator 出错）
# ============================================================
def make_supervised_dataset(train_data, tokenizer, cfg: ReFTConfigCinePile) -> Dataset:
    texts = [
        build_inference_prompt(s, cfg.max_scene_length) + " " + s["answer_key"]
        for s in train_data
    ]

    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=cfg.max_seq_length,
        padding=False,
    )

    max_len = max(len(ids) for ids in encodings["input_ids"])
    input_ids_list, attention_mask_list, labels_list = [], [], []

    for ids, mask in zip(encodings["input_ids"], encodings["attention_mask"]):
        pad_len = max_len - len(ids)
        input_ids_list.append(ids + [tokenizer.pad_token_id] * pad_len)
        attention_mask_list.append(mask + [0] * pad_len)
        labels_list.append(ids + [-100] * pad_len)   # pad 部分不计入 loss

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_mask_list,
        "labels":         labels_list,
    })


# ============================================================
# 6. 评估
# ============================================================
def predict_choice(model_wrapper: ReFTModelWrapper, tokenizer, sample,
                   cfg: ReFTConfigCinePile, device: str) -> str:
    model_wrapper.eval()
    prompt = build_inference_prompt(sample, cfg.max_scene_length)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(device)

    with torch.no_grad():
        outputs = model_wrapper.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().upper()

    for ch in decoded:
        if ch in "ABCDE":
            return ch
    return "A"

def evaluate_reft(model_wrapper: ReFTModelWrapper, tokenizer,
                  test_data: list, cfg: ReFTConfigCinePile) -> dict:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    records = []
    for s in tqdm(test_data, desc="Eval ReFT"):
        pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
        records.append({
            "pred":              pred,
            "label":             s["answer_key"],
            "question_category": s["question_category"],
            "hard_split":        s["hard_split"],
        })

    df      = pd.DataFrame(records)
    correct = df["pred"] == df["label"]
    metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

    for abbr, keyword in cfg.category_map.items():
        mask = df["question_category"].str.contains(keyword, case=False, na=False)
        n    = int(mask.sum())
        metrics[abbr]        = correct[mask].mean() if n > 0 else None
        metrics[f"{abbr}_n"] = n

    hard_mask = df["hard_split"] == True
    n_hard    = int(hard_mask.sum())
    metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
    metrics["Hard_n"] = n_hard
    return metrics


# ============================================================
# 7. 训练主函数
# ============================================================
def train_reft_with_trainer(cfg: ReFTConfigCinePile):
    os.makedirs(cfg.output_dir, exist_ok=True)
    train_data, test_data = load_cinepile(cfg)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("[ReFT] Loading base model...")
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    # 多层 LoReFT 干预配置
    print(f"[ReFT] Building LoReFT config: layers={cfg.reft_layers}, rank={cfg.reft_rank}")
    representations = [
        {
            "layer":             l,
            "component":         cfg.component,
            "low_rank_dimension": cfg.reft_rank,
            "intervention":      pyreft.LoreftIntervention(
                embed_dim          = base_model.config.hidden_size,
                low_rank_dimension = cfg.reft_rank,
            ),
        }
        for l in cfg.reft_layers
    ]
    reft_cfg   = pyreft.ReftConfig(representations=representations)
    reft_model = pyreft.get_reft_model(base_model, reft_cfg)
    reft_model.set_device(device)

    model_wrapper = ReFTModelWrapper(reft_model)

    train_ds = make_supervised_dataset(train_data, tokenizer, cfg)
    print(f"[ReFT] Train dataset size: {len(train_ds)}")

    training_args = TrainingArguments(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_train_epochs,
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        learning_rate               = cfg.learning_rate,
        warmup_ratio                = cfg.warmup_ratio,
        bf16                        = cfg.bf16,
        logging_steps               = cfg.logging_steps,
        report_to                   = [],
        save_strategy               = "no",
    )

    trainer = Trainer(
        model         = model_wrapper,
        args          = training_args,
        train_dataset = train_ds,
    )

    print(f"[ReFT] Training... epochs={cfg.num_train_epochs}, lr={cfg.learning_rate}")
    trainer.train()

    model_wrapper.save(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"[ReFT] Saved to: {cfg.output_dir}")

    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)
    print("\n[ReFT] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "reft_metrics.csv"), index=False
    )
    print(f"[ReFT] Metrics saved → {cfg.output_dir}/reft_metrics.csv")

    return model_wrapper, tokenizer, metrics


# ============================================================
# 8. 入口
# ============================================================
if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    # ---- 调试配置（先用小数据跑通）----
    cfg = ReFTConfigCinePile(
        base_model_id            = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples        = 2000,    # 调试：200 → 正式：2000
        max_test_samples         = 500,    # 调试：100 → 正式：500
        num_train_epochs         = 2.0,    # 调试：1   → 正式：2
        reft_rank                = 16,
        reft_layers              = [15, 24],
        learning_rate            = 2e-3,
        output_dir               = "./reft_cinepile_llama3_multilayer",
    )

    model, tokenizer, metrics = train_reft_with_trainer(cfg)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


transformers: 5.3.0
[Dataset] Train: 2000 | Test: 500
[ReFT] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[ReFT] Building LoReFT config: layers=[8, 15, 24], rank=16


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[ReFT] Train dataset size: 2000
[ReFT] Training... epochs=1.0, lr=0.002


Step,Training Loss
20,7.654836
40,6.425362
60,5.239740
80,4.091982
100,3.138413
120,2.256777
140,1.687708
160,1.158774
180,0.954117
200,0.849488


Directory './reft_cinepile_llama3_multilayer' already exists.
[ReFT] Saved to: ./reft_cinepile_llama3_multilayer


Eval ReFT: 100%|██████████| 500/500 [01:52<00:00,  4.44it/s]



[ReFT] ===== Eval Metrics =====
  Overall   : 50.80%  (n=500)
  TEMP      : 36.36%  (n=66)
  CRD       : 50.93%  (n=214)
  NPA       : 55.00%  (n=40)
  STA       : 57.96%  (n=157)
  TH        : 34.78%  (n=23)
  Hard      : 34.55%  (n=110)
[ReFT] Metrics saved → ./reft_cinepile_llama3_multilayer/reft_metrics.csv


In [ ]:
# ============================================================
# ReFT (LoReFT) on CinePile — Full Script
#   - 兼容 transformers 5.3.0
#   - 支持多层 LoReFT 干预
#   - ExperimentTracker 记录所有实验
# pip install pyreft datasets accelerate
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime
from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from transformers.modeling_outputs import CausalLMOutputWithPast

import pyreft


# ============================================================
# 1. 配置
# ============================================================
@dataclass
class ReFTConfigCinePile:
    # 数据集
    dataset_name: str               = "tomg-group-umd/cinepile"
    train_split: str                = "train"
    test_split: str                 = "test"
    max_train_samples: Optional[int] = None   # None = 完整训练集
    max_test_samples: Optional[int]  = None   # None = 完整测试集

    # 模型
    base_model_id: str              = "meta-llama/Llama-3.1-8B-Instruct"

    # 输入
    max_scene_length: int           = 1000
    max_seq_length: int             = 1024

    # 训练
    num_train_epochs: float         = 3.0
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float            = 2e-4
    warmup_ratio: float             = 0.05
    bf16: bool                      = True
    logging_steps: int              = 50
    output_dir: str                 = "./reft_full_train"

    # LoReFT 配置
    reft_rank: int                  = 8
    reft_layers: list               = field(default_factory=lambda: [8, 15, 24])
    component: str                  = "block_output"

    # 推理
    max_new_tokens: int             = 5

    # 评估类别
    category_map: dict = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


# ============================================================
# 2. ExperimentTracker
# ============================================================
class ExperimentTracker:
    def __init__(self, log_path: str = "./reft_experiment_log.csv"):
        self.log_path = log_path

    def log(self, cfg: ReFTConfigCinePile, metrics: dict) -> pd.DataFrame:
        record = {
            "timestamp":             datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "base_model":            cfg.base_model_id.split("/")[-1],
            "max_train_samples":     cfg.max_train_samples if cfg.max_train_samples else "full",
            "max_test_samples":      cfg.max_test_samples  if cfg.max_test_samples  else "full",
            "num_train_epochs":      cfg.num_train_epochs,
            "reft_rank":             cfg.reft_rank,
            "reft_layers":           str(cfg.reft_layers),
            "learning_rate":         cfg.learning_rate,
            "batch_size":            cfg.per_device_train_batch_size,
            "grad_accum":            cfg.gradient_accumulation_steps,
            "component":             cfg.component,
            # Accuracy metrics
            "Overall": self._fmt(metrics.get("Overall")),
            "TEMP":    self._fmt(metrics.get("TEMP")),
            "CRD":     self._fmt(metrics.get("CRD")),
            "NPA":     self._fmt(metrics.get("NPA")),
            "STA":     self._fmt(metrics.get("STA")),
            "TH":      self._fmt(metrics.get("TH")),
            "Hard":    self._fmt(metrics.get("Hard")),
            # Sample counts
            "n_test":  metrics.get("Overall_n"),
            "n_TEMP":  metrics.get("TEMP_n"),
            "n_CRD":   metrics.get("CRD_n"),
            "n_NPA":   metrics.get("NPA_n"),
            "n_STA":   metrics.get("STA_n"),
            "n_TH":    metrics.get("TH_n"),
            "n_Hard":  metrics.get("Hard_n"),
            "output_dir": cfg.output_dir,
        }

        df_new = pd.DataFrame([record])
        if os.path.exists(self.log_path):
            df_old = pd.read_csv(self.log_path)
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_all = df_new

        df_all.to_csv(self.log_path, index=False)
        print(f"\n[Tracker] Logged → {self.log_path}  "
              f"(total experiments: {len(df_all)})")
        return df_all

    @staticmethod
    def _fmt(val) -> Optional[float]:
        if val is None:
            return None
        try:
            return round(float(val), 4)
        except (TypeError, ValueError):
            return None

    def show(self) -> Optional[pd.DataFrame]:
        if not os.path.exists(self.log_path):
            print("[Tracker] No experiments logged yet.")
            return None
        df = pd.read_csv(self.log_path)
        cols = [
            "timestamp", "reft_rank", "reft_layers", "learning_rate",
            "max_train_samples", "num_train_epochs",
            "Overall", "TEMP", "CRD", "NPA", "STA", "TH", "Hard",
        ]
        df_show = df[cols].sort_values("Overall", ascending=False)
        print("\n" + "=" * 110)
        print("REFT EXPERIMENT HISTORY  (sorted by Overall ↓)")
        print("=" * 110)
        print(df_show.to_string(index=True))
        print("=" * 110)
        return df_show

    def best(self) -> Optional[pd.Series]:
        if not os.path.exists(self.log_path):
            print("[Tracker] No experiments logged yet.")
            return None
        df  = pd.read_csv(self.log_path)
        row = df.loc[df["Overall"].idxmax()]
        print("\n[Tracker] ★ Best experiment so far:")
        print(row.to_string())
        return row


# ============================================================
# 3. ReFT 模型包装类
# ============================================================
class ReFTModelWrapper(nn.Module):
    """
    把 pyreft.IntervenableModel 包装成标准 HuggingFace forward 接口。
    训练：forward(input_ids, attention_mask, labels) → loss
    推理：generate() 逐步走干预层，保证 train/inference 一致
    """

    def __init__(self, reft_model):
        super().__init__()
        self.reft_model = reft_model
        self.config     = reft_model.model.config

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, **kwargs):
        base = {"input_ids": input_ids, "attention_mask": attention_mask}
        _, counterfactual_outputs = self.reft_model(
            base,
            unit_locations=None,
            output_original_output=False,
        )
        logits = counterfactual_outputs[0]   # (batch, seq_len, vocab)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )
        return CausalLMOutputWithPast(loss=loss, logits=logits)

    def generate(self, input_ids=None, attention_mask=None, **kwargs):
        """推理时也走干预层，保证 train/inference 行为一致"""
        device         = input_ids.device
        generated      = input_ids.clone()
        max_new_tokens = kwargs.get("max_new_tokens", 5)
        eos_id         = kwargs.get("pad_token_id", None)

        for _ in range(max_new_tokens):
            attn = torch.ones_like(generated)
            base = {"input_ids": generated, "attention_mask": attn}
            with torch.no_grad():
                _, outputs = self.reft_model(
                    base,
                    unit_locations=None,
                    output_original_output=False,
                )
            logits    = outputs[0]
            next_tok  = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tok], dim=1)
            if eos_id is not None and (next_tok == eos_id).all():
                break
        return generated

    def save(self, path: str):
        self.reft_model.save(path)


# ============================================================
# 4. 数据集加载
# ============================================================
LETTERS = ["A", "B", "C", "D", "E"]

def answer_text_to_letter(answer_text: str, choices: list) -> str:
    for i, c in enumerate(choices):
        if answer_text.strip() == c.strip():
            return LETTERS[i]
    for i, c in enumerate(choices):
        if answer_text.strip() in c or c.strip() in answer_text:
            return LETTERS[i]
    return "A"

def normalize_hard_split(raw) -> bool:
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() == "true"

def load_cinepile(cfg: ReFTConfigCinePile):
    raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
    raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

    def _convert(ex):
        return {
            "movie_scene":       ex["movie_scene"],
            "question":          ex["question"],
            "choices":           ex["choices"],
            "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
            "question_category": ex["question_category"],
            "hard_split":        normalize_hard_split(ex["hard_split"]),
        }

    train_data = [_convert(ex) for ex in raw_train]
    test_data  = [_convert(ex) for ex in raw_test]

    if cfg.max_train_samples:
        train_data = train_data[:cfg.max_train_samples]
    if cfg.max_test_samples:
        test_data  = test_data[:cfg.max_test_samples]

    print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
    return train_data, test_data


# ============================================================
# 5. Prompt 构建
# ============================================================
def build_inference_prompt(sample, max_scene_length: int) -> str:
    scene   = sample["movie_scene"][:max_scene_length]
    options = "\n".join(
        f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
    )
    return (
        "You are watching a movie. Based on the scene description, "
        "answer the multiple choice question.\n\n"
        f"Scene: {scene}\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


# ============================================================
# 6. 训练数据集（统一 padding）
# ============================================================
def make_supervised_dataset(train_data, tokenizer,
                             cfg: ReFTConfigCinePile) -> Dataset:
    texts = [
        build_inference_prompt(s, cfg.max_scene_length) + " " + s["answer_key"]
        for s in train_data
    ]
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=cfg.max_seq_length,
        padding=False,
    )

    max_len = max(len(ids) for ids in encodings["input_ids"])
    input_ids_list, attention_mask_list, labels_list = [], [], []

    for ids, mask in zip(encodings["input_ids"], encodings["attention_mask"]):
        pad_len = max_len - len(ids)
        input_ids_list.append(ids + [tokenizer.pad_token_id] * pad_len)
        attention_mask_list.append(mask + [0] * pad_len)
        labels_list.append(ids + [-100] * pad_len)

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_mask_list,
        "labels":         labels_list,
    })


# ============================================================
# 7. 评估
# ============================================================
def predict_choice(model_wrapper: ReFTModelWrapper, tokenizer,
                   sample, cfg: ReFTConfigCinePile, device: str) -> str:
    model_wrapper.eval()
    prompt = build_inference_prompt(sample, cfg.max_scene_length)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(device)

    with torch.no_grad():
        outputs = model_wrapper.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().upper()

    for ch in decoded:
        if ch in "ABCDE":
            return ch
    return "A"

def evaluate_reft(model_wrapper: ReFTModelWrapper, tokenizer,
                  test_data: list, cfg: ReFTConfigCinePile) -> dict:
    device  = "cuda" if torch.cuda.is_available() else "cpu"
    records = []
    for s in tqdm(test_data, desc="Eval ReFT"):
        pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
        records.append({
            "pred":              pred,
            "label":             s["answer_key"],
            "question_category": s["question_category"],
            "hard_split":        s["hard_split"],
        })

    df      = pd.DataFrame(records)
    correct = df["pred"] == df["label"]
    metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

    for abbr, keyword in cfg.category_map.items():
        mask = df["question_category"].str.contains(
            keyword, case=False, na=False
        )
        n = int(mask.sum())
        metrics[abbr]        = correct[mask].mean() if n > 0 else None
        metrics[f"{abbr}_n"] = n

    hard_mask = df["hard_split"] == True
    n_hard    = int(hard_mask.sum())
    metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
    metrics["Hard_n"] = n_hard
    return metrics


# ============================================================
# 8. 训练主函数
# ============================================================
def train_reft_with_trainer(cfg: ReFTConfigCinePile,
                             tracker: ExperimentTracker = None):
    os.makedirs(cfg.output_dir, exist_ok=True)
    train_data, test_data = load_cinepile(cfg)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("[ReFT] Loading base model...")
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    # 多层 LoReFT 干预配置
    print(f"[ReFT] LoReFT: layers={cfg.reft_layers}, rank={cfg.reft_rank}")
    representations = [
        {
            "layer":              l,
            "component":          cfg.component,
            "low_rank_dimension": cfg.reft_rank,
            "intervention":       pyreft.LoreftIntervention(
                embed_dim           = base_model.config.hidden_size,
                low_rank_dimension  = cfg.reft_rank,
            ),
        }
        for l in cfg.reft_layers
    ]
    reft_cfg      = pyreft.ReftConfig(representations=representations)
    reft_model    = pyreft.get_reft_model(base_model, reft_cfg)
    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    train_ds = make_supervised_dataset(train_data, tokenizer, cfg)
    print(f"[ReFT] Train samples (after processing): {len(train_ds)}")

    training_args = TrainingArguments(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_train_epochs,
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        learning_rate               = cfg.learning_rate,
        warmup_ratio                = cfg.warmup_ratio,
        bf16                        = cfg.bf16,
        logging_steps               = cfg.logging_steps,
        report_to                   = [],
        save_strategy               = "no",
    )

    trainer = Trainer(
        model         = model_wrapper,
        args          = training_args,
        train_dataset = train_ds,
    )

    print(f"[ReFT] Training... "
          f"epochs={cfg.num_train_epochs}, lr={cfg.learning_rate}, "
          f"rank={cfg.reft_rank}, layers={cfg.reft_layers}")
    trainer.train()

    model_wrapper.save(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"[ReFT] Saved → {cfg.output_dir}")

    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)

    print("\n[ReFT] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "reft_metrics.csv"), index=False
    )

    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 9. 入口
# ============================================================
if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    tracker = ExperimentTracker(log_path="./reft_experiment_log.csv")

    cfg = ReFTConfigCinePile(
        base_model_id            = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples        = None,   # 完整训练集
        max_test_samples         = None,   # 完整测试集
        num_train_epochs         = 3.0,
        per_device_train_batch_size = 2,   # OOM 时改回 2
        gradient_accumulation_steps = 4,
        reft_rank                = 8,
        reft_layers              = [20, 24, 28],  # 全部集中在后层因为 CinePile 的核心难点（TEMP、TH）都是高层推理任务，论文说后层干预理论上更有效
        learning_rate            = 2e-4,
        output_dir               = "./reft_full_train",
    )

    model, tokenizer, metrics = train_reft_with_trainer(cfg, tracker=tracker)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    # 查看所有历史实验
    tracker.show()
    tracker.best()


transformers: 5.3.0
[Dataset] Train: 298888 | Test: 4941
[ReFT] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[ReFT] LoReFT: layers=[20, 24, 28], rank=8


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[ReFT] Train samples (after processing): 298888
[ReFT] Training... epochs=3.0, lr=0.0002, rank=8, layers=[20, 24, 28]


Step,Training Loss
50,9.556777
100,9.384412
150,9.172968
200,9.005863
250,8.623847
300,8.231758
350,7.941486
400,7.673420
450,7.540659
500,7.525275


Step,Training Loss
50,9.556777
100,9.384412
150,9.172968
200,9.005863
250,8.623847
300,8.231758
350,7.941486
400,7.673420
450,7.540659
500,7.525275


## 2.1 ReFT

In [ ]:
!pip install "transformers==4.39.3" "accelerate==0.27.2" -U
!pip install "pyreft==0.0.7" -U
!pip install datasets bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installation: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 创建项目根目录
import os
PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_ReFT"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project dir: {PROJECT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/266_final/phase2/CinePile_ReFT


In [ ]:
# # ============================================================
# # ReFT (LoReFT) on CinePile — Full Script with Stratified Sampling
# #   - 兼容 transformers 5.3.0
# #   - 分层抽样：按 question_category × hard_split
# #   - ExperimentTracker 记录所有实验
# #   - Checkpoint 保存与续训
# # pip install pyreft datasets accelerate
# # ============================================================

# import os
# import gc
# import math
# import torch
# import torch.nn as nn
# import pandas as pd
# from collections import defaultdict
# from dataclasses import dataclass, field
# from datetime import datetime
# from typing import Optional
# from datasets import load_dataset, Dataset
# from tqdm import tqdm

# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     TrainingArguments,
#     Trainer,
# )
# from transformers.modeling_outputs import CausalLMOutputWithPast

# import pyreft


# # ============================================================
# # 1. 配置
# # ============================================================
# @dataclass
# class ReFTConfigCinePile:
#     # 数据集
#     dataset_name: str                = "tomg-group-umd/cinepile"
#     train_split: str                 = "train"
#     test_split: str                  = "test"

#     # 分层抽样（None = 不抽样，使用全量）
#     max_train_samples: Optional[int] = 30000
#     max_test_samples: Optional[int]  = None    # 测试集全量

#     # 分层权重：hard 样本的抽样倍率（相对于 easy 样本）
#     # 1.0 = 完全按比例；2.0 = hard 样本抽取 2 倍比例
#     hard_oversample_ratio: float     = 1.5

#     # 模型
#     base_model_id: str               = "meta-llama/Llama-3.1-8B-Instruct"

#     # 输入
#     max_scene_length: int            = 1000
#     max_seq_length: int              = 1024

#     # 训练
#     num_train_epochs: float          = 3.0
#     per_device_train_batch_size: int = 4
#     gradient_accumulation_steps: int = 4
#     learning_rate: float             = 2e-4
#     warmup_ratio: float              = 0.05
#     bf16: bool                       = True
#     logging_steps: int               = 50
#     save_steps: int                  = 200
#     save_total_limit: int            = 3
#     output_dir: str                  = "./reft_cinepile"

#     # LoReFT
#     reft_rank: int                   = 8
#     reft_layers: list                = field(default_factory=lambda: [20, 24, 28])
#     component: str                   = "block_output"

#     # 推理
#     max_new_tokens: int              = 5

#     # 评估类别
#     category_map: dict = field(default_factory=lambda: {
#         "TEMP": "Temporal",
#         "CRD":  "Character and",
#         "NPA":  "Narrative and",
#         "STA":  "Setting and",
#         "TH":   "Theme Exploration",
#     })


# # ============================================================
# # 2. 分层抽样器
# # ============================================================
# class StratifiedSampler:
#     """
#     按 question_category × hard_split 分层抽样。

#     设计逻辑：
#     1. 把所有样本按 (category, hard) 分组
#     2. 计算每组的目标抽样数：
#        - 基础配额 = 目标总量 / 层数（均匀分配）
#        - hard 层额外乘以 hard_oversample_ratio
#        - 若某组样本不足，按实际有多少取多少
#     3. 打印抽样分布报告，方便验证
#     """

#     def __init__(self, cfg: ReFTConfigCinePile):
#         self.cfg = cfg

#     def sample(self, data: list) -> list:
#         if self.cfg.max_train_samples is None:
#             print(f"[Sampler] Using full dataset: {len(data)} samples")
#             return data

#         target = self.cfg.max_train_samples

#         # 按 (category_key, hard) 分组
#         groups = defaultdict(list)
#         for s in data:
#             cat  = self._get_category_key(s["question_category"])
#             hard = bool(s["hard_split"])
#             groups[(cat, hard)].append(s)

#         # 计算各层权重
#         n_categories = len(set(k[0] for k in groups))
#         n_strata     = n_categories * 2   # × hard/easy

#         # 先按比例算 easy 的基础配额
#         base_quota = target / (
#             n_categories * (1 + self.cfg.hard_oversample_ratio)
#         )
#         hard_quota  = base_quota * self.cfg.hard_oversample_ratio
#         easy_quota  = base_quota

#         sampled = []
#         report  = []

#         for (cat, hard), group in sorted(groups.items()):
#             quota   = int(hard_quota if hard else easy_quota)
#             n_take  = min(quota, len(group))
#             # 固定随机种子保证可复现
#             import random
#             random.seed(42)
#             taken   = random.sample(group, n_take)
#             sampled.extend(taken)
#             report.append({
#                 "category":  cat,
#                 "hard":      hard,
#                 "available": len(group),
#                 "quota":     quota,
#                 "sampled":   n_take,
#             })

#         # 如果抽到的总量不足 target，从剩余样本里补足
#         if len(sampled) < target:
#             sampled_ids = set(id(s) for s in sampled)
#             remaining   = [s for s in data if id(s) not in sampled_ids]
#             import random
#             random.seed(99)
#             extra = random.sample(remaining, min(target - len(sampled), len(remaining)))
#             sampled.extend(extra)

#         self._print_report(report, len(sampled))
#         return sampled

#     def _get_category_key(self, category_str: str) -> str:
#         for abbr, keyword in self.cfg.category_map.items():
#             if keyword.lower() in category_str.lower():
#                 return abbr
#         return "OTHER"

#     @staticmethod
#     def _print_report(report: list, total_sampled: int):
#         df = pd.DataFrame(report)
#         print("\n[Sampler] ===== Stratified Sampling Report =====")
#         print(df.to_string(index=False))
#         print(f"[Sampler] Total sampled: {total_sampled}")
#         print("[Sampler] " + "=" * 45)


# # ============================================================
# # 3. ExperimentTracker
# # ============================================================
# class ExperimentTracker:
#     def __init__(self, log_path: str = "./reft_experiment_log.csv"):
#         self.log_path = log_path

#     def log(self, cfg: ReFTConfigCinePile, metrics: dict) -> pd.DataFrame:
#         record = {
#             "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#             "base_model":          cfg.base_model_id.split("/")[-1],
#             "max_train_samples":   cfg.max_train_samples if cfg.max_train_samples else "full",
#             "hard_oversample":     cfg.hard_oversample_ratio,
#             "num_train_epochs":    cfg.num_train_epochs,
#             "reft_rank":           cfg.reft_rank,
#             "reft_layers":         str(cfg.reft_layers),
#             "learning_rate":       cfg.learning_rate,
#             "batch_size":          cfg.per_device_train_batch_size,
#             "grad_accum":          cfg.gradient_accumulation_steps,
#             "Overall":  self._fmt(metrics.get("Overall")),
#             "TEMP":     self._fmt(metrics.get("TEMP")),
#             "CRD":      self._fmt(metrics.get("CRD")),
#             "NPA":      self._fmt(metrics.get("NPA")),
#             "STA":      self._fmt(metrics.get("STA")),
#             "TH":       self._fmt(metrics.get("TH")),
#             "Hard":     self._fmt(metrics.get("Hard")),
#             "n_test":   metrics.get("Overall_n"),
#             "n_TEMP":   metrics.get("TEMP_n"),
#             "n_CRD":    metrics.get("CRD_n"),
#             "n_NPA":    metrics.get("NPA_n"),
#             "n_STA":    metrics.get("STA_n"),
#             "n_TH":     metrics.get("TH_n"),
#             "n_Hard":   metrics.get("Hard_n"),
#             "output_dir": cfg.output_dir,
#         }
#         df_new = pd.DataFrame([record])
#         if os.path.exists(self.log_path):
#             df_old = pd.read_csv(self.log_path)
#             df_all = pd.concat([df_old, df_new], ignore_index=True)
#         else:
#             df_all = df_new
#         df_all.to_csv(self.log_path, index=False)
#         print(f"\n[Tracker] Logged → {self.log_path}  "
#               f"(total: {len(df_all)} experiments)")
#         return df_all

#     @staticmethod
#     def _fmt(val) -> Optional[float]:
#         if val is None:
#             return None
#         try:
#             return round(float(val), 4)
#         except (TypeError, ValueError):
#             return None

#     def show(self) -> Optional[pd.DataFrame]:
#         if not os.path.exists(self.log_path):
#             print("[Tracker] No experiments logged yet.")
#             return None
#         df = pd.read_csv(self.log_path)
#         cols = [
#             "timestamp", "reft_rank", "reft_layers", "learning_rate",
#             "max_train_samples", "hard_oversample", "num_train_epochs",
#             "Overall", "TEMP", "CRD", "NPA", "STA", "TH", "Hard",
#         ]
#         df_show = df[cols].sort_values("Overall", ascending=False)
#         print("\n" + "=" * 120)
#         print("REFT EXPERIMENT HISTORY  (sorted by Overall ↓)")
#         print("=" * 120)
#         print(df_show.to_string(index=True))
#         print("=" * 120)
#         return df_show

#     def best(self) -> Optional[pd.Series]:
#         if not os.path.exists(self.log_path):
#             print("[Tracker] No experiments logged yet.")
#             return None
#         df  = pd.read_csv(self.log_path)
#         row = df.loc[df["Overall"].idxmax()]
#         print("\n[Tracker] ★ Best experiment so far:")
#         print(row.to_string())
#         return row


# # ============================================================
# # 4. ReFT 模型包装类
# # ============================================================
# class ReFTModelWrapper(nn.Module):
#     def __init__(self, reft_model):
#         super().__init__()
#         self.reft_model = reft_model
#         self.config     = reft_model.model.config

#     def forward(self, input_ids=None, attention_mask=None,
#                 labels=None, **kwargs):
#         base = {"input_ids": input_ids, "attention_mask": attention_mask}
#         _, counterfactual_outputs = self.reft_model(
#             base,
#             unit_locations=None,
#             output_original_output=False,
#         )
#         logits = counterfactual_outputs[0]

#         loss = None
#         if labels is not None:
#             shift_logits = logits[..., :-1, :].contiguous()
#             shift_labels = labels[..., 1:].contiguous()
#             loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
#             loss = loss_fn(
#                 shift_logits.view(-1, shift_logits.size(-1)),
#                 shift_labels.view(-1),
#             )
#         return CausalLMOutputWithPast(loss=loss, logits=logits)

#     def generate(self, input_ids=None, attention_mask=None, **kwargs):
#         device         = input_ids.device
#         generated      = input_ids.clone()
#         max_new_tokens = kwargs.get("max_new_tokens", 5)
#         eos_id         = kwargs.get("pad_token_id", None)

#         for _ in range(max_new_tokens):
#             attn = torch.ones_like(generated)
#             base = {"input_ids": generated, "attention_mask": attn}
#             with torch.no_grad():
#                 _, outputs = self.reft_model(
#                     base,
#                     unit_locations=None,
#                     output_original_output=False,
#                 )
#             logits    = outputs[0]
#             next_tok  = logits[:, -1, :].argmax(dim=-1, keepdim=True)
#             generated = torch.cat([generated, next_tok], dim=1)
#             if eos_id is not None and (next_tok == eos_id).all():
#                 break
#         return generated

#     def save(self, path: str):
#         self.reft_model.save(path)


# # ============================================================
# # 5. 数据加载
# # ============================================================
# LETTERS = ["A", "B", "C", "D", "E"]

# def answer_text_to_letter(answer_text: str, choices: list) -> str:
#     for i, c in enumerate(choices):
#         if answer_text.strip() == c.strip():
#             return LETTERS[i]
#     for i, c in enumerate(choices):
#         if answer_text.strip() in c or c.strip() in answer_text:
#             return LETTERS[i]
#     return "A"

# def normalize_hard_split(raw) -> bool:
#     if isinstance(raw, bool):
#         return raw
#     return str(raw).strip().lower() == "true"

# def load_cinepile(cfg: ReFTConfigCinePile):
#     raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
#     raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

#     def _convert(ex):
#         return {
#             "movie_scene":       ex["movie_scene"],
#             "question":          ex["question"],
#             "choices":           ex["choices"],
#             "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
#             "question_category": ex["question_category"],
#             "hard_split":        normalize_hard_split(ex["hard_split"]),
#         }

#     train_data = [_convert(ex) for ex in raw_train]
#     test_data  = [_convert(ex) for ex in raw_test]

#     # 训练集分层抽样
#     sampler    = StratifiedSampler(cfg)
#     train_data = sampler.sample(train_data)

#     # 测试集（可选截断）
#     if cfg.max_test_samples:
#         test_data = test_data[:cfg.max_test_samples]

#     print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
#     return train_data, test_data


# # ============================================================
# # 6. Prompt 与数据集
# # ============================================================
# def build_inference_prompt(sample, max_scene_length: int) -> str:
#     scene   = sample["movie_scene"][:max_scene_length]
#     options = "\n".join(
#         f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
#     )
#     return (
#         "You are watching a movie. Based on the scene description, "
#         "answer the multiple choice question.\n\n"
#         f"Scene: {scene}\n\n"
#         f"Question: {sample['question']}\n\n"
#         f"Options:\n{options}\n\n"
#         "Answer:"
#     )

# def make_supervised_dataset(train_data, tokenizer,
#                              cfg: ReFTConfigCinePile) -> Dataset:
#     texts = [
#         build_inference_prompt(s, cfg.max_scene_length) + " " + s["answer_key"]
#         for s in train_data
#     ]
#     encodings = tokenizer(
#         texts,
#         truncation=True,
#         max_length=cfg.max_seq_length,
#         padding=False,
#     )
#     max_len = max(len(ids) for ids in encodings["input_ids"])
#     input_ids_list, attention_mask_list, labels_list = [], [], []

#     for ids, mask in zip(encodings["input_ids"], encodings["attention_mask"]):
#         pad_len = max_len - len(ids)
#         input_ids_list.append(ids + [tokenizer.pad_token_id] * pad_len)
#         attention_mask_list.append(mask + [0] * pad_len)
#         labels_list.append(ids + [-100] * pad_len)

#     return Dataset.from_dict({
#         "input_ids":      input_ids_list,
#         "attention_mask": attention_mask_list,
#         "labels":         labels_list,
#     })


# # ============================================================
# # 7. 评估
# # ============================================================
# def predict_choice(model_wrapper, tokenizer, sample,
#                    cfg: ReFTConfigCinePile, device: str) -> str:
#     model_wrapper.eval()
#     prompt = build_inference_prompt(sample, cfg.max_scene_length)
#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=cfg.max_seq_length,
#     ).to(device)

#     with torch.no_grad():
#         outputs = model_wrapper.generate(
#             **inputs,
#             max_new_tokens=cfg.max_new_tokens,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#     decoded = tokenizer.decode(
#         outputs[0][inputs["input_ids"].shape[1]:],
#         skip_special_tokens=True,
#     ).strip().upper()

#     for ch in decoded:
#         if ch in "ABCDE":
#             return ch
#     return "A"

# def evaluate_reft(model_wrapper, tokenizer,
#                   test_data: list, cfg: ReFTConfigCinePile) -> dict:
#     device  = "cuda" if torch.cuda.is_available() else "cpu"
#     records = []
#     for s in tqdm(test_data, desc="Eval ReFT"):
#         pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
#         records.append({
#             "pred":              pred,
#             "label":             s["answer_key"],
#             "question_category": s["question_category"],
#             "hard_split":        s["hard_split"],
#         })

#     df      = pd.DataFrame(records)
#     correct = df["pred"] == df["label"]
#     metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

#     for abbr, keyword in cfg.category_map.items():
#         mask = df["question_category"].str.contains(keyword, case=False, na=False)
#         n    = int(mask.sum())
#         metrics[abbr]        = correct[mask].mean() if n > 0 else None
#         metrics[f"{abbr}_n"] = n

#     hard_mask = df["hard_split"] == True
#     n_hard    = int(hard_mask.sum())
#     metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
#     metrics["Hard_n"] = n_hard
#     return metrics


# # ============================================================
# # 8. 工具：获取最新 checkpoint
# # ============================================================
# def get_latest_checkpoint(output_dir: str) -> Optional[str]:
#     if not os.path.exists(output_dir):
#         return None
#     checkpoints = [
#         os.path.join(output_dir, d)
#         for d in os.listdir(output_dir)
#         if d.startswith("checkpoint-")
#     ]
#     if not checkpoints:
#         return None
#     latest = max(checkpoints, key=lambda x: int(x.split("-")[-1]))
#     print(f"[Resume] Latest checkpoint found: {latest}")
#     return latest


# # ============================================================
# # 9. 训练主函数
# # ============================================================
# def train_reft_with_trainer(
#     cfg: ReFTConfigCinePile,
#     tracker: ExperimentTracker = None,
#     resume_from_checkpoint: str = None,
# ):
#     os.makedirs(cfg.output_dir, exist_ok=True)
#     train_data, test_data = load_cinepile(cfg)

#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     print("[ReFT] Loading base model...")
#     tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token
#     tokenizer.padding_side = "right"

#     base_model = AutoModelForCausalLM.from_pretrained(
#         cfg.base_model_id,
#         dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
#         device_map=device,
#     )

#     print(f"[ReFT] LoReFT: layers={cfg.reft_layers}, rank={cfg.reft_rank}")
#     representations = [
#         {
#             "layer":              l,
#             "component":          cfg.component,
#             "low_rank_dimension": cfg.reft_rank,
#             "intervention":       pyreft.LoreftIntervention(
#                 embed_dim          = base_model.config.hidden_size,
#                 low_rank_dimension = cfg.reft_rank,
#             ),
#         }
#         for l in cfg.reft_layers
#     ]
#     reft_cfg   = pyreft.ReftConfig(representations=representations)
#     reft_model = pyreft.get_reft_model(base_model, reft_cfg)

#     if resume_from_checkpoint is not None:
#         print(f"[ReFT] Loading LoReFT weights from: {resume_from_checkpoint}")
#         reft_model.load(resume_from_checkpoint)

#     reft_model.set_device(device)
#     model_wrapper = ReFTModelWrapper(reft_model)
#     train_ds      = make_supervised_dataset(train_data, tokenizer, cfg)
#     print(f"[ReFT] Train dataset size: {len(train_ds)}")

#     training_args = TrainingArguments(
#         output_dir                  = cfg.output_dir,
#         num_train_epochs            = cfg.num_train_epochs,
#         per_device_train_batch_size = cfg.per_device_train_batch_size,
#         gradient_accumulation_steps = cfg.gradient_accumulation_steps,
#         learning_rate               = cfg.learning_rate,
#         warmup_ratio                = cfg.warmup_ratio,
#         bf16                        = cfg.bf16,
#         logging_steps               = cfg.logging_steps,
#         report_to                   = [],
#         save_strategy               = "no",
#         save_steps                  = cfg.save_steps,
#         save_total_limit            = cfg.save_total_limit,
#     )

#     trainer = Trainer(
#         model         = model_wrapper,
#         args          = training_args,
#         train_dataset = train_ds,
#     )

#     print(f"[ReFT] Training... epochs={cfg.num_train_epochs}, "
#           f"lr={cfg.learning_rate}, rank={cfg.reft_rank}, layers={cfg.reft_layers}")
#     trainer.train(resume_from_checkpoint=resume_from_checkpoint)

#     model_wrapper.save(cfg.output_dir)
#     tokenizer.save_pretrained(cfg.output_dir)
#     print(f"[ReFT] Final model saved → {cfg.output_dir}")

#     metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)
#     print("\n[ReFT] ===== Eval Metrics =====")
#     for k, v in metrics.items():
#         if not k.endswith("_n") and v is not None:
#             n = metrics.get(f"{k}_n", "")
#             print(f"  {k:<10}: {v:.2%}  (n={n})")

#     pd.DataFrame([metrics]).to_csv(
#         os.path.join(cfg.output_dir, "reft_metrics.csv"), index=False
#     )
#     if tracker is not None:
#         tracker.log(cfg, metrics)

#     return model_wrapper, tokenizer, metrics


# # ============================================================
# # 10. 加载已保存模型推理
# # ============================================================
# def load_and_evaluate(
#     output_dir: str,
#     cfg: ReFTConfigCinePile,
#     tracker: ExperimentTracker = None,
#     checkpoint: str = None,
# ):
#     device   = "cuda" if torch.cuda.is_available() else "cpu"
#     load_dir = checkpoint if checkpoint is not None else output_dir

#     print(f"[Load] Tokenizer ← {output_dir}")
#     tokenizer = AutoTokenizer.from_pretrained(output_dir)
#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token
#     tokenizer.padding_side = "right"

#     print(f"[Load] Base model ← {cfg.base_model_id}")
#     base_model = AutoModelForCausalLM.from_pretrained(
#         cfg.base_model_id,
#         dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
#         device_map=device,
#     )

#     representations = [
#         {
#             "layer":              l,
#             "component":          cfg.component,
#             "low_rank_dimension": cfg.reft_rank,
#             "intervention":       pyreft.LoreftIntervention(
#                 embed_dim          = base_model.config.hidden_size,
#                 low_rank_dimension = cfg.reft_rank,
#             ),
#         }
#         for l in cfg.reft_layers
#     ]
#     reft_cfg   = pyreft.ReftConfig(representations=representations)
#     reft_model = pyreft.get_reft_model(base_model, reft_cfg)

#     print(f"[Load] LoReFT weights ← {load_dir}")
#     reft_model.load(load_dir)
#     reft_model.set_device(device)
#     model_wrapper = ReFTModelWrapper(reft_model)

#     _, test_data = load_cinepile(cfg)
#     metrics      = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)

#     print("\n[Load] ===== Eval Metrics =====")
#     for k, v in metrics.items():
#         if not k.endswith("_n") and v is not None:
#             n = metrics.get(f"{k}_n", "")
#             print(f"  {k:<10}: {v:.2%}  (n={n})")

#     save_name = (f"reft_metrics_{os.path.basename(load_dir)}.csv"
#                  if checkpoint else "reft_metrics_reeval.csv")
#     pd.DataFrame([metrics]).to_csv(
#         os.path.join(output_dir, save_name), index=False
#     )
#     if tracker is not None:
#         tracker.log(cfg, metrics)

#     return model_wrapper, tokenizer, metrics


# # ============================================================
# # 11. 入口
# # ============================================================
# if __name__ == "__main__":
#     import transformers
#     print("transformers:", transformers.__version__)

#     # Drive 路径
#     PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_ReFT"
#     os.makedirs(PROJECT_DIR, exist_ok=True)

#     # Tracker 保存到 Drive
#     tracker = ExperimentTracker(
#         log_path=f"{PROJECT_DIR}/reft_experiment_log.csv"
#     )

#     cfg = ReFTConfigCinePile(
#         base_model_id               = "meta-llama/Llama-3.1-8B-Instruct",
#         max_train_samples           = 300,   # =None 完整训练集
#         max_test_samples            = 100,   # =None 完整测试集
#         hard_oversample_ratio       = 1.5,
#         num_train_epochs            = 3.0,
#         per_device_train_batch_size = 4,  # OOM 时改回 2
#         gradient_accumulation_steps = 4,
#         reft_rank                   = 8,
#         reft_layers                 = [20, 24, 28],
#         learning_rate               = 2e-4,
#         save_steps                  = 200,
#         save_total_limit            = 3,
#         # 模型和 checkpoint 直接保存到 Drive
#         output_dir                  = f"{PROJECT_DIR}/reft_30k_stratified",
#     )

#     # 正常训练
#     model, tokenizer, metrics = train_reft_with_trainer(cfg, tracker=tracker)

#     # # 中断续训（取消注释）
#     # latest = get_latest_checkpoint(cfg.output_dir)
#     # model, tokenizer, metrics = train_reft_with_trainer(
#     #     cfg, tracker=tracker, resume_from_checkpoint=latest
#     # )

#     # 只做推理（取消注释）
#     # model, tokenizer, metrics = load_and_evaluate(
#     #     output_dir = cfg.output_dir,
#     #     cfg        = cfg,
#     #     tracker    = tracker,
#     # )

#     del model, tokenizer
#     gc.collect()
#     torch.cuda.empty_cache()

#     tracker.show()
#     tracker.best()



transformers: 5.3.0

[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127468     24       24
     NPA False      32238     24       24
     STA False      85604     24       24
    TEMP False      41517     24       24
      TH False      12061     24       24
[Sampler] Total sampled: 300
[Sampler] =============================================
[Dataset] Train: 300 | Test: 100
[ReFT] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[ReFT] LoReFT: layers=[20, 24, 28], rank=8


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[ReFT] Train dataset size: 300
[ReFT] Training... epochs=3.0, lr=0.0002, rank=8, layers=[20, 24, 28]


Step,Training Loss
50,7.864659


Directory '/content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_30k_stratified' already exists.
[ReFT] Final model saved → /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_30k_stratified


Eval ReFT: 100%|██████████| 100/100 [00:22<00:00,  4.50it/s]



[ReFT] ===== Eval Metrics =====
  Overall   : 50.00%  (n=100)
  TEMP      : 30.77%  (n=13)
  CRD       : 52.38%  (n=42)
  NPA       : 60.00%  (n=5)
  STA       : 57.58%  (n=33)
  TH        : 28.57%  (n=7)
  Hard      : 53.85%  (n=26)

[Tracker] Logged → /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_experiment_log.csv  (total: 1 experiments)

REFT EXPERIMENT HISTORY  (sorted by Overall ↓)
             timestamp  reft_rank   reft_layers  learning_rate  max_train_samples  hard_oversample  num_train_epochs  Overall    TEMP     CRD  NPA     STA      TH    Hard
0  2026-03-07 07:57:32          8  [20, 24, 28]         0.0002                300              1.5               3.0      0.5  0.3077  0.5238  0.6  0.5758  0.2857  0.5385

[Tracker] ★ Best experiment so far:
timestamp                                          2026-03-07 07:57:32
base_model                                       Llama-3.1-8B-Instruct
max_train_samples                                                  300
har

## 2.1 ReFT

In [ ]:
# ============================================================
# ReFT (LoReFT) on CinePile — Compact + Stratified + Logging
#   - transformers 5.3.0
#   - Stratified sampling on CinePile (category × hard)
#   - Only save small LoReFT weights (no big model checkpoints)
#   - Track used sample_ids to avoid re-training on same samples
#   - Save training_log_history.csv for loss analysis
# pip install pyreft datasets accelerate
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Set
from datetime import datetime
from collections import defaultdict

from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from transformers.modeling_outputs import CausalLMOutputWithPast

import pyreft


# ============================================================
# 1. 配置
# ============================================================
@dataclass
class ReFTConfigCinePile:
    # 数据集
    dataset_name: str                = "tomg-group-umd/cinepile"
    train_split: str                 = "train"
    test_split: str                  = "test"

    # 分层抽样（None = 用完整 split）
    max_train_samples: Optional[int] = 30000   # e.g. 从 ~30 万中抽 3 万
    max_test_samples: Optional[int]  = 500     # 测试集可截断或 full

    # hard 样本在分层抽样中的额外倍率
    hard_oversample_ratio: float     = 1.5

    # 模型
    base_model_id: str               = "meta-llama/Llama-3.1-8B-Instruct"

    # 输入
    max_scene_length: int            = 1000
    max_seq_length: int              = 1024

    # 训练
    num_train_epochs: float          = 3.0
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float             = 2e-4
    warmup_ratio: float              = 0.05
    bf16: bool                       = True
    logging_steps: int               = 50
    output_dir: str                  = "./reft_cinepile"  # LoReFT 小权重保存目录

    # LoReFT 配置
    reft_rank: int                   = 8
    reft_layers: List[int]           = field(default_factory=lambda: [20, 24, 28])
    component: str                   = "block_output"

    # 推理生成
    max_new_tokens: int              = 5

    # 评估类别
    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


# ============================================================
# 2. Stratified Sampler（按类别 × hard 分层抽样）
# ============================================================
class StratifiedSampler:
    def __init__(self, cfg: ReFTConfigCinePile):
        self.cfg = cfg

    def sample(self, data: List[dict]) -> List[dict]:
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples
        groups = defaultdict(list)

        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q       = base_quota
        hard_q       = base_quota * self.cfg.hard_oversample_ratio

        sampled = []
        report  = []

        import random
        random.seed(42)

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        self._print_report(report, len(sampled))
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"

    @staticmethod
    def _print_report(report: List[dict], total_sampled: int):
        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {total_sampled}")
        print("[Sampler] " + "=" * 60)


# ============================================================
# 3. ExperimentTracker
# ============================================================
class ExperimentTracker:
    def __init__(self, log_path: str = "./reft_experiment_log.csv"):
        self.log_path = log_path

    def log(self, cfg: ReFTConfigCinePile, metrics: dict) -> pd.DataFrame:
        record = {
            "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "base_model":          cfg.base_model_id.split("/")[-1],
            "max_train_samples":   cfg.max_train_samples if cfg.max_train_samples else "full",
            "max_test_samples":    cfg.max_test_samples  if cfg.max_test_samples  else "full",
            "hard_oversample":     cfg.hard_oversample_ratio,
            "num_train_epochs":    cfg.num_train_epochs,
            "reft_rank":           cfg.reft_rank,
            "reft_layers":         str(cfg.reft_layers),
            "learning_rate":       cfg.learning_rate,
            "batch_size":          cfg.per_device_train_batch_size,
            "grad_accum":          cfg.gradient_accumulation_steps,
            "Overall":             self._fmt(metrics.get("Overall")),
            "TEMP":                self._fmt(metrics.get("TEMP")),
            "CRD":                 self._fmt(metrics.get("CRD")),
            "NPA":                 self._fmt(metrics.get("NPA")),
            "STA":                 self._fmt(metrics.get("STA")),
            "TH":                  self._fmt(metrics.get("TH")),
            "Hard":                self._fmt(metrics.get("Hard")),
            "n_test":              metrics.get("Overall_n"),
            "n_TEMP":              metrics.get("TEMP_n"),
            "n_CRD":               metrics.get("CRD_n"),
            "n_NPA":               metrics.get("NPA_n"),
            "n_STA":               metrics.get("STA_n"),
            "n_TH":                metrics.get("TH_n"),
            "n_Hard":              metrics.get("Hard_n"),
            "output_dir":          cfg.output_dir,
        }
        df_new = pd.DataFrame([record])
        if os.path.exists(self.log_path):
            df_old = pd.read_csv(self.log_path)
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_all = df_new
        df_all.to_csv(self.log_path, index=False)
        print(f"\n[Tracker] Logged → {self.log_path}  (total: {len(df_all)})")
        return df_all

    @staticmethod
    def _fmt(val):
        if val is None:
            return None
        try:
            return round(float(val), 4)
        except Exception:
            return None


# ============================================================
# 4. ReFT 模型包装类
# ============================================================
class ReFTModelWrapper(nn.Module):
    def __init__(self, reft_model):
        super().__init__()
        self.reft_model = reft_model
        self.config     = reft_model.model.config

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, **kwargs):
        base = {"input_ids": input_ids, "attention_mask": attention_mask}
        _, counterfactual_outputs = self.reft_model(
            base,
            unit_locations=None,
            output_original_output=False,
        )
        logits = counterfactual_outputs[0]

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )
        return CausalLMOutputWithPast(loss=loss, logits=logits)

    def generate(self, input_ids=None, attention_mask=None, **kwargs):
        device         = input_ids.device
        generated      = input_ids.clone()
        max_new_tokens = kwargs.get("max_new_tokens", 5)
        eos_id         = kwargs.get("pad_token_id", None)

        for _ in range(max_new_tokens):
            attn = torch.ones_like(generated)
            base = {"input_ids": generated, "attention_mask": attn}
            with torch.no_grad():
                _, outputs = self.reft_model(
                    base,
                    unit_locations=None,
                    output_original_output=False,
                )
            logits    = outputs[0]
            next_tok  = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tok], dim=1)
            if eos_id is not None and (next_tok == eos_id).all():
                break
        return generated

    def save(self, path: str):
        self.reft_model.save(path)


# ============================================================
# 5. 数据加载（带 sample_id，用于避免重复样本）
# ============================================================
LETTERS = ["A", "B", "C", "D", "E"]

def answer_text_to_letter(answer_text: str, choices: list) -> str:
    for i, c in enumerate(choices):
        if answer_text.strip() == c.strip():
            return LETTERS[i]
    for i, c in enumerate(choices):
        if answer_text.strip() in c or c.strip() in answer_text:
            return LETTERS[i]
    return "A"

def normalize_hard_split(raw) -> bool:
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() == "true"

def load_cinepile(
    cfg: ReFTConfigCinePile,
    seen_ids: Optional[Set[int]] = None,
):
    raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
    raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

    def _convert(ex, idx):
        return {
            "sample_id":         idx,  # 稳定 ID
            "movie_scene":       ex["movie_scene"],
            "question":          ex["question"],
            "choices":           ex["choices"],
            "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
            "question_category": ex["question_category"],
            "hard_split":        normalize_hard_split(ex["hard_split"]),
        }

    train_data = [_convert(ex, i) for i, ex in enumerate(raw_train)]
    test_data  = [_convert(ex, i) for i, ex in enumerate(raw_test)]

    if seen_ids is not None and len(seen_ids) > 0:
        before = len(train_data)
        train_data = [s for s in train_data if s["sample_id"] not in seen_ids]
        print(f"[Dataset] Filtered seen samples: {before} → {len(train_data)}")

    sampler    = StratifiedSampler(cfg)
    train_data = sampler.sample(train_data)

    if cfg.max_test_samples:
        test_data = test_data[:cfg.max_test_samples]

    print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
    return train_data, test_data


# ============================================================
# 6. Prompt & 训练数据集
# ============================================================
def build_inference_prompt(sample, max_scene_length: int) -> str:
    scene   = sample["movie_scene"][:max_scene_length]
    options = "\n".join(
        f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
    )
    return (
        "You are watching a movie. Based on the scene description, "
        "answer the multiple choice question.\n\n"
        f"Scene: {scene}\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )

def make_supervised_dataset(train_data, tokenizer,
                             cfg: ReFTConfigCinePile) -> Dataset:
    texts = [
        build_inference_prompt(s, cfg.max_scene_length) + " " + s["answer_key"]
        for s in train_data
    ]
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=cfg.max_seq_length,
        padding=False,
    )
    max_len = max(len(ids) for ids in encodings["input_ids"])
    input_ids_list, attention_mask_list, labels_list = [], [], []

    for ids, mask in zip(encodings["input_ids"], encodings["attention_mask"]):
        pad_len = max_len - len(ids)
        input_ids_list.append(ids + [tokenizer.pad_token_id] * pad_len)
        attention_mask_list.append(mask + [0] * pad_len)
        labels_list.append(ids + [-100] * pad_len)

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_mask_list,
        "labels":         labels_list,
    })


# ============================================================
# 7. 评估
# ============================================================
def predict_choice(model_wrapper, tokenizer, sample,
                   cfg: ReFTConfigCinePile, device: str) -> str:
    model_wrapper.eval()
    prompt = build_inference_prompt(sample, cfg.max_scene_length)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(device)

    with torch.no_grad():
        outputs = model_wrapper.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().upper()

    for ch in decoded:
        if ch in "ABCDE":
            return ch
    return "A"

def evaluate_reft(model_wrapper, tokenizer,
                  test_data: list, cfg: ReFTConfigCinePile) -> dict:
    device  = "cuda" if torch.cuda.is_available() else "cpu"
    records = []
    for s in tqdm(test_data, desc="Eval ReFT"):
        pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
        records.append({
            "pred":              pred,
            "label":             s["answer_key"],
            "question_category": s["question_category"],
            "hard_split":        s["hard_split"],
        })

    df      = pd.DataFrame(records)
    correct = df["pred"] == df["label"]
    metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

    for abbr, keyword in cfg.category_map.items():
        mask = df["question_category"].str.contains(keyword, case=False, na=False)
        n    = int(mask.sum())
        metrics[abbr]        = correct[mask].mean() if n > 0 else None
        metrics[f"{abbr}_n"] = n

    hard_mask = df["hard_split"] == True
    n_hard    = int(hard_mask.sum())
    metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
    metrics["Hard_n"] = n_hard
    return metrics


# ============================================================
# 8. 训练主函数（不保存大模型，记录每步日志）
# ============================================================
def train_reft_with_trainer(
    cfg: ReFTConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
    resume_from_loreft_dir: Optional[str] = None,
    used_ids_path: str = "./used_sample_ids.txt",
):
    os.makedirs(cfg.output_dir, exist_ok=True)

    # 1) 读取已使用样本 ID
    seen_ids: Set[int] = set()
    if os.path.exists(used_ids_path):
        with open(used_ids_path, "r") as f:
            for line in f:
                try:
                    seen_ids.add(int(line.strip()))
                except ValueError:
                    continue
        print(f"[UsedIDs] Loaded {len(seen_ids)} seen samples from {used_ids_path}")

    # 2) 加载数据并过滤 seen_ids
    train_data, test_data = load_cinepile(cfg, seen_ids=seen_ids)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("[ReFT] Loading base model...")
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    print(f"[ReFT] LoReFT: layers={cfg.reft_layers}, rank={cfg.reft_rank}")
    representations = [
        {
            "layer":              l,
            "component":          cfg.component,
            "low_rank_dimension": cfg.reft_rank,
            "intervention":       pyreft.LoreftIntervention(
                embed_dim          = base_model.config.hidden_size,
                low_rank_dimension = cfg.reft_rank,
            ),
        }
        for l in cfg.reft_layers
    ]
    reft_cfg   = pyreft.ReftConfig(representations=representations)
    reft_model = pyreft.get_reft_model(base_model, reft_cfg)

    if resume_from_loreft_dir is not None:
        print(f"[ReFT] Loading LoReFT weights from: {resume_from_loreft_dir}")
        reft_model.load(resume_from_loreft_dir)

    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    train_ds = make_supervised_dataset(train_data, tokenizer, cfg)
    print(f"[ReFT] Train dataset size: {len(train_ds)}")

    training_args = TrainingArguments(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_train_epochs,
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        learning_rate               = cfg.learning_rate,
        warmup_ratio                = cfg.warmup_ratio,
        bf16                        = cfg.bf16,
        logging_steps               = cfg.logging_steps,
        report_to                   = [],
        save_strategy               = "no",   # 不自动保存大模型
    )

    trainer = Trainer(
        model         = model_wrapper,
        args          = training_args,
        train_dataset = train_ds,
    )

    print(f"[ReFT] Training... epochs={cfg.num_train_epochs}, "
          f"lr={cfg.learning_rate}, rank={cfg.reft_rank}, layers={cfg.reft_layers}")
    trainer.train()

    # 3) 保存 step-wise training log
    log_history = trainer.state.log_history
    if len(log_history) > 0:
        df_log = pd.DataFrame(log_history)
        log_path = os.path.join(cfg.output_dir, "training_log_history.csv")
        df_log.to_csv(log_path, index=False)
        print(f"[ReFT] Step-wise log saved → {log_path}")

    # 4) 只保存 LoReFT 小权重 + tokenizer
    model_wrapper.save(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"[ReFT] LoReFT saved → {cfg.output_dir}")

    # 5) 记录本轮使用的 sample_id，避免下次重复
    with open(used_ids_path, "a") as f:
        for s in train_data:
            f.write(str(s["sample_id"]) + "\n")
    print(f"[UsedIDs] Appended {len(train_data)} ids → {used_ids_path}")

    # 6) 评估
    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)
    print("\n[ReFT] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "reft_metrics.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 9. 加载已保存 LoReFT 进行推理/评估
# ============================================================
def load_and_evaluate(
    output_dir: str,
    cfg: ReFTConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"[Load] Tokenizer ← {output_dir}")
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"[Load] Base model ← {cfg.base_model_id}")
    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    representations = [
        {
            "layer":              l,
            "component":          cfg.component,
            "low_rank_dimension": cfg.reft_rank,
            "intervention":       pyreft.LoreftIntervention(
                embed_dim          = base_model.config.hidden_size,
                low_rank_dimension = cfg.reft_rank,
            ),
        }
        for l in cfg.reft_layers
    ]
    reft_cfg   = pyreft.ReftConfig(representations=representations)
    reft_model = pyreft.get_reft_model(base_model, reft_cfg)

    print(f"[Load] LoReFT weights ← {output_dir}")
    reft_model.load(output_dir)
    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    # 注意：加载评估时，不再需要避免重复样本，直接用当前 cfg 的采样逻辑即可
    train_data, test_data = load_cinepile(cfg, seen_ids=None)
    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)

    print("\n[Load] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(output_dir, "reft_metrics_reeval.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 10. 入口示例
# ============================================================
if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    #
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_ReFT"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)


    tracker = ExperimentTracker(
        log_path=os.path.join(PROJECT_DIR, "reft_experiment_log.csv")
    )

    cfg = ReFTConfigCinePile(
        base_model_id            = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples        = 300,   # 分层抽取 3 万
        max_test_samples         = 100,     # 或 None = full
        hard_oversample_ratio    = 1.5,
        num_train_epochs         = 5.0,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        reft_rank                = 8,
        reft_layers              = [20, 24, 28],  # 推荐：后层更有效
        learning_rate            = 2e-4,
        output_dir               = os.path.join(PROJECT_DIR, "reft_30k_stratified_compact")
    )

    used_ids_path = os.path.join(PROJECT_DIR, "used_sample_ids.txt")

    # 第一次训练：见到第一批 3 万样本
    # model, tokenizer, metrics = train_reft_with_trainer(
    #     cfg,
    #     tracker=tracker,
    #     resume_from_loreft_dir=None,
    #     used_ids_path=used_ids_path,
    # )

    # 第二次训练（可选）：只在未见过的样本上继续训练
    model2, tok2, metrics2 = train_reft_with_trainer(
        cfg,
        tracker=tracker,
        resume_from_loreft_dir=cfg.output_dir,  # 从上次 LoReFT 续训
        used_ids_path=used_ids_path,
    )

    # 重新加载评估（验证保存是否正确）
    # model3, tok3, metrics3 = load_and_evaluate(cfg.output_dir, cfg, tracker=tracker)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


transformers: 5.3.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/266_final/phase2/CinePile_ReFT
[UsedIDs] Loaded 500 seen samples from /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/used_sample_ids.txt
[Dataset] Filtered seen samples: 298888 → 298388

[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127308     24       24
     NPA False      32167     24       24
     STA False      85469     24       24
    TEMP False      41435     24       24
      TH False      12009     24       24
[Sampler] Total sampled: 300
[Sampler] ============================================================
[Dataset] Train: 300 | Test: 100
[ReFT] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[ReFT] LoReFT: layers=[20, 24, 28], rank=8
[ReFT] Loading LoReFT weights from: /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_30k_stratified_compact


TypeError: IntervenableModel.load() missing 1 required positional argument: 'model'

In [ ]:
# ============================================================
# ReFT (LoReFT) on CinePile — Compact + Stratified + Logging
#   - transformers 5.3.0
#   - Stratified sampling on CinePile (category × hard)
#   - Only save small LoReFT weights (no big model checkpoints)
#   - Track used sample_ids to avoid re-training on same samples
#   - Save training_log_history.csv for loss analysis
# pip install pyreft datasets accelerate
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Set
from datetime import datetime
from collections import defaultdict

from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from transformers.modeling_outputs import CausalLMOutputWithPast

import pyreft


# ============================================================
# 1. 配置
# ============================================================
@dataclass
class ReFTConfigCinePile:
    # 数据集
    dataset_name: str                = "tomg-group-umd/cinepile"
    train_split: str                 = "train"
    test_split: str                  = "test"

    # 分层抽样（None = 用完整 split）
    max_train_samples: Optional[int] = 30000
    max_test_samples: Optional[int]  = 500

    # hard 样本在分层抽样中的额外倍率
    hard_oversample_ratio: float     = 1.5

    # 模型
    base_model_id: str               = "meta-llama/Llama-3.1-8B-Instruct"

    # 输入
    max_scene_length: int            = 1000
    max_seq_length: int              = 1024

    # 训练
    num_train_epochs: float          = 3.0
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float             = 2e-4
    warmup_ratio: float              = 0.05
    bf16: bool                       = True
    logging_steps: int               = 50
    output_dir: str                  = "./reft_cinepile"  # LoReFT 小权重保存目录

    # LoReFT 配置
    reft_rank: int                   = 8
    reft_layers: List[int]           = field(default_factory=lambda: [20, 24, 28])
    component: str                   = "block_output"

    # 推理生成
    max_new_tokens: int              = 5

    # 评估类别
    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


# ============================================================
# 2. Stratified Sampler（按类别 × hard 分层抽样）
# ============================================================
class StratifiedSampler:
    def __init__(self, cfg: ReFTConfigCinePile):
        self.cfg = cfg

    def sample(self, data: List[dict]) -> List[dict]:
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples
        groups = defaultdict(list)

        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q       = base_quota
        hard_q       = base_quota * self.cfg.hard_oversample_ratio

        sampled = []
        report  = []

        import random
        random.seed(42)

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        self._print_report(report, len(sampled))
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"

    @staticmethod
    def _print_report(report: List[dict], total_sampled: int):
        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {total_sampled}")
        print("[Sampler] " + "=" * 60)


# ============================================================
# 3. ExperimentTracker
# ============================================================
class ExperimentTracker:
    def __init__(self, log_path: str = "./reft_experiment_log.csv"):
        self.log_path = log_path

    def log(self, cfg: ReFTConfigCinePile, metrics: dict) -> pd.DataFrame:
        record = {
            "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "base_model":          cfg.base_model_id.split("/")[-1],
            "max_train_samples":   cfg.max_train_samples if cfg.max_train_samples else "full",
            "max_test_samples":    cfg.max_test_samples  if cfg.max_test_samples  else "full",
            "hard_oversample":     cfg.hard_oversample_ratio,
            "num_train_epochs":    cfg.num_train_epochs,
            "reft_rank":           cfg.reft_rank,
            "reft_layers":         str(cfg.reft_layers),
            "learning_rate":       cfg.learning_rate,
            "batch_size":          cfg.per_device_train_batch_size,
            "grad_accum":          cfg.gradient_accumulation_steps,
            "Overall":             self._fmt(metrics.get("Overall")),
            "TEMP":                self._fmt(metrics.get("TEMP")),
            "CRD":                 self._fmt(metrics.get("CRD")),
            "NPA":                 self._fmt(metrics.get("NPA")),
            "STA":                 self._fmt(metrics.get("STA")),
            "TH":                  self._fmt(metrics.get("TH")),
            "Hard":                self._fmt(metrics.get("Hard")),
            "n_test":              metrics.get("Overall_n"),
            "n_TEMP":              metrics.get("TEMP_n"),
            "n_CRD":               metrics.get("CRD_n"),
            "n_NPA":               metrics.get("NPA_n"),
            "n_STA":               metrics.get("STA_n"),
            "n_TH":                metrics.get("TH_n"),
            "n_Hard":              metrics.get("Hard_n"),
            "output_dir":          cfg.output_dir,
        }
        df_new = pd.DataFrame([record])
        if os.path.exists(self.log_path):
            df_old = pd.read_csv(self.log_path)
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_all = df_new
        df_all.to_csv(self.log_path, index=False)
        print(f"\n[Tracker] Logged → {self.log_path}  (total: {len(df_all)})")
        return df_all

    @staticmethod
    def _fmt(val):
        if val is None:
            return None
        try:
            return round(float(val), 4)
        except Exception:
            return None


# ============================================================
# 4. ReFT 模型包装类
# ============================================================
class ReFTModelWrapper(nn.Module):
    def __init__(self, reft_model):
        super().__init__()
        self.reft_model = reft_model
        self.config     = reft_model.model.config

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, **kwargs):
        base = {"input_ids": input_ids, "attention_mask": attention_mask}
        _, counterfactual_outputs = self.reft_model(
            base,
            unit_locations=None,
            output_original_output=False,
        )
        logits = counterfactual_outputs[0]

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )
        return CausalLMOutputWithPast(loss=loss, logits=logits)

    def generate(self, input_ids=None, attention_mask=None, **kwargs):
        device          = input_ids.device
        generated       = input_ids.clone()
        max_new_tokens  = kwargs.get("max_new_tokens", 5)
        eos_id          = kwargs.get("pad_token_id", None)

        for _ in range(max_new_tokens):
            attn = torch.ones_like(generated)
            base = {"input_ids": generated, "attention_mask": attn}
            with torch.no_grad():
                _, outputs = self.reft_model(
                    base,
                    unit_locations=None,
                    output_original_output=False,
                )
            logits    = outputs[0]
            next_tok  = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tok], dim=1)
            if eos_id is not None and (next_tok == eos_id).all():
                break
        return generated

    def save(self, path: str):
        self.reft_model.save(path)


# ============================================================
# 5. 数据加载（带 sample_id，用于避免重复样本）
# ============================================================
LETTERS = ["A", "B", "C", "D", "E"]


def answer_text_to_letter(answer_text: str, choices: list) -> str:
    for i, c in enumerate(choices):
        if answer_text.strip() == c.strip():
            return LETTERS[i]
    for i, c in enumerate(choices):
        if answer_text.strip() in c or c.strip() in answer_text:
            return LETTERS[i]
    return "A"


def normalize_hard_split(raw) -> bool:
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() == "true"


def load_cinepile(
    cfg: ReFTConfigCinePile,
    seen_ids: Optional[Set[int]] = None,
):
    raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
    raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

    def _convert(ex, idx):
        return {
            "sample_id":         idx,
            "movie_scene":       ex["movie_scene"],
            "question":          ex["question"],
            "choices":           ex["choices"],
            "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
            "question_category": ex["question_category"],
            "hard_split":        normalize_hard_split(ex["hard_split"]),
        }

    train_data = [_convert(ex, i) for i, ex in enumerate(raw_train)]
    test_data  = [_convert(ex, i) for i, ex in enumerate(raw_test)]

    if seen_ids is not None and len(seen_ids) > 0:
        before = len(train_data)
        train_data = [s for s in train_data if s["sample_id"] not in seen_ids]
        print(f"[Dataset] Filtered seen samples: {before} → {len(train_data)}")

    sampler    = StratifiedSampler(cfg)
    train_data = sampler.sample(train_data)

    if cfg.max_test_samples:
        test_data = test_data[:cfg.max_test_samples]

    print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
    return train_data, test_data


# ============================================================
# 6. Prompt & 训练数据集
# ============================================================
def build_inference_prompt(sample, max_scene_length: int) -> str:
    scene   = sample["movie_scene"][:max_scene_length]
    options = "\n".join(
        f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
    )
    return (
        "You are watching a movie. Based on the scene description, "
        "answer the multiple choice question.\n\n"
        f"Scene: {scene}\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


def make_supervised_dataset(train_data, tokenizer,
                            cfg: ReFTConfigCinePile) -> Dataset:
    texts = [
        build_inference_prompt(s, cfg.max_scene_length) + " " + s["answer_key"]
        for s in train_data
    ]
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=cfg.max_seq_length,
        padding=False,
    )
    max_len = max(len(ids) for ids in encodings["input_ids"])
    input_ids_list, attention_mask_list, labels_list = [], [], []

    for ids, mask in zip(encodings["input_ids"], encodings["attention_mask"]):
        pad_len = max_len - len(ids)
        input_ids_list.append(ids + [tokenizer.pad_token_id] * pad_len)
        attention_mask_list.append(mask + [0] * pad_len)
        labels_list.append(ids + [-100] * pad_len)

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_mask_list,
        "labels":         labels_list,
    })


# ============================================================
# 7. 评估
# ============================================================
def predict_choice(model_wrapper, tokenizer, sample,
                  cfg: ReFTConfigCinePile, device: str) -> str:
    model_wrapper.eval()
    prompt = build_inference_prompt(sample, cfg.max_scene_length)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(device)

    with torch.no_grad():
        outputs = model_wrapper.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().upper()

    for ch in decoded:
        if ch in "ABCDE":
            return ch
    return "A"


def evaluate_reft(model_wrapper, tokenizer,
                  test_data: list, cfg: ReFTConfigCinePile) -> dict:
    device  = "cuda" if torch.cuda.is_available() else "cpu"
    records = []
    for s in tqdm(test_data, desc="Eval ReFT"):
        pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
        records.append({
            "pred":              pred,
            "label":             s["answer_key"],
            "question_category": s["question_category"],
            "hard_split":        s["hard_split"],
        })

    df      = pd.DataFrame(records)
    correct = df["pred"] == df["label"]
    metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

    for abbr, keyword in cfg.category_map.items():
        mask = df["question_category"].str.contains(keyword, case=False, na=False)
        n    = int(mask.sum())
        metrics[abbr]        = correct[mask].mean() if n > 0 else None
        metrics[f"{abbr}_n"] = n

    hard_mask = df["hard_split"] == True
    n_hard    = int(hard_mask.sum())
    metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
    metrics["Hard_n"] = n_hard
    return metrics


# ============================================================
# 8. 训练主函数（支持断点续训）
# ============================================================
def train_reft_with_trainer(
    cfg: ReFTConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
    resume_from_loreft_dir: Optional[str] = None,
    used_ids_path: str = "./used_sample_ids.txt",
):
    os.makedirs(cfg.output_dir, exist_ok=True)

    # 1) 读取已使用样本 ID
    seen_ids: Set[int] = set()
    if os.path.exists(used_ids_path):
        with open(used_ids_path, "r") as f:
            for line in f:
                try:
                    seen_ids.add(int(line.strip()))
                except ValueError:
                    continue
        print(f"[UsedIDs] Loaded {len(seen_ids)} seen samples from {used_ids_path}")

    # 2) 加载数据并过滤 seen_ids
    train_data, test_data = load_cinepile(cfg, seen_ids=seen_ids)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("[ReFT] Loading base model...")
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    print(f"[ReFT] LoReFT: layers={cfg.reft_layers}, rank={cfg.reft_rank}")
    if resume_from_loreft_dir is None:
        # 第一次训练：新建 ReFT 模型
        representations = [
            {
                "layer":              l,
                "component":          cfg.component,
                "low_rank_dimension": cfg.reft_rank,
                "intervention":       pyreft.LoreftIntervention(
                    embed_dim          = base_model.config.hidden_size,
                    low_rank_dimension = cfg.reft_rank,
                ),
            }
            for l in cfg.reft_layers
        ]
        reft_cfg   = pyreft.ReftConfig(representations=representations)
        reft_model = pyreft.get_reft_model(base_model, reft_cfg)
        print("[ReFT] Initialized new ReFT model.")
    else:
        # 续训：从已有 LoReFT 目录加载
        print(f"[ReFT] Resuming from LoReFT dir: {resume_from_loreft_dir}")
        reft_model = pyreft.ReftModel.load(
            resume_from_loreft_dir,
            base_model
        )
        print("[ReFT] Loaded ReFT model from disk.")

    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    train_ds = make_supervised_dataset(train_data, tokenizer, cfg)
    print(f"[ReFT] Train dataset size: {len(train_ds)}")

    training_args = TrainingArguments(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_train_epochs,
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        learning_rate               = cfg.learning_rate,
        warmup_ratio                = cfg.warmup_ratio,
        bf16                        = cfg.bf16,
        logging_steps               = cfg.logging_steps,
        report_to                   = [],
        save_strategy               = "no",   # 不自动保存大模型
    )

    trainer = Trainer(
        model         = model_wrapper,
        args          = training_args,
        train_dataset = train_ds,
    )

    print(f"[ReFT] Training... epochs={cfg.num_train_epochs}, "
          f"lr={cfg.learning_rate}, rank={cfg.reft_rank}, layers={cfg.reft_layers}")
    trainer.train()

    # 3) 保存 step-wise training log
    log_history = trainer.state.log_history
    if len(log_history) > 0:
        df_log = pd.DataFrame(log_history)
        log_path = os.path.join(cfg.output_dir, "training_log_history.csv")
        df_log.to_csv(log_path, index=False)
        print(f"[ReFT] Step-wise log saved → {log_path}")

    # 4) 只保存 LoReFT 小权重 + tokenizer
    model_wrapper.save(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"[ReFT] LoReFT saved → {cfg.output_dir}")

    # 5) 记录本轮使用的 sample_id，避免下次重复
    with open(used_ids_path, "a") as f:
        for s in train_data:
            f.write(str(s["sample_id"]) + "\n")
    print(f"[UsedIDs] Appended {len(train_data)} ids → {used_ids_path}")

    # 6) 评估当前采样 test 子集
    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)
    print("\n[ReFT] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "reft_metrics.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 9. 从磁盘加载 ReFT 并评估（可用于完整 test）
# ============================================================
def load_and_evaluate(
    output_dir: str,
    cfg: ReFTConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"[Load] Tokenizer ← {output_dir}")
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"[Load] Base model ← {cfg.base_model_id}")
    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    print(f"[Load] LoReFT weights ← {output_dir}")
    reft_model = pyreft.ReftModel.load(output_dir, base_model)
    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    # 加载当前 cfg 设置下的 test（可把 cfg.max_test_samples=None 做“完整测试集”）
    train_data, test_data = load_cinepile(cfg, seen_ids=None)
    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)

    print("\n[Load] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(output_dir, "reft_metrics_reeval.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 10. 入口示例
# ============================================================
if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    # Colab 环境可注释/启用
    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_ReFT"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)

    tracker = ExperimentTracker(
        log_path=os.path.join(PROJECT_DIR, "reft_experiment_log.csv")
    )

    cfg = ReFTConfigCinePile(
        base_model_id                = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples            = 20000,   # 这里示例用 300，正式实验改回 30000
        max_test_samples             = 1000,   # 或 None = full
        hard_oversample_ratio        = 1.5,
        num_train_epochs             = 5.0,
        per_device_train_batch_size  = 4,
        gradient_accumulation_steps  = 4,
        reft_rank                    = 8,
        reft_layers                  = [20, 24, 28],
        learning_rate                = 2e-4,
        output_dir                   = os.path.join(PROJECT_DIR, "reft_30k_stratified_compact"),
    )

    used_ids_path = os.path.join(PROJECT_DIR, "used_sample_ids.txt")

    # 第一次训练
    # model1, tokenizer1, metrics1 = train_reft_with_trainer(
    #     cfg,
    #     tracker=tracker,
    #     resume_from_loreft_dir=None,
    #     used_ids_path=used_ids_path,
    # )

    # 第二次训练（续训，仅在未见过的样本上）
    model2, tok2, metrics2 = train_reft_with_trainer(
        cfg,
        tracker=tracker,
        resume_from_loreft_dir=cfg.output_dir,
        used_ids_path=used_ids_path,
    )

    del model2, tok2
    gc.collect()
    torch.cuda.empty_cache()

    # 完整测试集评估：拷贝 cfg，只改 max_test_samples=None
    cfg_full_test = ReFTConfigCinePile(
        **{**cfg.__dict__, "max_test_samples": None}
    )
    load_and_evaluate(cfg.output_dir, cfg_full_test, tracker=tracker)

    del model2, tok2
    gc.collect()
    torch.cuda.empty_cache()


transformers: 5.3.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/266_final/phase2/CinePile_ReFT
[UsedIDs] Loaded 800 seen samples from /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/used_sample_ids.txt
[Dataset] Filtered seen samples: 298888 → 298088

[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127198   1600     1600
     NPA False      32129   1600     1600
     STA False      85391   1600     1600
    TEMP False      41389   1600     1600
      TH False      11981   1600     1600
[Sampler] Total sampled: 20000
[Sampler] ============================================================
[Dataset] Train: 20000 | Test: 1000
[ReFT] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[ReFT] LoReFT: layers=[20, 24, 28], rank=8
[ReFT] Resuming from LoReFT dir: /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_30k_stratified_compact


[ReFT] Loaded ReFT model from disk.


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

[ReFT] Step-wise log saved → /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_30k_stratified_compact/training_log_history.csv
Directory '/content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_30k_stratified_compact' already exists.
[ReFT] LoReFT saved → /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_30k_stratified_compact
[UsedIDs] Appended 20000 ids → /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/used_sample_ids.txt


Eval ReFT: 100%|██████████| 1000/1000 [03:50<00:00,  4.33it/s]



[ReFT] ===== Eval Metrics =====
  Overall   : 54.60%  (n=1000)
  TEMP      : 38.97%  (n=136)
  CRD       : 56.90%  (n=420)
  NPA       : 56.67%  (n=90)
  STA       : 58.65%  (n=312)
  TH        : 47.62%  (n=42)
  Hard      : 36.63%  (n=202)

[Tracker] Logged → /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_experiment_log.csv  (total: 5)
[Load] Tokenizer ← /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_30k_stratified_compact
[Load] Base model ← meta-llama/Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 51.44 MiB is free. Including non-PyTorch memory, this process has 39.43 GiB memory in use. Of the allocated memory 38.77 GiB is allocated by PyTorch, and 128.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
!pip install "transformers==4.39.3" "accelerate==0.27.2" -U
!pip install "pyreft==0.0.7" -U
!pip install datasets bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installation:

In [ ]:
# ============================================================
# ReFT (LoReFT) on CinePile — Compact + Stratified + Logging
#   - transformers 5.3.0
#   - Stratified sampling on CinePile (category × hard)
#   - Only save small LoReFT weights (no big model checkpoints)
#   - Track used sample_ids to avoid re-training on same samples
#   - Save training_log_history.csv for loss analysis
# pip install pyreft datasets accelerate
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Set
from datetime import datetime
from collections import defaultdict

from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from transformers.modeling_outputs import CausalLMOutputWithPast

import pyreft


# ============================================================
# 1. 配置
# ============================================================
@dataclass
class ReFTConfigCinePile:
    # 数据集
    dataset_name: str                = "tomg-group-umd/cinepile"
    train_split: str                 = "train"
    test_split: str                  = "test"

    # 分层抽样（None = 用完整 split）
    max_train_samples: Optional[int] = 30000
    max_test_samples: Optional[int]  = 500

    # hard 样本在分层抽样中的额外倍率
    hard_oversample_ratio: float     = 1.5

    # 模型
    base_model_id: str               = "meta-llama/Llama-3.1-8B-Instruct"

    # 输入
    max_scene_length: int            = 1000
    max_seq_length: int              = 1024

    # 训练
    num_train_epochs: float          = 3.0
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float             = 2e-4
    warmup_ratio: float              = 0.05
    bf16: bool                       = True
    logging_steps: int               = 50
    output_dir: str                  = "./reft_cinepile"  # LoReFT 小权重保存目录

    # LoReFT 配置
    reft_rank: int                   = 8
    reft_layers: List[int]           = field(default_factory=lambda: [20, 24, 28])
    component: str                   = "block_output"

    # 推理生成
    max_new_tokens: int              = 5

    # 评估类别
    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


# ============================================================
# 2. Stratified Sampler（按类别 × hard 分层抽样）
# ============================================================
class StratifiedSampler:
    def __init__(self, cfg: ReFTConfigCinePile):
        self.cfg = cfg

    def sample(self, data: List[dict]) -> List[dict]:
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples
        groups = defaultdict(list)

        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q       = base_quota
        hard_q       = base_quota * self.cfg.hard_oversample_ratio

        sampled = []
        report  = []

        import random
        random.seed(42)

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        self._print_report(report, len(sampled))
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"

    @staticmethod
    def _print_report(report: List[dict], total_sampled: int):
        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {total_sampled}")
        print("[Sampler] " + "=" * 60)


# ============================================================
# 3. ExperimentTracker
# ============================================================
class ExperimentTracker:
    def __init__(self, log_path: str = "./reft_experiment_log.csv"):
        self.log_path = log_path

    def log(self, cfg: ReFTConfigCinePile, metrics: dict) -> pd.DataFrame:
        record = {
            "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "base_model":          cfg.base_model_id.split("/")[-1],
            "max_train_samples":   cfg.max_train_samples if cfg.max_train_samples else "full",
            "max_test_samples":    cfg.max_test_samples  if cfg.max_test_samples  else "full",
            "hard_oversample":     cfg.hard_oversample_ratio,
            "num_train_epochs":    cfg.num_train_epochs,
            "reft_rank":           cfg.reft_rank,
            "reft_layers":         str(cfg.reft_layers),
            "learning_rate":       cfg.learning_rate,
            "batch_size":          cfg.per_device_train_batch_size,
            "grad_accum":          cfg.gradient_accumulation_steps,
            "Overall":             self._fmt(metrics.get("Overall")),
            "TEMP":                self._fmt(metrics.get("TEMP")),
            "CRD":                 self._fmt(metrics.get("CRD")),
            "NPA":                 self._fmt(metrics.get("NPA")),
            "STA":                 self._fmt(metrics.get("STA")),
            "TH":                  self._fmt(metrics.get("TH")),
            "Hard":                self._fmt(metrics.get("Hard")),
            "n_test":              metrics.get("Overall_n"),
            "n_TEMP":              metrics.get("TEMP_n"),
            "n_CRD":               metrics.get("CRD_n"),
            "n_NPA":               metrics.get("NPA_n"),
            "n_STA":               metrics.get("STA_n"),
            "n_TH":                metrics.get("TH_n"),
            "n_Hard":              metrics.get("Hard_n"),
            "output_dir":          cfg.output_dir,
        }
        df_new = pd.DataFrame([record])
        if os.path.exists(self.log_path):
            df_old = pd.read_csv(self.log_path)
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_all = df_new
        df_all.to_csv(self.log_path, index=False)
        print(f"\n[Tracker] Logged → {self.log_path}  (total: {len(df_all)})")
        return df_all

    @staticmethod
    def _fmt(val):
        if val is None:
            return None
        try:
            return round(float(val), 4)
        except Exception:
            return None


# ============================================================
# 4. ReFT 模型包装类
# ============================================================
class ReFTModelWrapper(nn.Module):
    def __init__(self, reft_model):
        super().__init__()
        self.reft_model = reft_model
        self.config     = reft_model.model.config

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, **kwargs):
        base = {"input_ids": input_ids, "attention_mask": attention_mask}
        _, counterfactual_outputs = self.reft_model(
            base,
            unit_locations=None,
            output_original_output=False,
        )
        logits = counterfactual_outputs[0]

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )
        return CausalLMOutputWithPast(loss=loss, logits=logits)

    def generate(self, input_ids=None, attention_mask=None, **kwargs):
        device          = input_ids.device
        generated       = input_ids.clone()
        max_new_tokens  = kwargs.get("max_new_tokens", 5)
        eos_id          = kwargs.get("pad_token_id", None)

        for _ in range(max_new_tokens):
            attn = torch.ones_like(generated)
            base = {"input_ids": generated, "attention_mask": attn}
            with torch.no_grad():
                _, outputs = self.reft_model(
                    base,
                    unit_locations=None,
                    output_original_output=False,
                )
            logits    = outputs[0]
            next_tok  = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tok], dim=1)
            if eos_id is not None and (next_tok == eos_id).all():
                break
        return generated

    def save(self, path: str):
        self.reft_model.save(path)


# ============================================================
# 5. 数据加载（带 sample_id，用于避免重复样本）
# ============================================================
LETTERS = ["A", "B", "C", "D", "E"]


def answer_text_to_letter(answer_text: str, choices: list) -> str:
    for i, c in enumerate(choices):
        if answer_text.strip() == c.strip():
            return LETTERS[i]
    for i, c in enumerate(choices):
        if answer_text.strip() in c or c.strip() in answer_text:
            return LETTERS[i]
    return "A"


def normalize_hard_split(raw) -> bool:
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() == "true"


def load_cinepile(
    cfg: ReFTConfigCinePile,
    seen_ids: Optional[Set[int]] = None,
):
    raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
    raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

    def _convert(ex, idx):
        return {
            "sample_id":         idx,
            "movie_scene":       ex["movie_scene"],
            "question":          ex["question"],
            "choices":           ex["choices"],
            "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
            "question_category": ex["question_category"],
            "hard_split":        normalize_hard_split(ex["hard_split"]),
        }

    train_data = [_convert(ex, i) for i, ex in enumerate(raw_train)]
    test_data  = [_convert(ex, i) for i, ex in enumerate(raw_test)]

    if seen_ids is not None and len(seen_ids) > 0:
        before = len(train_data)
        train_data = [s for s in train_data if s["sample_id"] not in seen_ids]
        print(f"[Dataset] Filtered seen samples: {before} → {len(train_data)}")

    sampler    = StratifiedSampler(cfg)
    train_data = sampler.sample(train_data)

    if cfg.max_test_samples:
        test_data = test_data[:cfg.max_test_samples]

    print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
    return train_data, test_data


# ============================================================
# 6. Prompt & 训练数据集
# ============================================================
def build_inference_prompt(sample, max_scene_length: int) -> str:
    scene   = sample["movie_scene"][:max_scene_length]
    options = "\n".join(
        f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
    )
    return (
        "You are watching a movie. Based on the scene description, "
        "answer the multiple choice question.\n\n"
        f"Scene: {scene}\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


def make_supervised_dataset(train_data, tokenizer,
                            cfg: ReFTConfigCinePile) -> Dataset:
    texts = [
        build_inference_prompt(s, cfg.max_scene_length) + " " + s["answer_key"]
        for s in train_data
    ]
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=cfg.max_seq_length,
        padding=False,
    )
    max_len = max(len(ids) for ids in encodings["input_ids"])
    input_ids_list, attention_mask_list, labels_list = [], [], []

    for ids, mask in zip(encodings["input_ids"], encodings["attention_mask"]):
        pad_len = max_len - len(ids)
        input_ids_list.append(ids + [tokenizer.pad_token_id] * pad_len)
        attention_mask_list.append(mask + [0] * pad_len)
        labels_list.append(ids + [-100] * pad_len)

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_mask_list,
        "labels":         labels_list,
    })


# ============================================================
# 7. 评估
# ============================================================
def predict_choice(model_wrapper, tokenizer, sample,
                  cfg: ReFTConfigCinePile, device: str) -> str:
    model_wrapper.eval()
    prompt = build_inference_prompt(sample, cfg.max_scene_length)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(device)

    with torch.no_grad():
        outputs = model_wrapper.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().upper()

    for ch in decoded:
        if ch in "ABCDE":
            return ch
    return "A"


def evaluate_reft(model_wrapper, tokenizer,
                  test_data: list, cfg: ReFTConfigCinePile) -> dict:
    device  = "cuda" if torch.cuda.is_available() else "cpu"
    records = []
    for s in tqdm(test_data, desc="Eval ReFT"):
        pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
        records.append({
            "pred":              pred,
            "label":             s["answer_key"],
            "question_category": s["question_category"],
            "hard_split":        s["hard_split"],
        })

    df      = pd.DataFrame(records)
    correct = df["pred"] == df["label"]
    metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

    for abbr, keyword in cfg.category_map.items():
        mask = df["question_category"].str.contains(keyword, case=False, na=False)
        n    = int(mask.sum())
        metrics[abbr]        = correct[mask].mean() if n > 0 else None
        metrics[f"{abbr}_n"] = n

    hard_mask = df["hard_split"] == True
    n_hard    = int(hard_mask.sum())
    metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
    metrics["Hard_n"] = n_hard
    return metrics


# ============================================================
# 8. 训练主函数（支持断点续训）
# ============================================================
def train_reft_with_trainer(
    cfg: ReFTConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
    resume_from_loreft_dir: Optional[str] = None,
    used_ids_path: str = "./used_sample_ids.txt",
):
    os.makedirs(cfg.output_dir, exist_ok=True)

    # 1) 读取已使用样本 ID
    seen_ids: Set[int] = set()
    if os.path.exists(used_ids_path):
        with open(used_ids_path, "r") as f:
            for line in f:
                try:
                    seen_ids.add(int(line.strip()))
                except ValueError:
                    continue
        print(f"[UsedIDs] Loaded {len(seen_ids)} seen samples from {used_ids_path}")

    # 2) 加载数据并过滤 seen_ids
    train_data, test_data = load_cinepile(cfg, seen_ids=seen_ids)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("[ReFT] Loading base model...")
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    print(f"[ReFT] LoReFT: layers={cfg.reft_layers}, rank={cfg.reft_rank}")
    if resume_from_loreft_dir is None:
        # 第一次训练：新建 ReFT 模型
        representations = [
            {
                "layer":              l,
                "component":          cfg.component,
                "low_rank_dimension": cfg.reft_rank,
                "intervention":       pyreft.LoreftIntervention(
                    embed_dim          = base_model.config.hidden_size,
                    low_rank_dimension = cfg.reft_rank,
                ),
            }
            for l in cfg.reft_layers
        ]
        reft_cfg   = pyreft.ReftConfig(representations=representations)
        reft_model = pyreft.get_reft_model(base_model, reft_cfg)
        print("[ReFT] Initialized new ReFT model.")
    else:
        # 续训：从已有 LoReFT 目录加载
        print(f"[ReFT] Resuming from LoReFT dir: {resume_from_loreft_dir}")
        reft_model = pyreft.ReftModel.load(
            resume_from_loreft_dir,
            base_model
        )
        print("[ReFT] Loaded ReFT model from disk.")

    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    train_ds = make_supervised_dataset(train_data, tokenizer, cfg)
    print(f"[ReFT] Train dataset size: {len(train_ds)}")

    training_args = TrainingArguments(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_train_epochs,
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        learning_rate               = cfg.learning_rate,
        warmup_ratio                = cfg.warmup_ratio,
        bf16                        = cfg.bf16,
        logging_steps               = cfg.logging_steps,
        report_to                   = [],
        save_strategy               = "no",   # 不自动保存大模型
    )

    trainer = Trainer(
        model         = model_wrapper,
        args          = training_args,
        train_dataset = train_ds,
    )

    print(f"[ReFT] Training... epochs={cfg.num_train_epochs}, "
          f"lr={cfg.learning_rate}, rank={cfg.reft_rank}, layers={cfg.reft_layers}")
    trainer.train()

    # 3) 保存 step-wise training log
    log_history = trainer.state.log_history
    if len(log_history) > 0:
        df_log = pd.DataFrame(log_history)
        log_path = os.path.join(cfg.output_dir, "training_log_history.csv")
        df_log.to_csv(log_path, index=False)
        print(f"[ReFT] Step-wise log saved → {log_path}")

    # 4) 只保存 LoReFT 小权重 + tokenizer
    model_wrapper.save(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"[ReFT] LoReFT saved → {cfg.output_dir}")

    # 5) 记录本轮使用的 sample_id，避免下次重复
    with open(used_ids_path, "a") as f:
        for s in train_data:
            f.write(str(s["sample_id"]) + "\n")
    print(f"[UsedIDs] Appended {len(train_data)} ids → {used_ids_path}")

    # 6) 评估当前采样 test 子集
    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)
    print("\n[ReFT] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "reft_metrics.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 9. 从磁盘加载 ReFT 并评估（可用于完整 test）
# ============================================================
def load_and_evaluate(
    output_dir: str,
    cfg: ReFTConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"[Load] Tokenizer ← {output_dir}")
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"[Load] Base model ← {cfg.base_model_id}")
    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    print(f"[Load] LoReFT weights ← {output_dir}")
    reft_model = pyreft.ReftModel.load(output_dir, base_model)
    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    # 加载当前 cfg 设置下的 test（可把 cfg.max_test_samples=None 做“完整测试集”）
    train_data, test_data = load_cinepile(cfg, seen_ids=None)
    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)

    print("\n[Load] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(output_dir, "reft_metrics_reeval.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 10. 入口示例
# ============================================================
if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    # Colab 环境可注释/启用
    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_ReFT"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)

    tracker = ExperimentTracker(
        log_path=os.path.join(PROJECT_DIR, "reft_experiment_log.csv")
    )

    cfg = ReFTConfigCinePile(
        base_model_id                = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples            = 20000,   # 这里示例用 300，正式实验改回 30000
        max_test_samples             = 1000,   # 或 None = full
        hard_oversample_ratio        = 1.5,
        num_train_epochs             = 5.0,
        per_device_train_batch_size  = 4,
        gradient_accumulation_steps  = 4,
        reft_rank                    = 8,
        reft_layers                  = [20, 24, 28],
        learning_rate                = 2e-4,
        output_dir                   = os.path.join(PROJECT_DIR, "reft_30k_stratified_compact"),
    )

    used_ids_path = os.path.join(PROJECT_DIR, "used_sample_ids.txt")

    # 第一次训练
    # model1, tokenizer1, metrics1 = train_reft_with_trainer(
    #     cfg,
    #     tracker=tracker,
    #     resume_from_loreft_dir=None,
    #     used_ids_path=used_ids_path,
    # )

    # 第二次训练（续训，仅在未见过的样本上）
    # model2, tok2, metrics2 = train_reft_with_trainer(
    #     cfg,
    #     tracker=tracker,
    #     resume_from_loreft_dir=cfg.output_dir,
    #     used_ids_path=used_ids_path,
    # )

    # del model2, tok2
    # gc.collect()
    # torch.cuda.empty_cache()

    # 完整测试集评估：拷贝 cfg，只改 max_test_samples=None
    cfg_full_test = ReFTConfigCinePile(
        **{**cfg.__dict__, "max_test_samples": None}
    )
    load_and_evaluate(cfg.output_dir, cfg_full_test, tracker=tracker)

    # del model2, tok2
    # gc.collect()
    # torch.cuda.empty_cache()


nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.
transformers: 5.3.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/266_final/phase2/CinePile_ReFT
[Load] Tokenizer ← /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_30k_stratified_compact
[Load] Base model ← meta-llama/Llama-3.1-8B-Instruct


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

Generating train split:   0%|          | 0/298888 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4941 [00:00<?, ? examples/s]


[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127468   1600     1600
     NPA False      32238   1600     1600
     STA False      85604   1600     1600
    TEMP False      41517   1600     1600
      TH False      12061   1600     1600
[Sampler] Total sampled: 20000
[Sampler] ============================================================
[Dataset] Train: 20000 | Test: 4941


Eval ReFT: 100%|██████████| 4941/4941 [19:15<00:00,  4.27it/s]



[Load] ===== Eval Metrics =====
  Overall   : 55.66%  (n=4941)
  TEMP      : 42.44%  (n=688)
  CRD       : 58.27%  (n=2068)
  NPA       : 56.59%  (n=463)
  STA       : 58.62%  (n=1532)
  TH        : 48.95%  (n=190)
  Hard      : 38.78%  (n=887)

[Tracker] Logged → /content/drive/MyDrive/266_final/phase2/CinePile_ReFT/reft_experiment_log.csv  (total: 6)


## 2.1 ReFT Experiment 2


In [ ]:
!pip install "transformers==4.39.3" "accelerate==0.27.2" -U
!pip install "pyreft==0.0.7" -U
!pip install datasets bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installation

In [ ]:
# ============================================================
# ReFT V2 (LoReFT) on CinePile — Answer-only Objective
#   - Stratified sampling on CinePile (category × hard)
#   - Only save small LoReFT weights (no big model checkpoints)
#   - Track used sample_ids to avoid re-training on same samples
#   - Save training_log_history.csv for loss analysis
#   - V2: only train on final answer letter token (A–E)
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Set
from datetime import datetime
from collections import defaultdict

from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from transformers.modeling_outputs import CausalLMOutputWithPast

import pyreft


# ============================================================
# 1. 配置
# ============================================================
@dataclass
class ReFTConfigCinePile:
    # 数据集
    dataset_name: str                = "tomg-group-umd/cinepile"
    train_split: str                 = "train"
    test_split: str                  = "test"

    # 分层抽样（None = 用完整 split）
    max_train_samples: Optional[int] = 30000
    max_test_samples: Optional[int]  = 500

    # hard 样本在分层抽样中的额外倍率
    hard_oversample_ratio: float     = 1.5

    # 模型
    base_model_id: str               = "meta-llama/Llama-3.1-8B-Instruct"

    # 输入
    max_scene_length: int            = 1000
    max_seq_length: int              = 1024

    # 训练
    num_train_epochs: float          = 3.0
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float             = 2e-4
    warmup_ratio: float              = 0.05
    bf16: bool                       = True
    logging_steps: int               = 50
    output_dir: str                  = "./reft_cinepile"  # LoReFT 小权重保存目录

    # LoReFT 配置
    reft_rank: int                   = 8
    reft_layers: List[int]           = field(default_factory=lambda: [20, 24, 28])
    component: str                   = "block_output"

    # 推理生成（V2：默认只生成 1 token）
    max_new_tokens: int              = 1

    # 评估类别
    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


# ============================================================
# 2. Stratified Sampler（按类别 × hard 分层抽样）
# ============================================================
class StratifiedSampler:
    def __init__(self, cfg: ReFTConfigCinePile):
        self.cfg = cfg

    def sample(self, data: List[dict]) -> List[dict]:
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples
        groups = defaultdict(list)

        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q       = base_quota
        hard_q       = base_quota * self.cfg.hard_oversample_ratio

        sampled = []
        report  = []

        import random
        random.seed(42)

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        self._print_report(report, len(sampled))
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"

    @staticmethod
    def _print_report(report: List[dict], total_sampled: int):
        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {total_sampled}")
        print("[Sampler] " + "=" * 60)


# ============================================================
# 3. ExperimentTracker
# ============================================================
class ExperimentTracker:
    def __init__(self, log_path: str = "./reft_experiment_log.csv"):
        self.log_path = log_path

    def log(self, cfg: ReFTConfigCinePile, metrics: dict) -> pd.DataFrame:
        record = {
            "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "base_model":          cfg.base_model_id.split("/")[-1],
            "max_train_samples":   cfg.max_train_samples if cfg.max_train_samples else "full",
            "max_test_samples":    cfg.max_test_samples  if cfg.max_test_samples  else "full",
            "hard_oversample":     cfg.hard_oversample_ratio,
            "num_train_epochs":    cfg.num_train_epochs,
            "reft_rank":           cfg.reft_rank,
            "reft_layers":         str(cfg.reft_layers),
            "learning_rate":       cfg.learning_rate,
            "batch_size":          cfg.per_device_train_batch_size,
            "grad_accum":          cfg.gradient_accumulation_steps,
            "Overall":             self._fmt(metrics.get("Overall")),
            "TEMP":                self._fmt(metrics.get("TEMP")),
            "CRD":                 self._fmt(metrics.get("CRD")),
            "NPA":                 self._fmt(metrics.get("NPA")),
            "STA":                 self._fmt(metrics.get("STA")),
            "TH":                  self._fmt(metrics.get("TH")),
            "Hard":                self._fmt(metrics.get("Hard")),
            "n_test":              metrics.get("Overall_n"),
            "n_TEMP":              metrics.get("TEMP_n"),
            "n_CRD":               metrics.get("CRD_n"),
            "n_NPA":               metrics.get("NPA_n"),
            "n_STA":               metrics.get("STA_n"),
            "n_TH":                metrics.get("TH_n"),
            "n_Hard":              metrics.get("Hard_n"),
            "output_dir":          cfg.output_dir,
        }
        df_new = pd.DataFrame([record])
        if os.path.exists(self.log_path):
            df_old = pd.read_csv(self.log_path)
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_all = df_new
        df_all.to_csv(self.log_path, index=False)
        print(f"\n[Tracker] Logged → {self.log_path}  (total: {len(df_all)})")
        return df_all

    @staticmethod
    def _fmt(val):
        if val is None:
            return None
        try:
            return round(float(val), 4)
        except Exception:
            return None


# ============================================================
# 4. ReFT 模型包装类
# ============================================================
class ReFTModelWrapper(nn.Module):
    def __init__(self, reft_model):
        super().__init__()
        self.reft_model = reft_model
        self.config     = reft_model.model.config

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, **kwargs):
        base = {"input_ids": input_ids, "attention_mask": attention_mask}
        _, counterfactual_outputs = self.reft_model(
            base,
            unit_locations=None,
            output_original_output=False,
        )
        logits = counterfactual_outputs[0]

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )
        return CausalLMOutputWithPast(loss=loss, logits=logits)

    def generate(self, input_ids=None, attention_mask=None, **kwargs):
        device          = input_ids.device
        generated       = input_ids.clone()
        max_new_tokens  = kwargs.get("max_new_tokens", 1)
        eos_id          = kwargs.get("pad_token_id", None)

        for _ in range(max_new_tokens):
            attn = torch.ones_like(generated)
            base = {"input_ids": generated, "attention_mask": attn}
            with torch.no_grad():
                _, outputs = self.reft_model(
                    base,
                    unit_locations=None,
                    output_original_output=False,
                )
            logits    = outputs[0]
            next_tok  = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tok], dim=1)
            if eos_id is not None and (next_tok == eos_id).all():
                break
        return generated

    def save(self, path: str):
        self.reft_model.save(path)


# ============================================================
# 5. 数据加载（带 sample_id，用于避免重复样本）
# ============================================================
LETTERS = ["A", "B", "C", "D", "E"]


def answer_text_to_letter(answer_text: str, choices: list) -> str:
    for i, c in enumerate(choices):
        if answer_text.strip() == c.strip():
            return LETTERS[i]
    for i, c in enumerate(choices):
        if answer_text.strip() in c or c.strip() in answer_text:
            return LETTERS[i]
    return "A"


def normalize_hard_split(raw) -> bool:
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() == "true"


def load_cinepile(
    cfg: ReFTConfigCinePile,
    seen_ids: Optional[Set[int]] = None,
):
    raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
    raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

    def _convert(ex, idx):
        return {
            "sample_id":         idx,
            "movie_scene":       ex["movie_scene"],
            "question":          ex["question"],
            "choices":           ex["choices"],
            "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
            "question_category": ex["question_category"],
            "hard_split":        normalize_hard_split(ex["hard_split"]),
        }

    train_data = [_convert(ex, i) for i, ex in enumerate(raw_train)]
    test_data  = [_convert(ex, i) for i, ex in enumerate(raw_test)]

    if seen_ids is not None and len(seen_ids) > 0:
        before = len(train_data)
        train_data = [s for s in train_data if s["sample_id"] not in seen_ids]
        print(f"[Dataset] Filtered seen samples: {before} → {len(train_data)}")

    sampler    = StratifiedSampler(cfg)
    train_data = sampler.sample(train_data)

    if cfg.max_test_samples:
        test_data = test_data[:cfg.max_test_samples]

    print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
    return train_data, test_data


# ============================================================
# 6. Prompt & 训练数据集（V2 目标：只在答案 token 上算 loss）
# ============================================================
def build_inference_prompt(sample, max_scene_length: int) -> str:
    scene   = sample["movie_scene"][:max_scene_length]
    options = "\n".join(
        f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
    )
    return (
        "You are watching a movie. Based on the scene description, "
        "answer the multiple choice question.\n\n"
        f"Scene: {scene}\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


def make_supervised_dataset(train_data, tokenizer,
                            cfg: ReFTConfigCinePile) -> Dataset:
    input_ids_list, attention_mask_list, labels_list = [], [], []

    max_len = cfg.max_seq_length

    for s in train_data:
        # 1) 只编码 prompt（不拼答案）
        prompt = build_inference_prompt(s, cfg.max_scene_length)
        enc = tokenizer(
            prompt,
            truncation=True,
            max_length=max_len,
            padding=False,
        )
        ids  = enc["input_ids"]
        mask = enc["attention_mask"]

        # 2) 把正确选项字母转成 token id
        answer_letter = s["answer_key"]  # "A"/"B"/...
        answer_token_ids = tokenizer(answer_letter, add_special_tokens=False)["input_ids"]
        assert len(answer_token_ids) == 1, "Answer letter should map to a single token."
        answer_id = answer_token_ids[0]

        # 3) 确保有一个位置用于放答案 label（prompt 结束后的第一个位置）
        if len(ids) >= max_len - 1:
            ids  = ids[:max_len - 1]
            mask = mask[:max_len - 1]

        pad_len = max_len - len(ids)
        input_ids = ids + [tokenizer.pad_token_id] * pad_len
        attention_mask = mask + [0] * pad_len

        # 4) labels：全 -100，只有 prompt 结束后的第一个位置是答案 id
        labels = [-100] * max_len
        labels[len(ids)] = answer_id

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_mask_list,
        "labels":         labels_list,
    })


# ============================================================
# 7. 评估（V2：优先直接使用生成的第一个 token）
# ============================================================
def predict_choice(model_wrapper, tokenizer, sample,
                  cfg: ReFTConfigCinePile, device: str) -> str:
    model_wrapper.eval()
    prompt = build_inference_prompt(sample, cfg.max_scene_length)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(device)

    with torch.no_grad():
        outputs = model_wrapper.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 取生成的第一个新 token
    gen_token_id = outputs[0][inputs["input_ids"].shape[1]].item()
    gen_token = tokenizer.decode([gen_token_id]).strip().upper()
    if gen_token in ["A", "B", "C", "D", "E"]:
        return gen_token

    # fallback：如果不是 A–E，再扫描整个生成片段
    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().upper()
    for ch in decoded:
        if ch in "ABCDE":
            return ch
    return "A"


def evaluate_reft(model_wrapper, tokenizer,
                  test_data: list, cfg: ReFTConfigCinePile) -> dict:
    device  = "cuda" if torch.cuda.is_available() else "cpu"
    records = []
    for s in tqdm(test_data, desc="Eval ReFT V2"):
        pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
        records.append({
            "pred":              pred,
            "label":             s["answer_key"],
            "question_category": s["question_category"],
            "hard_split":        s["hard_split"],
        })

    df      = pd.DataFrame(records)
    correct = df["pred"] == df["label"]
    metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

    for abbr, keyword in cfg.category_map.items():
        mask = df["question_category"].str.contains(keyword, case=False, na=False)
        n    = int(mask.sum())
        metrics[abbr]        = correct[mask].mean() if n > 0 else None
        metrics[f"{abbr}_n"] = n

    hard_mask = df["hard_split"] == True
    n_hard    = int(hard_mask.sum())
    metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
    metrics["Hard_n"] = n_hard
    return metrics


# ============================================================
# 8. 训练主函数（支持断点续训）
# ============================================================
def train_reft_with_trainer(
    cfg: ReFTConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
    resume_from_loreft_dir: Optional[str] = None,
    used_ids_path: str = "./used_sample_ids.txt",
):
    os.makedirs(cfg.output_dir, exist_ok=True)

    # 1) 读取已使用样本 ID
    seen_ids: Set[int] = set()
    if os.path.exists(used_ids_path):
        with open(used_ids_path, "r") as f:
            for line in f:
                try:
                    seen_ids.add(int(line.strip()))
                except ValueError:
                    continue
        print(f"[UsedIDs] Loaded {len(seen_ids)} seen samples from {used_ids_path}")

    # 2) 加载数据并过滤 seen_ids
    train_data, test_data = load_cinepile(cfg, seen_ids=seen_ids)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("[ReFT V2] Loading base model...")
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    print(f"[ReFT V2] LoReFT: layers={cfg.reft_layers}, rank={cfg.reft_rank}")
    if resume_from_loreft_dir is None:
        # 第一次训练：新建 ReFT 模型
        representations = [
            {
                "layer":              l,
                "component":          cfg.component,
                "low_rank_dimension": cfg.reft_rank,
                "intervention":       pyreft.LoreftIntervention(
                    embed_dim          = base_model.config.hidden_size,
                    low_rank_dimension = cfg.reft_rank,
                ),
            }
            for l in cfg.reft_layers
        ]
        reft_cfg   = pyreft.ReftConfig(representations=representations)
        reft_model = pyreft.get_reft_model(base_model, reft_cfg)
        print("[ReFT V2] Initialized new ReFT model.")
    else:
        # 续训：从已有 LoReFT 目录加载
        print(f"[ReFT V2] Resuming from LoReFT dir: {resume_from_loreft_dir}")
        reft_model = pyreft.ReftModel.load(
            resume_from_loreft_dir,
            base_model
        )
        print("[ReFT V2] Loaded ReFT model from disk.")

    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    train_ds = make_supervised_dataset(train_data, tokenizer, cfg)
    print(f"[ReFT V2] Train dataset size: {len(train_ds)}")

    training_args = TrainingArguments(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_train_epochs,
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        learning_rate               = cfg.learning_rate,
        warmup_ratio                = cfg.warmup_ratio,
        bf16                        = cfg.bf16,
        logging_steps               = cfg.logging_steps,
        report_to                   = [],
        save_strategy               = "no",   # 不自动保存大模型
    )

    trainer = Trainer(
        model         = model_wrapper,
        args          = training_args,
        train_dataset = train_ds,
    )

    print(f"[ReFT V2] Training... epochs={cfg.num_train_epochs}, "
          f"lr={cfg.learning_rate}, rank={cfg.reft_rank}, layers={cfg.reft_layers}")
    trainer.train()

    # 3) 保存 step-wise training log
    log_history = trainer.state.log_history
    if len(log_history) > 0:
        df_log = pd.DataFrame(log_history)
        log_path = os.path.join(cfg.output_dir, "training_log_history.csv")
        df_log.to_csv(log_path, index=False)
        print(f"[ReFT V2] Step-wise log saved → {log_path}")

    # 4) 只保存 LoReFT 小权重 + tokenizer
    model_wrapper.save(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"[ReFT V2] LoReFT saved → {cfg.output_dir}")

    # 5) 记录本轮使用的 sample_id，避免下次重复
    with open(used_ids_path, "a") as f:
        for s in train_data:
            f.write(str(s["sample_id"]) + "\n")
    print(f"[UsedIDs] Appended {len(train_data)} ids → {used_ids_path}")

    # 6) 评估当前采样 test 子集
    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)
    print("\n[ReFT V2] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "reft_v2_metrics.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 9. 从磁盘加载 ReFT V2 并评估（可用于完整 test）
# ============================================================
def load_and_evaluate(
    output_dir: str,
    cfg: ReFTConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"[Load V2] Tokenizer ← {output_dir}")
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"[Load V2] Base model ← {cfg.base_model_id}")
    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
        device_map=device,
    )

    print(f"[Load V2] LoReFT weights ← {output_dir}")
    reft_model = pyreft.ReftModel.load(output_dir, base_model)
    reft_model.set_device(device)
    model_wrapper = ReFTModelWrapper(reft_model)

    # 根据 cfg 的 max_test_samples 采样测试集（设为 None 即用完整 test）
    train_data, test_data = load_cinepile(cfg, seen_ids=None)
    metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)

    print("\n[Load V2] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(output_dir, "reft_v2_metrics_reeval.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 10. 入口示例
# ============================================================
if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    # 如在 Colab，可挂载 Drive
    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_ReFT_V2"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)

    tracker = ExperimentTracker(
        log_path=os.path.join(PROJECT_DIR, "reft_v2_experiment_log.csv")
    )

    cfg = ReFTConfigCinePile(
        base_model_id                = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples            = 20000,   # 或 3000/15000 做 sanity / 主实验
        max_test_samples             = 1000,    # 完整测试用 None
        hard_oversample_ratio        = 1.0,
        num_train_epochs             = 3.0,
        per_device_train_batch_size  = 4,
        gradient_accumulation_steps  = 4,
        reft_rank                    = 8,
        reft_layers                  = [20, 24, 28],
        learning_rate                = 2e-4,
        output_dir                   = os.path.join(PROJECT_DIR, "reft_v2_20k_stratified_rank8"),
    )

    used_ids_path = os.path.join(PROJECT_DIR, "used_sample_ids_v2.txt")

    # 第一次训练
    model1, tokenizer1, metrics1 = train_reft_with_trainer(
        cfg,
        tracker=tracker,
        resume_from_loreft_dir=None,
        used_ids_path=used_ids_path,
    )

    del model1, tokenizer1
    gc.collect()
    torch.cuda.empty_cache()

    # （可选）第二次续训
    # model2, tok2, metrics2 = train_reft_with_trainer(
    #     cfg,
    #     tracker=tracker,
    #     resume_from_loreft_dir=cfg.output_dir,
    #     used_ids_path=used_ids_path,
    # )
    # del model1, tokenizer1
    # gc.collect()
    # torch.cuda.empty_cache()

    # 完整测试集评估
    cfg_full_test = ReFTConfigCinePile(
        **{**cfg.__dict__, "max_test_samples": None}
    )
    model_eval, tok_eval, metrics_eval = load_and_evaluate(cfg.output_dir, cfg_full_test, tracker=tracker)

    del model_eval, tok_eval
    gc.collect()
    torch.cuda.empty_cache()




nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.
transformers: 5.3.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/266_final/phase2/CinePile_ReFT_V2


README.md: 0.00B [00:00, ?B/s]

v2/train-00000-of-00003.parquet:   0%|          | 0.00/21.9M [00:00<?, ?B/s]

v2/train-00001-of-00003.parquet:   0%|          | 0.00/23.2M [00:00<?, ?B/s]

v2/train-00002-of-00003.parquet:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[Load V2] LoReFT weights ← /content/drive/MyDrive/266_final/phase2/CinePile_ReFT_V2/reft_v2_20k_stratified_rank8



[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127468   2000     2000
     NPA False      32238   2000     2000
     STA False      85604   2000     2000
    TEMP False      41517   2000     2000
      TH False      12061   2000     2000
[Sampler] Total sampled: 20000
[Sampler] ============================================================
[Dataset] Train: 20000 | Test: 4941


Eval ReFT V2: 100%|██████████| 4941/4941 [04:05<00:00, 20.11it/s]



[Load V2] ===== Eval Metrics =====
  Overall   : 55.78%  (n=4941)
  TEMP      : 43.46%  (n=688)
  CRD       : 57.83%  (n=2068)
  NPA       : 55.51%  (n=463)
  STA       : 59.53%  (n=1532)
  TH        : 48.42%  (n=190)
  Hard      : 40.14%  (n=887)

[Tracker] Logged → /content/drive/MyDrive/266_final/phase2/CinePile_ReFT_V2/reft_v2_experiment_log.csv  (total: 2)


## (ignore) 2.2 QLoRA and Prefix Tuning

In [ ]:
# # ============================================================
# # ReFT (LoReFT) on CinePile — Compact + Stratified + Logging
# #   - transformers 5.3.0
# #   - Stratified sampling on CinePile (category × hard)
# #   - Only save small LoReFT weights (no big model checkpoints)
# #   - Track used sample_ids to avoid re-training on same samples
# #   - Save training_log_history.csv for loss analysis
# # pip install pyreft datasets accelerate
# # ============================================================

# import os
# import gc
# import torch
# import torch.nn as nn
# import pandas as pd
# from dataclasses import dataclass, field
# from typing import Optional, List, Dict, Set
# from datetime import datetime
# from collections import defaultdict

# from datasets import load_dataset, Dataset
# from tqdm import tqdm

# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     TrainingArguments,
#     Trainer,
# )
# from transformers.modeling_outputs import CausalLMOutputWithPast

# import pyreft


# # ============================================================
# # 1. 配置
# # ============================================================
# @dataclass
# class ReFTConfigCinePile:
#     # 数据集
#     dataset_name: str                = "tomg-group-umd/cinepile"
#     train_split: str                 = "train"
#     test_split: str                  = "test"

#     # 分层抽样（None = 用完整 split）
#     max_train_samples: Optional[int] = 30000   # e.g. 从 ~30 万中抽 3 万
#     max_test_samples: Optional[int]  = 500     # 测试集可截断或 full

#     # hard 样本在分层抽样中的额外倍率
#     hard_oversample_ratio: float     = 1.5

#     # 模型
#     base_model_id: str               = "meta-llama/Llama-3.1-8B-Instruct"

#     # 输入
#     max_scene_length: int            = 1000
#     max_seq_length: int              = 1024

#     # 训练
#     num_train_epochs: float          = 3.0
#     per_device_train_batch_size: int = 4
#     gradient_accumulation_steps: int = 4
#     learning_rate: float             = 2e-4
#     warmup_ratio: float              = 0.05
#     bf16: bool                       = True
#     logging_steps: int               = 50
#     output_dir: str                  = "./reft_cinepile"  # LoReFT 小权重保存目录

#     # LoReFT 配置
#     reft_rank: int                   = 8
#     reft_layers: List[int]           = field(default_factory=lambda: [20, 24, 28])
#     component: str                   = "block_output"

#     # 推理生成
#     max_new_tokens: int              = 5

#     # 评估类别
#     category_map: Dict[str, str] = field(default_factory=lambda: {
#         "TEMP": "Temporal",
#         "CRD":  "Character and",
#         "NPA":  "Narrative and",
#         "STA":  "Setting and",
#         "TH":   "Theme Exploration",
#     })


# # ============================================================
# # 2. Stratified Sampler（按类别 × hard 分层抽样）
# # ============================================================
# class StratifiedSampler:
#     def __init__(self, cfg: ReFTConfigCinePile):
#         self.cfg = cfg

#     def sample(self, data: List[dict]) -> List[dict]:
#         if self.cfg.max_train_samples is None:
#             print(f"[Sampler] Using full train split: {len(data)} samples")
#             return data

#         target = self.cfg.max_train_samples
#         groups = defaultdict(list)

#         for s in data:
#             cat  = self._get_category_key(s["question_category"])
#             hard = bool(s["hard_split"])
#             groups[(cat, hard)].append(s)

#         n_categories = len(set(k[0] for k in groups))
#         base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
#         easy_q       = base_quota
#         hard_q       = base_quota * self.cfg.hard_oversample_ratio

#         sampled = []
#         report  = []

#         import random
#         random.seed(42)

#         for (cat, hard), group in sorted(groups.items()):
#             quota  = int(hard_q if hard else easy_q)
#             n_take = min(quota, len(group))
#             taken  = random.sample(group, n_take)
#             sampled.extend(taken)
#             report.append({
#                 "category":  cat,
#                 "hard":      hard,
#                 "available": len(group),
#                 "quota":     quota,
#                 "sampled":   n_take,
#             })

#         if len(sampled) < target:
#             sampled_ids = set(id(s) for s in sampled)
#             remaining   = [s for s in data if id(s) not in sampled_ids]
#             extra = random.sample(
#                 remaining, min(target - len(sampled), len(remaining))
#             )
#             sampled.extend(extra)

#         self._print_report(report, len(sampled))
#         return sampled

#     def _get_category_key(self, cat_str: str) -> str:
#         for abbr, kw in self.cfg.category_map.items():
#             if kw.lower() in cat_str.lower():
#                 return abbr
#         return "OTHER"

#     @staticmethod
#     def _print_report(report: List[dict], total_sampled: int):
#         df = pd.DataFrame(report)
#         print("\n[Sampler] ===== Stratified Sampling Report =====")
#         print(df.to_string(index=False))
#         print(f"[Sampler] Total sampled: {total_sampled}")
#         print("[Sampler] " + "=" * 60)


# # ============================================================
# # 3. ExperimentTracker
# # ============================================================
# class ExperimentTracker:
#     def __init__(self, log_path: str = "./reft_experiment_log.csv"):
#         self.log_path = log_path

#     def log(self, cfg: ReFTConfigCinePile, metrics: dict) -> pd.DataFrame:
#         record = {
#             "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#             "base_model":          cfg.base_model_id.split("/")[-1],
#             "max_train_samples":   cfg.max_train_samples if cfg.max_train_samples else "full",
#             "max_test_samples":    cfg.max_test_samples  if cfg.max_test_samples  else "full",
#             "hard_oversample":     cfg.hard_oversample_ratio,
#             "num_train_epochs":    cfg.num_train_epochs,
#             "reft_rank":           cfg.reft_rank,
#             "reft_layers":         str(cfg.reft_layers),
#             "learning_rate":       cfg.learning_rate,
#             "batch_size":          cfg.per_device_train_batch_size,
#             "grad_accum":          cfg.gradient_accumulation_steps,
#             "Overall":             self._fmt(metrics.get("Overall")),
#             "TEMP":                self._fmt(metrics.get("TEMP")),
#             "CRD":                 self._fmt(metrics.get("CRD")),
#             "NPA":                 self._fmt(metrics.get("NPA")),
#             "STA":                 self._fmt(metrics.get("STA")),
#             "TH":                  self._fmt(metrics.get("TH")),
#             "Hard":                self._fmt(metrics.get("Hard")),
#             "n_test":              metrics.get("Overall_n"),
#             "n_TEMP":              metrics.get("TEMP_n"),
#             "n_CRD":               metrics.get("CRD_n"),
#             "n_NPA":               metrics.get("NPA_n"),
#             "n_STA":               metrics.get("STA_n"),
#             "n_TH":                metrics.get("TH_n"),
#             "n_Hard":              metrics.get("Hard_n"),
#             "output_dir":          cfg.output_dir,
#         }
#         df_new = pd.DataFrame([record])
#         if os.path.exists(self.log_path):
#             df_old = pd.read_csv(self.log_path)
#             df_all = pd.concat([df_old, df_new], ignore_index=True)
#         else:
#             df_all = df_new
#         df_all.to_csv(self.log_path, index=False)
#         print(f"\n[Tracker] Logged → {self.log_path}  (total: {len(df_all)})")
#         return df_all

#     @staticmethod
#     def _fmt(val):
#         if val is None:
#             return None
#         try:
#             return round(float(val), 4)
#         except Exception:
#             return None


# # ============================================================
# # 4. ReFT 模型包装类
# # ============================================================
# class ReFTModelWrapper(nn.Module):
#     def __init__(self, reft_model):
#         super().__init__()
#         self.reft_model = reft_model
#         self.config     = reft_model.model.config

#     def forward(self, input_ids=None, attention_mask=None,
#                 labels=None, **kwargs):
#         base = {"input_ids": input_ids, "attention_mask": attention_mask}
#         _, counterfactual_outputs = self.reft_model(
#             base,
#             unit_locations=None,
#             output_original_output=False,
#         )
#         logits = counterfactual_outputs[0]

#         loss = None
#         if labels is not None:
#             shift_logits = logits[..., :-1, :].contiguous()
#             shift_labels = labels[..., 1:].contiguous()
#             loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
#             loss = loss_fn(
#                 shift_logits.view(-1, shift_logits.size(-1)),
#                 shift_labels.view(-1),
#             )
#         return CausalLMOutputWithPast(loss=loss, logits=logits)

#     def generate(self, input_ids=None, attention_mask=None, **kwargs):
#         device         = input_ids.device
#         generated      = input_ids.clone()
#         max_new_tokens = kwargs.get("max_new_tokens", 5)
#         eos_id         = kwargs.get("pad_token_id", None)

#         for _ in range(max_new_tokens):
#             attn = torch.ones_like(generated)
#             base = {"input_ids": generated, "attention_mask": attn}
#             with torch.no_grad():
#                 _, outputs = self.reft_model(
#                     base,
#                     unit_locations=None,
#                     output_original_output=False,
#                 )
#             logits    = outputs[0]
#             next_tok  = logits[:, -1, :].argmax(dim=-1, keepdim=True)
#             generated = torch.cat([generated, next_tok], dim=1)
#             if eos_id is not None and (next_tok == eos_id).all():
#                 break
#         return generated

#     def save(self, path: str):
#         self.reft_model.save(path)


# # ============================================================
# # 5. 数据加载（带 sample_id，用于避免重复样本）
# # ============================================================
# LETTERS = ["A", "B", "C", "D", "E"]

# def answer_text_to_letter(answer_text: str, choices: list) -> str:
#     for i, c in enumerate(choices):
#         if answer_text.strip() == c.strip():
#             return LETTERS[i]
#     for i, c in enumerate(choices):
#         if answer_text.strip() in c or c.strip() in answer_text:
#             return LETTERS[i]
#     return "A"

# def normalize_hard_split(raw) -> bool:
#     if isinstance(raw, bool):
#         return raw
#     return str(raw).strip().lower() == "true"

# def load_cinepile(
#     cfg: ReFTConfigCinePile,
#     seen_ids: Optional[Set[int]] = None,
# ):
#     raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
#     raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

#     def _convert(ex, idx):
#         return {
#             "sample_id":         idx,  # 稳定 ID
#             "movie_scene":       ex["movie_scene"],
#             "question":          ex["question"],
#             "choices":           ex["choices"],
#             "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
#             "question_category": ex["question_category"],
#             "hard_split":        normalize_hard_split(ex["hard_split"]),
#         }

#     train_data = [_convert(ex, i) for i, ex in enumerate(raw_train)]
#     test_data  = [_convert(ex, i) for i, ex in enumerate(raw_test)]

#     if seen_ids is not None and len(seen_ids) > 0:
#         before = len(train_data)
#         train_data = [s for s in train_data if s["sample_id"] not in seen_ids]
#         print(f"[Dataset] Filtered seen samples: {before} → {len(train_data)}")

#     sampler    = StratifiedSampler(cfg)
#     train_data = sampler.sample(train_data)

#     if cfg.max_test_samples:
#         test_data = test_data[:cfg.max_test_samples]

#     print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
#     return train_data, test_data


# # ============================================================
# # 6. Prompt & 训练数据集
# # ============================================================
# def build_inference_prompt(sample, max_scene_length: int) -> str:
#     scene   = sample["movie_scene"][:max_scene_length]
#     options = "\n".join(
#         f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
#     )
#     return (
#         "You are watching a movie. Based on the scene description, "
#         "answer the multiple choice question.\n\n"
#         f"Scene: {scene}\n\n"
#         f"Question: {sample['question']}\n\n"
#         f"Options:\n{options}\n\n"
#         "Answer:"
#     )

# def make_supervised_dataset(train_data, tokenizer,
#                              cfg: ReFTConfigCinePile) -> Dataset:
#     texts = [
#         build_inference_prompt(s, cfg.max_scene_length) + " " + s["answer_key"]
#         for s in train_data
#     ]
#     encodings = tokenizer(
#         texts,
#         truncation=True,
#         max_length=cfg.max_seq_length,
#         padding=False,
#     )
#     max_len = max(len(ids) for ids in encodings["input_ids"])
#     input_ids_list, attention_mask_list, labels_list = [], [], []

#     for ids, mask in zip(encodings["input_ids"], encodings["attention_mask"]):
#         pad_len = max_len - len(ids)
#         input_ids_list.append(ids + [tokenizer.pad_token_id] * pad_len)
#         attention_mask_list.append(mask + [0] * pad_len)
#         labels_list.append(ids + [-100] * pad_len)

#     return Dataset.from_dict({
#         "input_ids":      input_ids_list,
#         "attention_mask": attention_mask_list,
#         "labels":         labels_list,
#     })


# # ============================================================
# # 7. 评估
# # ============================================================
# def predict_choice(model_wrapper, tokenizer, sample,
#                    cfg: ReFTConfigCinePile, device: str) -> str:
#     model_wrapper.eval()
#     prompt = build_inference_prompt(sample, cfg.max_scene_length)
#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=cfg.max_seq_length,
#     ).to(device)

#     with torch.no_grad():
#         outputs = model_wrapper.generate(
#             **inputs,
#             max_new_tokens=cfg.max_new_tokens,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#     decoded = tokenizer.decode(
#         outputs[0][inputs["input_ids"].shape[1]:],
#         skip_special_tokens=True,
#     ).strip().upper()

#     for ch in decoded:
#         if ch in "ABCDE":
#             return ch
#     return "A"

# def evaluate_reft(model_wrapper, tokenizer,
#                   test_data: list, cfg: ReFTConfigCinePile) -> dict:
#     device  = "cuda" if torch.cuda.is_available() else "cpu"
#     records = []
#     for s in tqdm(test_data, desc="Eval ReFT"):
#         pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
#         records.append({
#             "pred":              pred,
#             "label":             s["answer_key"],
#             "question_category": s["question_category"],
#             "hard_split":        s["hard_split"],
#         })

#     df      = pd.DataFrame(records)
#     correct = df["pred"] == df["label"]
#     metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

#     for abbr, keyword in cfg.category_map.items():
#         mask = df["question_category"].str.contains(keyword, case=False, na=False)
#         n    = int(mask.sum())
#         metrics[abbr]        = correct[mask].mean() if n > 0 else None
#         metrics[f"{abbr}_n"] = n

#     hard_mask = df["hard_split"] == True
#     n_hard    = int(hard_mask.sum())
#     metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
#     metrics["Hard_n"] = n_hard
#     return metrics


# # ============================================================
# # 8. 训练主函数（不保存大模型，记录每步日志）
# # ============================================================
# def train_reft_with_trainer(
#     cfg: ReFTConfigCinePile,
#     tracker: Optional[ExperimentTracker] = None,
#     resume_from_loreft_dir: Optional[str] = None,
#     used_ids_path: str = "./used_sample_ids.txt",
# ):
#     os.makedirs(cfg.output_dir, exist_ok=True)

#     # 1) 读取已使用样本 ID
#     seen_ids: Set[int] = set()
#     if os.path.exists(used_ids_path):
#         with open(used_ids_path, "r") as f:
#             for line in f:
#                 try:
#                     seen_ids.add(int(line.strip()))
#                 except ValueError:
#                     continue
#         print(f"[UsedIDs] Loaded {len(seen_ids)} seen samples from {used_ids_path}")

#     # 2) 加载数据并过滤 seen_ids
#     train_data, test_data = load_cinepile(cfg, seen_ids=seen_ids)

#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     print("[ReFT] Loading base model...")
#     tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token
#     tokenizer.padding_side = "right"

#     base_model = AutoModelForCausalLM.from_pretrained(
#         cfg.base_model_id,
#         dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
#         device_map=device,
#     )

#     print(f"[ReFT] LoReFT: layers={cfg.reft_layers}, rank={cfg.reft_rank}")
#     representations = [
#         {
#             "layer":              l,
#             "component":          cfg.component,
#             "low_rank_dimension": cfg.reft_rank,
#             "intervention":       pyreft.LoreftIntervention(
#                 embed_dim          = base_model.config.hidden_size,
#                 low_rank_dimension = cfg.reft_rank,
#             ),
#         }
#         for l in cfg.reft_layers
#     ]
#     reft_cfg   = pyreft.ReftConfig(representations=representations)
#     reft_model = pyreft.get_reft_model(base_model, reft_cfg)

#     if resume_from_loreft_dir is not None:
#         print(f"[ReFT] Loading LoReFT weights from: {resume_from_loreft_dir}")
#         reft_model.load(resume_from_loreft_dir)

#     reft_model.set_device(device)
#     model_wrapper = ReFTModelWrapper(reft_model)

#     train_ds = make_supervised_dataset(train_data, tokenizer, cfg)
#     print(f"[ReFT] Train dataset size: {len(train_ds)}")

#     training_args = TrainingArguments(
#         output_dir                  = cfg.output_dir,
#         num_train_epochs            = cfg.num_train_epochs,
#         per_device_train_batch_size = cfg.per_device_train_batch_size,
#         gradient_accumulation_steps = cfg.gradient_accumulation_steps,
#         learning_rate               = cfg.learning_rate,
#         warmup_ratio                = cfg.warmup_ratio,
#         bf16                        = cfg.bf16,
#         logging_steps               = cfg.logging_steps,
#         report_to                   = [],
#         save_strategy               = "no",   # 不自动保存大模型
#     )

#     trainer = Trainer(
#         model         = model_wrapper,
#         args          = training_args,
#         train_dataset = train_ds,
#     )

#     print(f"[ReFT] Training... epochs={cfg.num_train_epochs}, "
#           f"lr={cfg.learning_rate}, rank={cfg.reft_rank}, layers={cfg.reft_layers}")
#     trainer.train()

#     # 3) 保存 step-wise training log
#     log_history = trainer.state.log_history
#     if len(log_history) > 0:
#         df_log = pd.DataFrame(log_history)
#         log_path = os.path.join(cfg.output_dir, "training_log_history.csv")
#         df_log.to_csv(log_path, index=False)
#         print(f"[ReFT] Step-wise log saved → {log_path}")

#     # 4) 只保存 LoReFT 小权重 + tokenizer
#     model_wrapper.save(cfg.output_dir)
#     tokenizer.save_pretrained(cfg.output_dir)
#     print(f"[ReFT] LoReFT saved → {cfg.output_dir}")

#     # 5) 记录本轮使用的 sample_id，避免下次重复
#     with open(used_ids_path, "a") as f:
#         for s in train_data:
#             f.write(str(s["sample_id"]) + "\n")
#     print(f"[UsedIDs] Appended {len(train_data)} ids → {used_ids_path}")

#     # 6) 评估
#     metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)
#     print("\n[ReFT] ===== Eval Metrics =====")
#     for k, v in metrics.items():
#         if not k.endswith("_n") and v is not None:
#             n = metrics.get(f"{k}_n", "")
#             print(f"  {k:<10}: {v:.2%}  (n={n})")

#     pd.DataFrame([metrics]).to_csv(
#         os.path.join(cfg.output_dir, "reft_metrics.csv"), index=False
#     )
#     if tracker is not None:
#         tracker.log(cfg, metrics)

#     return model_wrapper, tokenizer, metrics


# # ============================================================
# # 9. 加载已保存 LoReFT 进行推理/评估
# # ============================================================
# def load_and_evaluate(
#     output_dir: str,
#     cfg: ReFTConfigCinePile,
#     tracker: Optional[ExperimentTracker] = None,
# ):
#     device = "cuda" if torch.cuda.is_available() else "cpu"

#     print(f"[Load] Tokenizer ← {output_dir}")
#     tokenizer = AutoTokenizer.from_pretrained(output_dir)
#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token
#     tokenizer.padding_side = "right"

#     print(f"[Load] Base model ← {cfg.base_model_id}")
#     base_model = AutoModelForCausalLM.from_pretrained(
#         cfg.base_model_id,
#         dtype=torch.bfloat16 if cfg.bf16 else torch.float32,
#         device_map=device,
#     )

#     representations = [
#         {
#             "layer":              l,
#             "component":          cfg.component,
#             "low_rank_dimension": cfg.reft_rank,
#             "intervention":       pyreft.LoreftIntervention(
#                 embed_dim          = base_model.config.hidden_size,
#                 low_rank_dimension = cfg.reft_rank,
#             ),
#         }
#         for l in cfg.reft_layers
#     ]
#     reft_cfg   = pyreft.ReftConfig(representations=representations)
#     reft_model = pyreft.get_reft_model(base_model, reft_cfg)

#     print(f"[Load] LoReFT weights ← {output_dir}")
#     reft_model.load(output_dir)
#     reft_model.set_device(device)
#     model_wrapper = ReFTModelWrapper(reft_model)

#     # 注意：加载评估时，不再需要避免重复样本，直接用当前 cfg 的采样逻辑即可
#     train_data, test_data = load_cinepile(cfg, seen_ids=None)
#     metrics = evaluate_reft(model_wrapper, tokenizer, test_data, cfg)

#     print("\n[Load] ===== Eval Metrics =====")
#     for k, v in metrics.items():
#         if not k.endswith("_n") and v is not None:
#             n = metrics.get(f"{k}_n", "")
#             print(f"  {k:<10}: {v:.2%}  (n={n})")

#     pd.DataFrame([metrics]).to_csv(
#         os.path.join(output_dir, "reft_metrics_reeval.csv"), index=False
#     )
#     if tracker is not None:
#         tracker.log(cfg, metrics)

#     return model_wrapper, tokenizer, metrics


# # ============================================================
# # 10. 入口示例
# # ============================================================
# if __name__ == "__main__":
#     import transformers
#     print("transformers:", transformers.__version__)

#     tracker = ExperimentTracker("./reft_experiment_log.csv")

#     cfg = ReFTConfigCinePile(
#         base_model_id            = "meta-llama/Llama-3.1-8B-Instruct",
#         max_train_samples        = 300,   # 分层抽取 3 万
#         max_test_samples         = 100,     # 或 None = full
#         hard_oversample_ratio    = 1.5,
#         num_train_epochs         = 3.0,
#         per_device_train_batch_size = 4,
#         gradient_accumulation_steps = 4,
#         reft_rank                = 8,
#         reft_layers              = [20, 24, 28],  # 推荐：后层更有效
#         learning_rate            = 2e-4,
#         output_dir               = "./reft_30k_stratified_compact",
#     )

#     used_ids_path = "./used_sample_ids.txt"

#     # 第一次训练：见到第一批 3 万样本
#     model, tokenizer, metrics = train_reft_with_trainer(
#         cfg,
#         tracker=tracker,
#         resume_from_loreft_dir=None,
#         used_ids_path=used_ids_path,
#     )

#     # 第二次训练（可选）：只在未见过的样本上继续训练
#     # model2, tok2, metrics2 = train_reft_with_trainer(
#     #     cfg,
#     #     tracker=tracker,
#     #     resume_from_loreft_dir=cfg.output_dir,  # 从上次 LoReFT 续训
#     #     used_ids_path=used_ids_path,
#     # )

#     # 重新加载评估（验证保存是否正确）
#     # model3, tok3, metrics3 = load_and_evaluate(cfg.output_dir, cfg, tracker=tracker)

#     del model, tokenizer
#     gc.collect()
#     torch.cuda.empty_cache()


transformers: 5.3.0

[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127468     24       24
     NPA False      32238     24       24
     STA False      85604     24       24
    TEMP False      41517     24       24
      TH False      12061     24       24
[Sampler] Total sampled: 300
[Sampler] ============================================================
[Dataset] Train: 300 | Test: 100
[ReFT] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[ReFT] LoReFT: layers=[20, 24, 28], rank=8


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[ReFT] Train dataset size: 300
[ReFT] Training... epochs=3.0, lr=0.0002, rank=8, layers=[20, 24, 28]


Step,Training Loss
50,7.895765


[ReFT] Step-wise log saved → ./reft_30k_stratified_compact/training_log_history.csv
Directory './reft_30k_stratified_compact' already exists.
[ReFT] LoReFT saved → ./reft_30k_stratified_compact
[UsedIDs] Appended 300 ids → ./used_sample_ids.txt


Eval ReFT: 100%|██████████| 100/100 [00:22<00:00,  4.48it/s]



[ReFT] ===== Eval Metrics =====
  Overall   : 50.00%  (n=100)
  TEMP      : 30.77%  (n=13)
  CRD       : 52.38%  (n=42)
  NPA       : 60.00%  (n=5)
  STA       : 60.61%  (n=33)
  TH        : 14.29%  (n=7)
  Hard      : 50.00%  (n=26)

[Tracker] Logged → ./reft_experiment_log.csv  (total: 1)


## 2.2 QloRA

In [ ]:
# ============================================================
# QLoRA on CinePile (Phase 2)
#   - Stratified sampling on CinePile (category × hard)
#   - Only save LoRA adapter (no full base model checkpoints)
#   - Track used sample_ids to avoid re-training on same samples
#   - Save training_log_history.csv for loss analysis
# Requirements:
#   pip install transformers datasets accelerate peft bitsandbytes trl
# ============================================================

import os
import gc
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Set
from datetime import datetime
from collections import defaultdict

import torch
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from transformers.trainer_utils import set_seed

from peft import (
    LoraConfig,
    TaskType,
)
from transformers import BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig


# ================= 1. 配置 =================

@dataclass
class QLoRAConfig:
    dataset_name: str          = "tomg-group-umd/cinepile"
    train_split: str           = "train"
    test_split: str            = "test"

    max_train_samples: Optional[int] = 30000
    max_test_samples: Optional[int]  = 500
    hard_oversample_ratio: float     = 1.5

    base_model_id: str         = "meta-llama/Llama-3.1-8B-Instruct"

    max_scene_length: int      = 1000
    max_seq_length: int        = 1024

    num_train_epochs: int      = 3
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float       = 2e-4
    warmup_ratio: float        = 0.05
    lr_scheduler_type: str     = "cosine"
    bf16: bool                 = True
    logging_steps: int         = 50

    output_dir: str            = "./qlora_cinepile"
    log_path: str              = "./qlora_experiment_log.csv"

    lora_r: int                = 16
    lora_alpha: int            = 32
    lora_dropout: float        = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])

    max_new_tokens: int        = 5

    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })

    seed: int                  = 42


# ================= 2. 分层抽样 + 数据集 =================

class StratifiedSampler:
    def __init__(self, cfg: QLoRAConfig):
        self.cfg = cfg

    def sample(self, data):
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples
        groups = defaultdict(list)

        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q       = base_quota
        hard_q       = base_quota * self.cfg.hard_oversample_ratio

        sampled = []
        report  = []

        import random
        random.seed(self.cfg.seed)

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {len(sampled)}")
        print("[Sampler] " + "=" * 60)
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"


class CinePileDataset:
    LETTERS = ["A", "B", "C", "D", "E"]

    def __init__(self, cfg: QLoRAConfig, seen_ids: Optional[Set[int]] = None):
        self.cfg      = cfg
        self.seen_ids = seen_ids or set()
        self.train_data, self.test_data = self._load_all()
        print(f"[Dataset] Train: {len(self.train_data)} | Test: {len(self.test_data)}")

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        for i, c in enumerate(choices):
            if answer_text.strip() == c.strip():
                return cls.LETTERS[i]
        for i, c in enumerate(choices):
            if answer_text.strip() in c or c.strip() in answer_text:
                return cls.LETTERS[i]
        return "A"

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == "true"

    def _load_split(self, split: str):
        raw = load_dataset(self.cfg.dataset_name, split=split)
        samples = []
        for idx, ex in enumerate(raw):
            letter = self._answer_text_to_letter(ex["answer_key"], ex["choices"])
            samples.append({
                "sample_id":         idx,
                "movie_scene":       ex["movie_scene"],
                "question":          ex["question"],
                "choices":           ex["choices"],
                "answer_key":        letter,
                "question_category": ex["question_category"],
                "hard_split":        self._normalize_hard_split(ex["hard_split"]),
            })
        return samples

    def _load_all(self):
        train_raw = self._load_split(self.cfg.train_split)
        test_raw  = self._load_split(self.cfg.test_split)

        # 过滤已用样本
        if self.seen_ids:
            before = len(train_raw)
            train_raw = [s for s in train_raw if s["sample_id"] not in self.seen_ids]
            print(f"[Dataset] Filtered seen samples: {before} → {len(train_raw)}")

        sampler    = StratifiedSampler(self.cfg)
        train_data = sampler.sample(train_raw)

        if self.cfg.max_test_samples:
            test_data = test_raw[:self.cfg.max_test_samples]
        else:
            test_data = test_raw

        return train_data, test_data


# ================= 3. Prompt 与评估 =================

class PromptBuilder:
    @staticmethod
    def build_train_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            f"Answer: {sample['answer_key']}"
        )

    @staticmethod
    def build_inference_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            "Answer:"
        )


class Evaluator:
    def __init__(self, cfg: QLoRAConfig):
        self.cfg = cfg

    def evaluate(self, model, tokenizer, test_data: list) -> dict:
        model.eval()
        device = next(model.parameters()).device
        records = []
        for s in tqdm(test_data, desc="Eval QLoRA"):
            pred = self._predict(model, tokenizer, s, device)
            records.append({
                "pred":              pred,
                "label":             s["answer_key"],
                "question_category": s["question_category"],
                "hard_split":        s["hard_split"],
            })
        df = pd.DataFrame(records)
        return self._compute_metrics(df)

    def _predict(self, model, tokenizer, sample: dict, device: torch.device) -> str:
        prompt = PromptBuilder.build_inference_prompt(sample, self.cfg.max_scene_length)
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.cfg.max_seq_length,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.cfg.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        decoded = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()

        for ch in decoded:
            if ch in "ABCDE":
                return ch
        return "A"

    def _compute_metrics(self, df: pd.DataFrame) -> dict:
        correct = df["pred"] == df["label"]
        metrics = {"Overall": correct.mean(), "Overall_n": len(df)}
        for abbr, keyword in self.cfg.category_map.items():
            mask = df["question_category"].str.contains(keyword, case=False, na=False)
            n    = int(mask.sum())
            metrics[abbr]        = correct[mask].mean() if n > 0 else None
            metrics[f"{abbr}_n"] = n
        hard_mask = df["hard_split"] == True
        n_hard    = int(hard_mask.sum())
        metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
        metrics["Hard_n"] = n_hard
        return metrics


# ================= 4. QLoRA Trainer =================

class QLoRATrainer:
    def __init__(self, cfg: QLoRAConfig):
        self.cfg       = cfg
        self.model     = None
        self.tokenizer = None

    def _output_dir(self) -> str:
        os.makedirs(self.cfg.output_dir, exist_ok=True)
        return self.cfg.output_dir

    def _make_hf_dataset(self, data: list) -> Dataset:
        return Dataset.from_list([
            {"text": PromptBuilder.build_train_prompt(s, self.cfg.max_scene_length)}
            for s in data
        ])

    def _load_base_model_4bit(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_use_double_quant = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = torch.bfloat16,
        )
        tokenizer = AutoTokenizer.from_pretrained(self.cfg.base_model_id)
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.padding_side = "right"

        model = AutoModelForCausalLM.from_pretrained(
            self.cfg.base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
        return model, tokenizer

    def train(self, train_data: list):
        print(f"\n[QLoRA] Loading base model (4-bit)...")
        model, tokenizer = self._load_base_model_4bit()

        lora_config = LoraConfig(
            r              = self.cfg.lora_r,
            lora_alpha     = self.cfg.lora_alpha,
            lora_dropout   = self.cfg.lora_dropout,
            target_modules = self.cfg.lora_target_modules,
            bias           = "none",
            task_type      = TaskType.CAUSAL_LM,
        )

        args = SFTConfig(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.cfg.num_train_epochs,
            per_device_train_batch_size = self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps = self.cfg.gradient_accumulation_steps,
            learning_rate               = self.cfg.learning_rate,
            warmup_ratio                = self.cfg.warmup_ratio,
            lr_scheduler_type           = self.cfg.lr_scheduler_type,
            bf16                        = self.cfg.bf16,
            logging_steps               = self.cfg.logging_steps,
            save_strategy               = "no",
            max_seq_length              = self.cfg.max_seq_length,
            dataset_text_field          = "text",
            report_to                   = "none",
            gradient_checkpointing      = True,
            optim                       = "paged_adamw_8bit",
        )

        hf_dataset = self._make_hf_dataset(train_data)
        trainer = SFTTrainer(
            model         = model,
            args          = args,
            train_dataset = hf_dataset,
            peft_config   = lora_config,
            tokenizer     = tokenizer,
        )

        print(f"[QLoRA] Training on {len(train_data)} samples...")
        trainer.train()

        log_history = trainer.state.log_history
        if len(log_history) > 0:
            df_log  = pd.DataFrame(log_history)
            log_path = os.path.join(self._output_dir(), "training_log_history.csv")
            df_log.to_csv(log_path, index=False)
            print(f"[QLoRA] Step-wise log saved → {log_path}")

        trainer.save_model(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[QLoRA] Saved adapter → {self._output_dir()}")

        self.model     = trainer.model
        self.tokenizer = tokenizer

    def free_memory(self):
        if self.model is not None:
            del self.model
        if self.tokenizer is not None:
            del self.tokenizer
        self.model = None
        self.tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()


# ================= 5. 入口（含 used_ids） =================

if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    from google.colab import drive
    drive.mount('/content/drive')

    import os
    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_QLORA"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)

    cfg = QLoRAConfig(
        base_model_id         = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples     = 30000,  # =None #完整训练集
        max_test_samples      = 500,    # =None #完整训练集
        hard_oversample_ratio = 1.5,
        num_train_epochs      = 3,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        learning_rate         = 2e-4,
        output_dir            = os.path.join(PROJECT_DIR, "qlora_cinepile"),
        log_path              = os.path.join(PROJECT_DIR, "qlora_experiment_log.csv"),
    )

    used_ids_path = os.path.join(PROJECT_DIR, "used_sample_ids_qlora.txt")

    set_seed(cfg.seed)

    # 读取已用 sample_ids
    seen_ids: Set[int] = set()
    if os.path.exists(used_ids_path):
        with open(used_ids_path, "r") as f:
            for line in f:
                try:
                    seen_ids.add(int(line.strip()))
                except:
                    pass
        print(f"[UsedIDs] QLoRA loaded {len(seen_ids)} seen ids from {used_ids_path}")

    dataset   = CinePileDataset(cfg, seen_ids=seen_ids)
    evaluator = Evaluator(cfg)
    trainer   = QLoRATrainer(cfg)

    trainer.train(dataset.train_data)
    metrics = evaluator.evaluate(trainer.model, trainer.tokenizer, dataset.test_data)

    print("\n[QLoRA] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "qlora_metrics.csv"), index=False
    )

    # 记录本轮使用的 sample_id，方便下一轮过滤
    with open(used_ids_path, "a") as f:
        for s in dataset.train_data:
            f.write(str(s["sample_id"]) + "\n")
    print(f"[UsedIDs] QLoRA appended {len(dataset.train_data)} ids → {used_ids_path}")

    trainer.free_memory()


## 2.3 Prefix Tuning

In [ ]:
!pip install "transformers==4.39.3" "accelerate==0.27.2" -U
!pip install "pyreft==0.0.7" -U
!pip install datasets bitsandbytes

  Using cached pyreft-0.0.7-py3-none-any.whl.metadata (16 kB)
  Using cached pyvene-0.1.8-py3-none-any.whl.metadata (4.5 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached ydata_profiling-4.18.1-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.3/293.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.5 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Prefix Tuning on CinePile (clean Trainer version)
#   - transformers Trainer + PEFT PrefixTuning
#   - Stratified sampling on CinePile (category × hard)
#   - CLM objective: prompt + answer letter
#   - Only save Prefix adapter (+ tokenizer)
#   - Track used sample_ids to avoid re-training on same samples
#   - training_log_history.csv + category/Hard metrics
# Requirements:
#   pip install transformers datasets accelerate peft bitsandbytes
# ============================================================

import os
import gc
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Set
from datetime import datetime
from collections import defaultdict

import torch
import torch.nn as nn
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from transformers.trainer_utils import set_seed

from peft import (
    PrefixTuningConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)


# ================= 1. Config =================

@dataclass
class PrefixConfig:
    dataset_name: str          = "tomg-group-umd/cinepile"
    train_split: str           = "train"
    test_split: str            = "test"

    max_train_samples: Optional[int] = 30000
    max_test_samples: Optional[int]  = 500
    hard_oversample_ratio: float     = 1.5

    base_model_id: str         = "meta-llama/Llama-3.1-8B-Instruct"

    max_scene_length: int      = 1000
    max_seq_length: int        = 1024

    num_train_epochs: float    = 3.0
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float       = 1e-3
    warmup_ratio: float        = 0.05
    bf16: bool                 = True
    logging_steps: int         = 50

    output_dir: str            = "./prefix_cinepile_trainer"
    log_path: str              = "./prefix_experiment_log_trainer.csv"

    prefix_num_virtual_tokens: int = 20
    prefix_projection: bool        = True

    max_new_tokens: int        = 5

    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })

    seed: int                  = 42


# ================= 2. Stratified Sampler + Dataset =================

class StratifiedSampler:
    def __init__(self, cfg: PrefixConfig):
        self.cfg = cfg

    def sample(self, data: List[dict]) -> List[dict]:
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples
        groups = defaultdict(list)

        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q       = base_quota
        hard_q       = base_quota * self.cfg.hard_oversample_ratio

        sampled = []
        report  = []

        import random
        random.seed(self.cfg.seed)

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {len(sampled)}")
        print("[Sampler] " + "=" * 60)
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"


LETTERS = ["A", "B", "C", "D", "E"]


def answer_text_to_letter(answer_text: str, choices: list) -> str:
    for i, c in enumerate(choices):
        if answer_text.strip() == c.strip():
            return LETTERS[i]
    for i, c in enumerate(choices):
        if answer_text.strip() in c or c.strip() in answer_text:
            return LETTERS[i]
    return "A"


def normalize_hard_split(raw) -> bool:
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() == "true"


def load_cinepile(
    cfg: PrefixConfig,
    seen_ids: Optional[Set[int]] = None,
):
    raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
    raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

    def _convert(ex, idx):
        return {
            "sample_id":         idx,
            "movie_scene":       ex["movie_scene"],
            "question":          ex["question"],
            "choices":           ex["choices"],
            "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
            "question_category": ex["question_category"],
            "hard_split":        normalize_hard_split(ex["hard_split"]),
        }

    train_data = [_convert(ex, i) for i, ex in enumerate(raw_train)]
    test_data  = [_convert(ex, i) for i, ex in enumerate(raw_test)]

    if seen_ids is not None and len(seen_ids) > 0:
        before = len(train_data)
        train_data = [s for s in train_data if s["sample_id"] not in seen_ids]
        print(f"[Dataset] Filtered seen samples: {before} → {len(train_data)}")

    sampler    = StratifiedSampler(cfg)
    train_data = sampler.sample(train_data)

    if cfg.max_test_samples:
        test_data = test_data[:cfg.max_test_samples]

    print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
    return train_data, test_data


# ================= 3. Prompts & Supervised Dataset =================

class PromptBuilder:
    @staticmethod
    def build_train_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            f"Answer: {sample['answer_key']}"
        )

    @staticmethod
    def build_inference_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            "Answer:"
        )


def make_supervised_dataset(train_data, tokenizer,
                            cfg: PrefixConfig) -> Dataset:
    texts = [
        PromptBuilder.build_train_prompt(s, cfg.max_scene_length)
        for s in train_data
    ]
    enc = tokenizer(
        texts,
        truncation=True,
        max_length=cfg.max_seq_length,
        padding=False,
    )
    max_len = max(len(ids) for ids in enc["input_ids"])

    input_ids_list, attention_mask_list, labels_list = [], [], []
    for ids, mask in zip(enc["input_ids"], enc["attention_mask"]):
        pad_len = max_len - len(ids)
        input_ids = ids + [tokenizer.pad_token_id] * pad_len
        attention_mask = mask + [0] * pad_len
        labels = ids + [-100] * pad_len  # full CLM, ignore pad
        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_mask_list,
        "labels":         labels_list,
    })


# ================= 4. Prefix Model Wrapper =================

class PrefixModelWrapper(nn.Module):
    def __init__(self, peft_model):
        super().__init__()
        self.model  = peft_model
        self.config = peft_model.config

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, **kwargs):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs,
        )

    def generate(self, *args, **kwargs):
        return self.model.generate(*args, **kwargs)

    def save(self, path: str):
        self.model.save_pretrained(path)


# ================= 5. Evaluation =================

def predict_choice(model_wrapper, tokenizer, sample,
                   cfg: PrefixConfig, device: str) -> str:
    model_wrapper.eval()
    prompt = PromptBuilder.build_inference_prompt(sample, cfg.max_scene_length)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(device)

    with torch.no_grad():
        outputs = model_wrapper.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().upper()

    for ch in decoded:
        if ch in "ABCDE":
            return ch
    return "A"


def evaluate_prefix(model_wrapper, tokenizer,
                    test_data: List[dict], cfg: PrefixConfig) -> dict:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    records = []
    for s in tqdm(test_data, desc="Eval Prefix Trainer"):
        pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
        records.append({
            "pred":              pred,
            "label":             s["answer_key"],
            "question_category": s["question_category"],
            "hard_split":        s["hard_split"],
        })

    df = pd.DataFrame(records)
    correct = df["pred"] == df["label"]
    metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

    for abbr, keyword in cfg.category_map.items():
        mask = df["question_category"].str.contains(keyword, case=False, na=False)
        n    = int(mask.sum())
        metrics[abbr]        = correct[mask].mean() if n > 0 else None
        metrics[f"{abbr}_n"] = n

    hard_mask = df["hard_split"] == True
    n_hard    = int(hard_mask.sum())
    metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
    metrics["Hard_n"] = n_hard
    return metrics


# ================= 6. Experiment Tracker =================

class ExperimentTracker:
    def __init__(self, log_path: str):
        self.log_path = log_path

    def log(self, cfg: PrefixConfig, metrics: dict) -> pd.DataFrame:
        record = {
            "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "base_model":          cfg.base_model_id.split("/")[-1],
            "max_train_samples":   cfg.max_train_samples if cfg.max_train_samples else "full",
            "max_test_samples":    cfg.max_test_samples  if cfg.max_test_samples  else "full",
            "hard_oversample":     cfg.hard_oversample_ratio,
            "num_train_epochs":    cfg.num_train_epochs,
            "prefix_tokens":       cfg.prefix_num_virtual_tokens,
            "learning_rate":       cfg.learning_rate,
            "batch_size":          cfg.per_device_train_batch_size,
            "grad_accum":          cfg.gradient_accumulation_steps,
            "Overall":             self._fmt(metrics.get("Overall")),
            "TEMP":                self._fmt(metrics.get("TEMP")),
            "CRD":                 self._fmt(metrics.get("CRD")),
            "NPA":                 self._fmt(metrics.get("NPA")),
            "STA":                 self._fmt(metrics.get("STA")),
            "TH":                  self._fmt(metrics.get("TH")),
            "Hard":                self._fmt(metrics.get("Hard")),
            "n_test":              metrics.get("Overall_n"),
            "n_TEMP":              metrics.get("TEMP_n"),
            "n_CRD":               metrics.get("CRD_n"),
            "n_NPA":               metrics.get("NPA_n"),
            "n_STA":               metrics.get("STA_n"),
            "n_TH":                metrics.get("TH_n"),
            "n_Hard":              metrics.get("Hard_n"),
            "output_dir":          cfg.output_dir,
        }
        df_new = pd.DataFrame([record])
        if os.path.exists(self.log_path):
            df_old = pd.read_csv(self.log_path)
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_all = df_new
        df_all.to_csv(self.log_path, index=False)
        print(f"\n[Tracker] Logged → {self.log_path}  (total: {len(df_all)})")
        return df_all

    @staticmethod
    def _fmt(val):
        if val is None:
            return None
        try:
            return round(float(val), 4)
        except Exception:
            return None


# ================= 7. 4-bit base model loader =================

def _load_quantized_base_model(cfg: PrefixConfig):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return model


# ================= 8. Training (supports resume) =================

def train_prefix_with_trainer(
    cfg: PrefixConfig,
    tracker: Optional[ExperimentTracker] = None,
    resume_from_prefix_dir: Optional[str] = None,
    used_ids_path: str = "./used_sample_ids_prefix_trainer.txt",
):
    os.makedirs(cfg.output_dir, exist_ok=True)

    # load used ids
    seen_ids: Set[int] = set()
    if os.path.exists(used_ids_path):
        with open(used_ids_path, "r") as f:
            for line in f:
                try:
                    seen_ids.add(int(line.strip()))
                except ValueError:
                    continue
        print(f"[UsedIDs] Loaded {len(seen_ids)} seen samples from {used_ids_path}")

    train_data, test_data = load_cinepile(cfg, seen_ids=seen_ids)

    print("[Prefix Trainer] Loading base model (4-bit)...")
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = _load_quantized_base_model(cfg)

    if resume_from_prefix_dir is None:
        print(f"[Prefix Trainer] Initializing new PrefixTuning model "
              f"(tokens={cfg.prefix_num_virtual_tokens})")
        prefix_config = PrefixTuningConfig(
            task_type          = TaskType.CAUSAL_LM,
            num_virtual_tokens = cfg.prefix_num_virtual_tokens,
            prefix_projection  = cfg.prefix_projection,
        )
        peft_model = get_peft_model(base_model, prefix_config)
    else:
        print(f"[Prefix Trainer] Resuming from prefix dir: {resume_from_prefix_dir}")
        peft_model = PeftModel.from_pretrained(
            base_model,
            resume_from_prefix_dir,
        )

    peft_model.print_trainable_parameters()
    model_wrapper = PrefixModelWrapper(peft_model)

    train_ds = make_supervised_dataset(train_data, tokenizer, cfg)
    print(f"[Prefix Trainer] Train dataset size: {len(train_ds)}")

    training_args = TrainingArguments(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_train_epochs,
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        learning_rate               = cfg.learning_rate,
        warmup_ratio                = cfg.warmup_ratio,
        bf16                        = cfg.bf16,
        logging_steps               = cfg.logging_steps,
        report_to                   = [],
        save_strategy               = "no",
    )

    trainer = Trainer(
        model         = model_wrapper,
        args          = training_args,
        train_dataset = train_ds,
    )

    print(f"[Prefix Trainer] Training... epochs={cfg.num_train_epochs}, "
          f"lr={cfg.learning_rate}, vt={cfg.prefix_num_virtual_tokens}")
    trainer.train()

    log_history = trainer.state.log_history
    if len(log_history) > 0:
        df_log = pd.DataFrame(log_history)
        log_path = os.path.join(cfg.output_dir, "training_log_history.csv")
        df_log.to_csv(log_path, index=False)
        print(f"[Prefix Trainer] Step-wise log saved → {log_path}")

    model_wrapper.save(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"[Prefix Trainer] Adapter saved → {cfg.output_dir}")

    with open(used_ids_path, "a") as f:
        for s in train_data:
            f.write(str(s["sample_id"]) + "\n")
    print(f"[UsedIDs] Appended {len(train_data)} ids → {used_ids_path}")

    metrics = evaluate_prefix(model_wrapper, tokenizer, test_data, cfg)
    print("\n[Prefix Trainer] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "prefix_trainer_metrics.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ================= 9. Load and full-test eval =================

def load_and_evaluate_prefix(
    output_dir: str,
    cfg: PrefixConfig,
    tracker: Optional[ExperimentTracker] = None,
):
    print(f"[Load Prefix Trainer] Tokenizer ← {output_dir}")
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"[Load Prefix Trainer] Base model ← {cfg.base_model_id}")
    base_model = _load_quantized_base_model(cfg)

    print(f"[Load Prefix Trainer] Adapter ← {output_dir}")
    peft_model = PeftModel.from_pretrained(base_model, output_dir)
    model_wrapper = PrefixModelWrapper(peft_model)

    train_data, test_data = load_cinepile(cfg, seen_ids=None)
    metrics = evaluate_prefix(model_wrapper, tokenizer, test_data, cfg)

    print("\n[Load Prefix Trainer] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(output_dir, "prefix_trainer_metrics_reeval.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ================= 10. Main =================

if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Trainer"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)

    tracker = ExperimentTracker(
        log_path=os.path.join(PROJECT_DIR, "prefix_trainer_experiment_log.csv")
    )

    cfg = PrefixConfig(
        base_model_id                = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples            = 200,
        max_test_samples             = 100,
        hard_oversample_ratio        = 1.0,
        num_train_epochs             = 3.0,
        per_device_train_batch_size  = 4,
        gradient_accumulation_steps  = 4,
        prefix_num_virtual_tokens    = 20,
        learning_rate                = 1e-3,
        output_dir                   = os.path.join(PROJECT_DIR, "prefix_trainer_20k_vt20_4bit"),
    )

    used_ids_path = os.path.join(PROJECT_DIR, "used_sample_ids_prefix_trainer.txt")

    set_seed(cfg.seed)

    # First train
    model1, tokenizer1, metrics1 = train_prefix_with_trainer(
        cfg,
        tracker=tracker,
        resume_from_prefix_dir=None,
        used_ids_path=used_ids_path,
    )

    del model1, tokenizer1
    gc.collect()
    torch.cuda.empty_cache()

    # Full test eval
    cfg_full_test = PrefixConfig(
        **{**cfg.__dict__, "max_test_samples": None}
    )
    model_eval, tok_eval, metrics_eval = load_and_evaluate_prefix(
        cfg.output_dir,
        cfg_full_test,
        tracker=tracker,
    )

    del model_eval, tok_eval
    gc.collect()
    torch.cuda.empty_cache()


transformers: 5.3.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Trainer
[UsedIDs] Loaded 200 seen samples from /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Trainer/used_sample_ids_prefix_trainer.txt
[Dataset] Filtered seen samples: 298888 → 298688

[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127394     20       20
     NPA False      32214     20       20
     STA False      85557     20       20
    TEMP False      41482     20       20
      TH False      12041     20       20
[Sampler] Total sampled: 200
[Sampler] ============================================================
[Dataset] Train: 200 | Test: 100
[Prefix Trainer] Loading base model (4-bit)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[Prefix Trainer] Initializing new PrefixTuning model (tokens=20)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 68,244,480 || all params: 8,098,505,728 || trainable%: 0.8427
[Prefix Trainer] Train dataset size: 200
[Prefix Trainer] Training... epochs=3.0, lr=0.001, vt=20


...


[Prefix Trainer] ===== Eval Metrics =====
  Overall   : 40.00%  (n=100)
  TEMP      : 15.38%  (n=13)
  CRD       : 50.00%  (n=42)
  NPA       : 60.00%  (n=5)
  STA       : 39.39%  (n=33)
  TH        : 14.29%  (n=7)
  Hard      : 30.77%  (n=26)

[Tracker] Logged → /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Trainer/prefix_trainer_experiment_log.csv  (total: 2)
[Load Prefix Trainer] Tokenizer ← /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Trainer/prefix_trainer_20k_vt20_4bit
[Load Prefix Trainer] Base model ← meta-llama/Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[Load Prefix Trainer] Adapter ← /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Trainer/prefix_trainer_20k_vt20_4bit

[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127468     20       20
     NPA False      32238     20       20
     STA False      85604     20       20
    TEMP False      41517     20       20
      TH False      12061     20       20
[Sampler] Total sampled: 200
[Sampler] ============================================================
[Dataset] Train: 200 | Test: 4941


Eval Prefix Trainer:   0%|          | 0/4941 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
Eval Prefix Trainer: 100%|██████████| 4941/4941 [33:30<00:00,  2.46it/s]



[Load Prefix Trainer] ===== Eval Metrics =====
  Overall   : 37.99%  (n=4941)
  TEMP      : 29.51%  (n=688)
  CRD       : 38.73%  (n=2068)
  NPA       : 38.66%  (n=463)
  STA       : 41.58%  (n=1532)
  TH        : 30.00%  (n=190)
  Hard      : 27.62%  (n=887)

[Tracker] Logged → /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Trainer/prefix_trainer_experiment_log.csv  (total: 3)


In [ ]:
### 2.3.1

### 2.3.1 Prefix Tuning V2

In [ ]:
!pip install transformers datasets accelerate
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# ============================================================
# Prefix Tuning Stable (4-bit) on CinePile
#   - Prefix tuning with PEFT (num_virtual_tokens)
#   - 4-bit NF4 quantized base model to avoid CUDA OOM
#   - Stratified sampling on CinePile (category × hard)
#   - Track used sample_ids to avoid re-training on same samples
#   - CLM objective: prompt + answer letter (full sequence loss)
#   - First train + optional second-stage train + full test eval
# Requirements:
#   pip install transformers peft datasets accelerate bitsandbytes
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Set
from datetime import datetime
from collections import defaultdict

from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from transformers.modeling_outputs import CausalLMOutputWithPast

from peft import (
    PrefixTuningConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)


# ============================================================
# 1. 配置
# ============================================================
@dataclass
class PrefixConfigCinePile:
    dataset_name: str                = "tomg-group-umd/cinepile"
    train_split: str                 = "train"
    test_split: str                  = "test"

    max_train_samples: Optional[int] = 30000
    max_test_samples: Optional[int]  = 500
    hard_oversample_ratio: float     = 1.5

    base_model_id: str               = "meta-llama/Llama-3.1-8B-Instruct"

    max_scene_length: int            = 1000
    max_seq_length: int              = 1024

    num_train_epochs: float          = 3.0
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float             = 1e-3
    warmup_ratio: float              = 0.05
    bf16: bool                       = True
    logging_steps: int               = 50
    output_dir: str                  = "./prefix_cinepile_stable_4bit"

    prefix_num_virtual_tokens: int   = 20
    prefix_projection: bool          = True

    max_new_tokens: int              = 5

    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


# ============================================================
# 2. Stratified Sampler
# ============================================================
class StratifiedSampler:
    def __init__(self, cfg: PrefixConfigCinePile):
        self.cfg = cfg

    def sample(self, data: list) -> list:
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples
        groups = defaultdict(list)

        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q       = base_quota
        hard_q       = base_quota * self.cfg.hard_oversample_ratio

        import random
        random.seed(42)

        sampled = []
        report  = []

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        self._print_report(report, len(sampled))
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"

    @staticmethod
    def _print_report(report: list, total_sampled: int):
        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {total_sampled}")
        print("[Sampler] " + "=" * 60)


# ============================================================
# 3. ExperimentTracker
# ============================================================
class ExperimentTracker:
    def __init__(self, log_path: str = "./prefix_experiment_log.csv"):
        self.log_path = log_path

    def log(self, cfg: PrefixConfigCinePile, metrics: dict) -> pd.DataFrame:
        record = {
            "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "base_model":          cfg.base_model_id.split("/")[-1],
            "max_train_samples":   cfg.max_train_samples if cfg.max_train_samples else "full",
            "max_test_samples":    cfg.max_test_samples  if cfg.max_test_samples  else "full",
            "hard_oversample":     cfg.hard_oversample_ratio,
            "num_train_epochs":    cfg.num_train_epochs,
            "prefix_tokens":       cfg.prefix_num_virtual_tokens,
            "learning_rate":       cfg.learning_rate,
            "batch_size":          cfg.per_device_train_batch_size,
            "grad_accum":          cfg.gradient_accumulation_steps,
            "Overall":             self._fmt(metrics.get("Overall")),
            "TEMP":                self._fmt(metrics.get("TEMP")),
            "CRD":                 self._fmt(metrics.get("CRD")),
            "NPA":                 self._fmt(metrics.get("NPA")),
            "STA":                 self._fmt(metrics.get("STA")),
            "TH":                  self._fmt(metrics.get("TH")),
            "Hard":                self._fmt(metrics.get("Hard")),
            "n_test":              metrics.get("Overall_n"),
            "n_TEMP":              metrics.get("TEMP_n"),
            "n_CRD":               metrics.get("CRD_n"),
            "n_NPA":               metrics.get("NPA_n"),
            "n_STA":               metrics.get("STA_n"),
            "n_TH":                metrics.get("TH_n"),
            "n_Hard":              metrics.get("Hard_n"),
            "output_dir":          cfg.output_dir,
        }
        df_new = pd.DataFrame([record])
        if os.path.exists(self.log_path):
            df_old = pd.read_csv(self.log_path)
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_all = df_new
        df_all.to_csv(self.log_path, index=False)
        print(f"\n[Tracker] Logged → {self.log_path}  (total: {len(df_all)})")
        return df_all

    @staticmethod
    def _fmt(val):
        if val is None:
            return None
        try:
            return round(float(val), 4)
        except Exception:
            return None


# ============================================================
# 4. Prefix 模型包装类
# ============================================================
class PrefixModelWrapper(nn.Module):
    def __init__(self, peft_model):
        super().__init__()
        self.model  = peft_model
        self.config = peft_model.config

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, **kwargs):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs,
        )

    def generate(self, *args, **kwargs):
        return self.model.generate(*args, **kwargs)

    def save(self, path: str):
        self.model.save_pretrained(path)


# ============================================================
# 5. 数据加载（带 sample_id）
# ============================================================
LETTERS = ["A", "B", "C", "D", "E"]


def answer_text_to_letter(answer_text: str, choices: list) -> str:
    for i, c in enumerate(choices):
        if answer_text.strip() == c.strip():
            return LETTERS[i]
    for i, c in enumerate(choices):
        if answer_text.strip() in c or c.strip() in answer_text:
            return LETTERS[i]
    return "A"


def normalize_hard_split(raw) -> bool:
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() == "true"


def load_cinepile(
    cfg: PrefixConfigCinePile,
    seen_ids: Optional[Set[int]] = None,
):
    raw_train = load_dataset(cfg.dataset_name, split=cfg.train_split)
    raw_test  = load_dataset(cfg.dataset_name, split=cfg.test_split)

    def _convert(ex, idx):
        return {
            "sample_id":         idx,
            "movie_scene":       ex["movie_scene"],
            "question":          ex["question"],
            "choices":           ex["choices"],
            "answer_key":        answer_text_to_letter(ex["answer_key"], ex["choices"]),
            "question_category": ex["question_category"],
            "hard_split":        normalize_hard_split(ex["hard_split"]),
        }

    train_data = [_convert(ex, i) for i, ex in enumerate(raw_train)]
    test_data  = [_convert(ex, i) for i, ex in enumerate(raw_test)]

    if seen_ids is not None and len(seen_ids) > 0:
        before = len(train_data)
        train_data = [s for s in train_data if s["sample_id"] not in seen_ids]
        print(f"[Dataset] Filtered seen samples: {before} → {len(train_data)}")

    sampler    = StratifiedSampler(cfg)
    train_data = sampler.sample(train_data)

    if cfg.max_test_samples:
        test_data = test_data[:cfg.max_test_samples]

    print(f"[Dataset] Train: {len(train_data)} | Test: {len(test_data)}")
    return train_data, test_data


# ============================================================
# 6. Prompt & 训练数据集（简单 CLM：prompt + letter）
# ============================================================
def build_inference_prompt(sample, max_scene_length: int) -> str:
    scene   = sample["movie_scene"][:max_scene_length]
    options = "\n".join(
        f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
    )
    return (
        "You are watching a movie. Based on the scene description, "
        "answer the multiple choice question.\n\n"
        f"Scene: {scene}\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:"
    )


def make_supervised_dataset(train_data, tokenizer,
                            cfg: PrefixConfigCinePile) -> Dataset:
    texts = [
        build_inference_prompt(s, cfg.max_scene_length) + " " + s["answer_key"]
        for s in train_data
    ]
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=cfg.max_seq_length,
        padding=False,
    )

    max_len = max(len(ids) for ids in encodings["input_ids"])
    input_ids_list, attention_mask_list, labels_list = [], [], []

    for ids, mask in zip(encodings["input_ids"], encodings["attention_mask"]):
        pad_len = max_len - len(ids)
        input_ids = ids + [tokenizer.pad_token_id] * pad_len
        attention_mask = mask + [0] * pad_len
        labels = ids + [-100] * pad_len

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_mask_list,
        "labels":         labels_list,
    })


# ============================================================
# 7. 评估：扫描生成结果中的 A–E
# ============================================================
def predict_choice(model_wrapper, tokenizer, sample,
                   cfg: PrefixConfigCinePile, device: str) -> str:
    model_wrapper.eval()
    prompt = build_inference_prompt(sample, cfg.max_scene_length)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(device)

    with torch.no_grad():
        outputs = model_wrapper.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().upper()
    for ch in decoded:
        if ch in "ABCDE":
            return ch
    return "A"


def evaluate_prefix(model_wrapper, tokenizer,
                    test_data: list, cfg: PrefixConfigCinePile) -> dict:
    device  = "cuda" if torch.cuda.is_available() else "cpu"
    records = []
    for s in tqdm(test_data, desc="Eval Prefix Stable 4-bit"):
        pred = predict_choice(model_wrapper, tokenizer, s, cfg, device)
        records.append({
            "pred":              pred,
            "label":             s["answer_key"],
            "question_category": s["question_category"],
            "hard_split":        s["hard_split"],
        })

    df      = pd.DataFrame(records)
    correct = df["pred"] == df["label"]
    metrics = {"Overall": correct.mean(), "Overall_n": len(df)}

    for abbr, keyword in cfg.category_map.items():
        mask = df["question_category"].str.contains(keyword, case=False, na=False)
        n    = int(mask.sum())
        metrics[abbr]        = correct[mask].mean() if n > 0 else None
        metrics[f"{abbr}_n"] = n

    hard_mask = df["hard_split"] == True
    n_hard    = int(hard_mask.sum())
    metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
    metrics["Hard_n"] = n_hard
    return metrics


# ============================================================
# 8. 4-bit base model loader
# ============================================================
def _load_quantized_base_model(cfg: PrefixConfigCinePile):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return model


# ============================================================
# 9. 训练主函数（支持续训）
# ============================================================
def train_prefix_with_trainer(
    cfg: PrefixConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
    resume_from_prefix_dir: Optional[str] = None,
    used_ids_path: str = "./used_sample_ids_prefix_stable.txt",
):
    os.makedirs(cfg.output_dir, exist_ok=True)

    seen_ids: Set[int] = set()
    if os.path.exists(used_ids_path):
        with open(used_ids_path, "r") as f:
            for line in f:
                try:
                    seen_ids.add(int(line.strip()))
                except ValueError:
                    continue
        print(f"[UsedIDs] Loaded {len(seen_ids)} seen samples from {used_ids_path}")

    train_data, test_data = load_cinepile(cfg, seen_ids=seen_ids)

    print("[Prefix Stable 4-bit] Loading quantized base model...")
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = _load_quantized_base_model(cfg)

    if resume_from_prefix_dir is None:
        print(f"[Prefix Stable 4-bit] Initializing new PrefixTuning model "
              f"(tokens={cfg.prefix_num_virtual_tokens})")
        prefix_config = PrefixTuningConfig(
            task_type          = TaskType.CAUSAL_LM,
            num_virtual_tokens = cfg.prefix_num_virtual_tokens,
            prefix_projection  = cfg.prefix_projection,
        )
        peft_model = get_peft_model(base_model, prefix_config)
    else:
        print(f"[Prefix Stable 4-bit] Resuming from prefix dir: {resume_from_prefix_dir}")
        peft_model = PeftModel.from_pretrained(
            base_model,
            resume_from_prefix_dir,
        )

    peft_model.print_trainable_parameters()
    model_wrapper = PrefixModelWrapper(peft_model)

    train_ds = make_supervised_dataset(train_data, tokenizer, cfg)
    print(f"[Prefix Stable 4-bit] Train dataset size: {len(train_ds)}")

    training_args = TrainingArguments(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_train_epochs,
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        learning_rate               = cfg.learning_rate,
        warmup_ratio                = cfg.warmup_ratio,
        bf16                        = cfg.bf16,
        logging_steps               = cfg.logging_steps,
        report_to                   = [],
        save_strategy               = "no",
    )

    trainer = Trainer(
        model         = model_wrapper,
        args          = training_args,
        train_dataset = train_ds,
    )

    print(f"[Prefix Stable 4-bit] Training... epochs={cfg.num_train_epochs}, "
          f"lr={cfg.learning_rate}, vt={cfg.prefix_num_virtual_tokens}")
    trainer.train()

    log_history = trainer.state.log_history
    if len(log_history) > 0:
        df_log = pd.DataFrame(log_history)
        log_path = os.path.join(cfg.output_dir, "training_log_history.csv")
        df_log.to_csv(log_path, index=False)
        print(f"[Prefix Stable 4-bit] Step-wise log saved → {log_path}")

    model_wrapper.save(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"[Prefix Stable 4-bit] Adapter saved → {cfg.output_dir}")

    with open(used_ids_path, "a") as f:
        for s in train_data:
            f.write(str(s["sample_id"]) + "\n")
    print(f"[UsedIDs] Appended {len(train_data)} ids → {used_ids_path}")

    metrics = evaluate_prefix(model_wrapper, tokenizer, test_data, cfg)
    print("\n[Prefix Stable 4-bit] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "prefix_stable_metrics.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 10. 加载并在完整 test 上评估
# ============================================================
def load_and_evaluate_prefix(
    output_dir: str,
    cfg: PrefixConfigCinePile,
    tracker: Optional[ExperimentTracker] = None,
):
    print(f"[Load Prefix Stable 4-bit] Tokenizer ← {output_dir}")
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"[Load Prefix Stable 4-bit] Base model ← {cfg.base_model_id}")
    base_model = _load_quantized_base_model(cfg)

    print(f"[Load Prefix Stable 4-bit] Adapter ← {output_dir}")
    peft_model = PeftModel.from_pretrained(base_model, output_dir)
    model_wrapper = PrefixModelWrapper(peft_model)

    train_data, test_data = load_cinepile(cfg, seen_ids=None)
    metrics = evaluate_prefix(model_wrapper, tokenizer, test_data, cfg)

    print("\n[Load Prefix Stable 4-bit] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(output_dir, "prefix_stable_metrics_reeval.csv"), index=False
    )
    if tracker is not None:
        tracker.log(cfg, metrics)

    return model_wrapper, tokenizer, metrics


# ============================================================
# 11. 入口示例
# ============================================================
if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Stable_4bit"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)

    tracker = ExperimentTracker(
        log_path=os.path.join(PROJECT_DIR, "prefix_stable_experiment_log.csv")
    )

    cfg = PrefixConfigCinePile(
        base_model_id                = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples            = 200,
        max_test_samples             = 100,
        hard_oversample_ratio        = 1.0,
        num_train_epochs             = 3.0,
        per_device_train_batch_size  = 4,
        gradient_accumulation_steps  = 4,
        prefix_num_virtual_tokens    = 20,
        learning_rate                = 1e-3,
        output_dir                   = os.path.join(PROJECT_DIR, "prefix_stable_20k_vt20_4bit"),
    )

    used_ids_path = os.path.join(PROJECT_DIR, "used_sample_ids_prefix_stable.txt")

    model1, tokenizer1, metrics1 = train_prefix_with_trainer(
        cfg,
        tracker=tracker,
        resume_from_prefix_dir=None,
        used_ids_path=used_ids_path,
    )

    del model1, tokenizer1
    gc.collect()
    torch.cuda.empty_cache()

    cfg_full_test = PrefixConfigCinePile(
        **{**cfg.__dict__, "max_test_samples": None}
    )
    model_eval, tok_eval, metrics_eval = load_and_evaluate_prefix(
        cfg.output_dir,
        cfg_full_test,
        tracker=tracker,
    )

    del model_eval, tok_eval
    gc.collect()
    torch.cuda.empty_cache()


transformers: 5.0.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Stable_4bit

[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127468     20       20
     NPA False      32238     20       20
     STA False      85604     20       20
    TEMP False      41517     20       20
      TH False      12061     20       20
[Sampler] Total sampled: 200
[Sampler] ============================================================
[Dataset] Train: 200 | Test: 100
[Prefix Stable 4-bit] Loading quantized base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[Prefix Stable 4-bit] Initializing new PrefixTuning model (tokens=20)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 68,244,480 || all params: 8,098,505,728 || trainable%: 0.8427
[Prefix Stable 4-bit] Train dataset size: 200
[Prefix Stable 4-bit] Training... epochs=3.0, lr=0.001, vt=20


...

Eval Prefix Stable 4-bit:   0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
Eval Prefix Stable 4-bit: 100%|██████████| 100/100 [00:40<00:00,  2.45it/s]



[Prefix Stable 4-bit] ===== Eval Metrics =====
  Overall   : 9.00%  (n=100)
  TEMP      : 7.69%  (n=13)
  CRD       : 9.52%  (n=42)
  NPA       : 0.00%  (n=5)
  STA       : 12.12%  (n=33)
  TH        : 0.00%  (n=7)
  Hard      : 3.85%  (n=26)

[Tracker] Logged → /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Stable_4bit/prefix_stable_experiment_log.csv  (total: 1)
[Load Prefix Stable 4-bit] Tokenizer ← /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Stable_4bit/prefix_stable_20k_vt20_4bit
[Load Prefix Stable 4-bit] Base model ← meta-llama/Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[Load Prefix Stable 4-bit] Adapter ← /content/drive/MyDrive/266_final/phase2/CinePile_Prefix_Stable_4bit/prefix_stable_20k_vt20_4bit


KeyboardInterrupt: 

#### debug

In [ ]:
# test_tokenization_prefix.py

import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# from prefix_cinepile_v3_4bit import (
#     PrefixConfigCinePile,
#     build_inference_prompt,
#     make_supervised_dataset,
# )

cfg = PrefixConfigCinePile(
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    max_scene_length=400,
    max_seq_length=256,
)

tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load a few raw examples
raw = load_dataset(cfg.dataset_name, split=cfg.test_split)
sample = raw[0]

# Build our internal sample dict
# from prefix_cinepile_v3_4bit import answer_text_to_letter, normalize_hard_split
s = {
    "sample_id": 0,
    "movie_scene": sample["movie_scene"],
    "question": sample["question"],
    "choices": sample["choices"],
    "answer_key": answer_text_to_letter(sample["answer_key"], sample["choices"]),
    "question_category": sample["question_category"],
    "hard_split": normalize_hard_split(sample["hard_split"]),
}

prompt = build_inference_prompt(s, cfg.max_scene_length)
print("PROMPT:\n", prompt)

# Run through dataset builder with just one sample
ds = make_supervised_dataset([s], tokenizer, cfg)
ids = ds[0]["input_ids"]
labels = ds[0]["labels"]

# Decode tokens and labels
decoded_input = tokenizer.decode([t for t in ids if t != tokenizer.pad_token_id])
print("\nDECODED INPUT:\n", decoded_input)

print("\nLABEL POSITIONS (index, token, decoded):")
for i, lab in enumerate(labels):
    if lab != -100:
        print(i, lab, tokenizer.decode([lab]))


PROMPT:
 You are watching a movie. Based on the scene description, answer the multiple choice question.

Scene: <subtitle> 5, 4, 3, 2, 1.
<subtitle> All right, I think I got it, guys.
<subtitle> You did it?
<subtitle> Yeah, I got it.
<visual descriptions> Darren moves up behind Reed and observes the broken sensor.
<subtitle> All right, let's keep moving, guys.
<subtitle> It's three miles that way, a little closer.
<subtitle> You hear that?
<subtitle> You hear that?
<subtitle> What is that?
<subtitle> Are th

Question: How does the emotional tone transition during the scene?

Options:
A. From despair to hope
B. From fear to acceptance
C. From confusion to understanding
D. From tension to panic
E. From anxiety to excitement

Answer:

DECODED INPUT:
 <|begin_of_text|>You are watching a movie. Based on the scene description, answer the multiple choice question.

Scene: <subtitle> 5, 4, 3, 2, 1.
<subtitle> All right, I think I got it, guys.
<subtitle> You did it?
<subtitle> Yeah, I got it.


In [ ]:
# -*- coding: utf-8 -*-
"""
Llama-3.1-8B on CinePile (text-only) + Roles of Layer Components visualization
-----------------------------------------------------------------------------

- 加载 CinePile，并把 answer_key(全文) 映射到 A~E
- 用 Llama-3.1-8B-Instruct 做 MCQ 前向，把 Answer: 位置的 logits 作为“答案符号”表示
- 实现 activation patching:
    clean vs corrupted(prompt 中选项字母打乱)
    per-layer:
        residual(近似 whole layer)、attention-only、MLP-only
- 生成 CSV + matplotlib/seaborn 图，类似论文中的 “Roles of Layer Components” 图
"""

!pip install -U bitsandbytes>=0.46.1

import os
import gc
import math
import random
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple

import torch
from torch import nn
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Config
# ============================================================

@dataclass
class ExperimentConfig:
    dataset_name: str          = "tomg-group-umd/cinepile"
    dataset_split: str         = "test"
    max_samples: Optional[int] = 200   # 先小点，激活 patching 非常慢

    max_scene_length: int      = 1500
    max_input_tokens: int      = 2048

    # Llama-3.1-8B
    llama_model_id: str        = "meta-llama/Llama-3.1-8B-Instruct"

    load_in_4bit: bool         = True
    bnb_4bit_quant_type: str   = "nf4"
    bnb_4bit_compute_dtype: torch.dtype = torch.bfloat16

    # output
    out_dir: str               = "results_llama31_roles"
    csv_layer_roles: str       = "layer_component_roles.csv"

    # corruption
    corruption: str            = "shuffle_options"

# cfg = ExperimentConfig()

cfg = ExperimentConfig(
    max_samples=10,        # 只取前 10 条样本做 patching
    max_scene_length=500,  # 可选：缩短场景，减负
    max_input_tokens=1024  # 可选：缩短上下文长度
)

os.makedirs(cfg.out_dir, exist_ok=True)

# ============================================================
# 2. CinePile dataset helper (与原代码一致的 answer_key→A~E)
# ============================================================

class CinePileDataset:
    LETTERS = ['A', 'B', 'C', 'D', 'E']

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.data   = self._load()

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        # 精确匹配
        for i, choice in enumerate(choices):
            if answer_text == choice:
                return cls.LETTERS[i]
        # strip 后匹配
        answer_stripped = answer_text.strip()
        for i, choice in enumerate(choices):
            if answer_stripped == choice.strip():
                return cls.LETTERS[i]
        # 包含匹配
        for i, choice in enumerate(choices):
            if answer_stripped in choice or choice.strip() in answer_stripped:
                return cls.LETTERS[i]
        print("[Warning] Cannot match answer_key to choices.")
        print("  answer_key :", repr(answer_text))
        print("  choices    :", choices)
        return 'A'

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == "true"

    def _load(self):
        raw = load_dataset(
            self.config.dataset_name,
            split=self.config.dataset_split
        )
        samples = []
        unmatched = 0

        for ex in raw:
            choices    = ex['choices']
            answer_key = ex['answer_key']

            letter = self._answer_text_to_letter(answer_key, choices)
            if letter == 'A' and answer_key != choices[0]:
                unmatched += 1

            samples.append({
                'movie_scene':       ex['movie_scene'],
                'question':          ex['question'],
                'choices':           choices,
                'answer_key':        letter,         # 'A'~'E'
                'answer_text':       answer_key,
                'question_category': ex['question_category'],
                'hard_split':        self._normalize_hard_split(ex['hard_split']),
            })

        if self.config.max_samples:
            samples = samples[:self.config.max_samples]

        print(f"[Dataset] Loaded {len(samples)} samples")
        print(f"          answer_key sample : {repr(samples[0]['answer_text'])} → {samples[0]['answer_key']}")
        print(f"          hard_split sample : {samples[0]['hard_split']}")
        if unmatched > 0:
            print(f"  [Warning] {unmatched} samples fell back to 'A' (answer not matched in choices)")
        return samples

    def run_sanity_check(self, n: int = 3):
        print("\n=== Sanity Check ===")
        for i, s in enumerate(self.data[:n]):
            idx = ord(s['answer_key']) - 65
            matched_text = s['choices'][idx]
            match_ok = matched_text.strip() == s['answer_text'].strip()
            print(f"  [{i}] {s['answer_key']} → \"{matched_text}\"")
            print(f"       original: \"{s['answer_text']}\"  match={match_ok}")
        print()

# ============================================================
# 3. Llama-3.1-8B loading + prompt构造（logit-based MCQ，而不是 generate）
# ============================================================

def load_llama_model(config: ExperimentConfig):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit             = config.load_in_4bit,
        bnb_4bit_use_double_quant= True,
        bnb_4bit_quant_type      = config.bnb_4bit_quant_type,
        bnb_4bit_compute_dtype   = config.bnb_4bit_compute_dtype,
    )
    tokenizer = AutoTokenizer.from_pretrained(config.llama_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        config.llama_model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.eval()
    print(f"[Loaded] {config.llama_model_id} (4-bit)")
    return model, tokenizer

def build_prompt_mcq(sample: dict, max_scene_length: int) -> str:
    scene = sample['movie_scene'][:max_scene_length]
    # 这里保留你原来的风格，只是最后不再“让模型生成”，而是取 Answer: 位置 logits
    options = "\n".join(
        f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
    )
    prompt = (
        "You are watching a movie. Based on the scene description, "
        "answer the multiple choice question.\n\n"
        f"Scene: {scene}\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        "Answer with only the letter (A, B, C, D, or E):\n"
        "Answer:"
    )
    return prompt

# ============================================================
# 4. Activation cache + hooks (与之前给你的版本一致，但改为 Llama-3.1)
# ============================================================

class ActivationCache:
    def __init__(self, num_layers: int):
        self.num_layers = num_layers
        self.reset()
    def reset(self):
        self.residual_in  = [None] * self.num_layers
        self.attn_out     = [None] * self.num_layers
        self.mlp_out      = [None] * self.num_layers
        self.residual_out = [None] * self.num_layers

def register_hooks(model: nn.Module,
                   cache: ActivationCache,
                   patch_source_cache: Optional[ActivationCache] = None,
                   patch_layer: Optional[int] = None,
                   patch_component: Optional[str] = None):
    """
    patch_component ∈ {None, "residual", "attn", "mlp"}
    None 表示用 residual_out 近似整个 block 的 patch。
    """
    hooks = []

    def make_block_hook(layer_idx):
        def forward_hook(block, inputs, output):
            x_in  = inputs[0]   # residual_in
            x_out = output      # residual_out
            cache.residual_in[layer_idx]  = x_in.detach()
            cache.residual_out[layer_idx] = x_out.detach()
            if (patch_source_cache is not None) and (patch_layer == layer_idx) and (patch_component in [None, "residual"]):
                with torch.no_grad():
                    x_out = patch_source_cache.residual_out[layer_idx]
                return x_out
            return x_out
        return forward_hook

    def make_attn_hook(layer_idx):
        def attn_forward_hook(module, inputs, output):
            cache.attn_out[layer_idx] = output.detach()
            if (patch_source_cache is not None) and (patch_layer == layer_idx) and (patch_component == "attn"):
                with torch.no_grad():
                    return patch_source_cache.attn_out[layer_idx]
            return output
        return attn_forward_hook

    def make_mlp_hook(layer_idx):
        def mlp_forward_hook(module, inputs, output):
            cache.mlp_out[layer_idx] = output.detach()
            if (patch_source_cache is not None) and (patch_layer == layer_idx) and (patch_component == "mlp"):
                with torch.no_grad():
                    return patch_source_cache.mlp_out[layer_idx]
            return output
        return mlp_forward_hook

    for i, block in enumerate(model.model.layers):
        hooks.append(block.register_forward_hook(make_block_hook(i)))
        if hasattr(block, "self_attn"):
            hooks.append(block.self_attn.register_forward_hook(make_attn_hook(i)))
        elif hasattr(block, "attention"):
            hooks.append(block.attention.register_forward_hook(make_attn_hook(i)))
        if hasattr(block, "mlp"):
            hooks.append(block.mlp.register_forward_hook(make_mlp_hook(i)))

    return hooks

def remove_hooks(hooks):
    for h in hooks:
        h.remove()

# ============================================================
# 5. Forward helpers：找到 Answer: 位置 logits + label token ids
# ============================================================

def get_label_token_ids(tokenizer: AutoTokenizer,
                        option_letters: List[str]) -> Dict[str, int]:
    mapping = {}
    for lab in option_letters:
        tok = tokenizer(" " + lab, add_special_tokens=False)
        assert len(tok.input_ids) == 1, f"Label {lab} not single-token: {tok.input_ids}"
        mapping[lab] = tok.input_ids[0]
    return mapping

def get_answer_position(tokenizer: AutoTokenizer,
                        input_ids: torch.Tensor) -> int:
    answer_str = "Answer:"
    ans_ids = tokenizer(answer_str, add_special_tokens=False).input_ids
    ids = input_ids[0].tolist()
    for i in range(len(ids) - len(ans_ids)):
        if ids[i: i + len(ans_ids)] == ans_ids:
            return i + len(ans_ids)
    return len(ids) - 1

@torch.no_grad()
def forward_with_cache(prompt: str,
                       model: AutoModelForCausalLM,
                       tokenizer: AutoTokenizer,
                       cache: ActivationCache,
                       max_len: int) -> Tuple[torch.Tensor, Dict]:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
    ).to(model.device)
    hooks = register_hooks(model, cache)
    outputs = model(**inputs)
    remove_hooks(hooks)
    logits = outputs.logits
    return logits, inputs

@torch.no_grad()
def forward_with_patching(prompt: str,
                          model: AutoModelForCausalLM,
                          tokenizer: AutoTokenizer,
                          patch_source_cache: ActivationCache,
                          patch_layer: int,
                          patch_component: Optional[str],
                          max_len: int) -> Tuple[torch.Tensor, Dict]:
    cache = ActivationCache(patch_source_cache.num_layers)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
    ).to(model.device)
    hooks = register_hooks(
        model, cache,
        patch_source_cache=patch_source_cache,
        patch_layer=patch_layer,
        patch_component=patch_component
    )
    outputs = model(**inputs)
    remove_hooks(hooks)
    logits = outputs.logits
    return logits, inputs

def extract_label_logits(logits: torch.Tensor,
                         inputs,
                         tokenizer: AutoTokenizer,
                         label_token_ids: Dict[str, int],
                         option_letters: List[str]) -> Dict[str, float]:
    pos = get_answer_position(tokenizer, inputs["input_ids"])
    last_logits = logits[0, pos, :]
    out = {}
    for lab in option_letters:
        tid = label_token_ids[lab]
        out[lab] = last_logits[tid].item()
    return out

# ============================================================
# 6. Corruption: shuffle option letters (保持文本不变，只换字母)
# ============================================================

def corrupt_prompt_shuffle_options(prompt: str,
                                   option_letters: List[str]) -> Tuple[str, Dict[str, str]]:
    shuffled = option_letters.copy()
    random.shuffle(shuffled)
    mapping = {orig: new for orig, new in zip(option_letters, shuffled)}

    corrupted_lines = []
    for line in prompt.splitlines():
        for lab in option_letters:
            token_dot = f"{lab}."
            if token_dot in line:
                line = line.replace(token_dot, f"{mapping[lab]}.")
        corrupted_lines.append(line)
    return "\n".join(corrupted_lines), mapping

# ============================================================
# 7. 主实验：layer/component activation patching
# ============================================================

def run_roles_of_layer_components():
    # 载数据
    dataset = CinePileDataset(cfg)
    dataset.run_sanity_check(n=3)

    # 加载 Llama-3.1-8B
    model, tokenizer = load_llama_model(cfg)
    num_layers = model.config.num_hidden_layers

    option_letters = ['A', 'B', 'C', 'D', 'E']
    label_token_ids = get_label_token_ids(tokenizer, option_letters)

    layer_contrib_residual = [0.0 for _ in range(num_layers)]
    layer_contrib_attn     = [0.0 for _ in range(num_layers)]
    layer_contrib_mlp      = [0.0 for _ in range(num_layers)]
    layer_counts           = [0   for _ in range(num_layers)]

    for idx, sample in enumerate(tqdm(dataset.data, desc="Activation patching")):
        # 构造 clean / corrupt prompt
        prompt_clean = build_prompt_mcq(sample, cfg.max_scene_length)
        prompt_corrupt, _ = corrupt_prompt_shuffle_options(prompt_clean, option_letters)

        # clean run
        clean_cache = ActivationCache(num_layers)
        clean_logits, clean_inputs = forward_with_cache(
            prompt_clean, model, tokenizer, clean_cache, cfg.max_input_tokens
        )
        clean_label_logits = extract_label_logits(
            clean_logits, clean_inputs, tokenizer, label_token_ids, option_letters
        )
        correct_label = sample['answer_key']
        clean_correct_logit = clean_label_logits[correct_label]

        # corrupted run
        corrupt_cache = ActivationCache(num_layers)
        corrupt_logits, corrupt_inputs = forward_with_cache(
            prompt_corrupt, model, tokenizer, corrupt_cache, cfg.max_input_tokens
        )
        corrupt_label_logits = extract_label_logits(
            corrupt_logits, corrupt_inputs, tokenizer, label_token_ids, option_letters
        )
        corrupt_correct_logit = corrupt_label_logits[correct_label]

        baseline_diff = clean_correct_logit - corrupt_correct_logit

        # 可选：如果 baseline_diff < 0 说明模型本身就没“纠偏”，可以跳过
        # if baseline_diff < 0:
        #     continue

        for layer in range(num_layers):
            # residual/whole-layer patch
            patched_logits_res, patched_inputs_res = forward_with_patching(
                prompt_corrupt, model, tokenizer, clean_cache,
                patch_layer=layer, patch_component=None,
                max_len=cfg.max_input_tokens
            )
            patched_label_logits_res = extract_label_logits(
                patched_logits_res, patched_inputs_res, tokenizer, label_token_ids, option_letters
            )
            contrib_res = patched_label_logits_res[correct_label] - corrupt_correct_logit

            # attention-only patch
            patched_logits_attn, patched_inputs_attn = forward_with_patching(
                prompt_corrupt, model, tokenizer, clean_cache,
                patch_layer=layer, patch_component="attn",
                max_len=cfg.max_input_tokens
            )
            patched_label_logits_attn = extract_label_logits(
                patched_logits_attn, patched_inputs_attn, tokenizer, label_token_ids, option_letters
            )
            contrib_attn = patched_label_logits_attn[correct_label] - corrupt_correct_logit

            # mlp-only patch
            patched_logits_mlp, patched_inputs_mlp = forward_with_patching(
                prompt_corrupt, model, tokenizer, clean_cache,
                patch_layer=layer, patch_component="mlp",
                max_len=cfg.max_input_tokens
            )
            patched_label_logits_mlp = extract_label_logits(
                patched_logits_mlp, patched_inputs_mlp, tokenizer, label_token_ids, option_letters
            )
            contrib_mlp = patched_label_logits_mlp[correct_label] - corrupt_correct_logit

            layer_contrib_residual[layer] += contrib_res
            layer_contrib_attn[layer]     += contrib_attn
            layer_contrib_mlp[layer]      += contrib_mlp
            layer_counts[layer]           += 1

        print(f"[{idx+1}/{len(dataset.data)}] baseline_diff={baseline_diff:.3f}")

    # 写 CSV
    csv_path = os.path.join(cfg.out_dir, cfg.csv_layer_roles)
    with open(csv_path, "w") as f:
        f.write("layer,avg_contrib_residual,avg_contrib_attn,avg_contrib_mlp,count\n")
        for layer in range(num_layers):
            c = max(layer_counts[layer], 1)
            f.write(
                f"{layer},"
                f"{layer_contrib_residual[layer]/c},"
                f"{layer_contrib_attn[layer]/c},"
                f"{layer_contrib_mlp[layer]/c},"
                f"{layer_counts[layer]}\n"
            )
    print("Saved layer/component roles to:", csv_path)

    # 画图
    plot_roles(csv_path, cfg.out_dir)

def plot_roles(csv_path: str, out_dir: str):
    df = pd.read_csv(csv_path)

    max_abs = max(
        df["avg_contrib_residual"].abs().max(),
        df["avg_contrib_attn"].abs().max(),
        df["avg_contrib_mlp"].abs().max(),
    )
    df["residual_norm"] = df["avg_contrib_residual"] / max_abs
    df["attn_norm"]     = df["avg_contrib_attn"]     / max_abs
    df["mlp_norm"]      = df["avg_contrib_mlp"]      / max_abs

    sns.set_theme(style="whitegrid", context="talk")

    # line plot
    plt.figure(figsize=(10, 6))
    plt.plot(df["layer"], df["attn_norm"], label="Attention", marker="o")
    plt.plot(df["layer"], df["mlp_norm"], label="MLP", marker="s")
    plt.plot(df["layer"], df["residual_norm"], label="Whole layer (residual)", linestyle="--", alpha=0.7)
    plt.xlabel("Layer index")
    plt.ylabel("Normalized contribution to correct option logit")
    plt.title("Roles of Layer Components (Llama-3.1-8B on CinePile MCQ)")
    plt.legend()
    plt.tight_layout()
    line_path = os.path.join(out_dir, "layer_component_roles_line.png")
    plt.savefig(line_path, dpi=200)
    plt.close()
    print("Saved line plot to:", line_path)

    # stacked bar
    df["total_pos"] = (
        df["avg_contrib_attn"].clip(lower=0) +
        df["avg_contrib_mlp"].clip(lower=0)
    )
    eps = 1e-8
    df["attn_frac"] = df["avg_contrib_attn"].clip(lower=0) / (df["total_pos"] + eps)
    df["mlp_frac"]  = df["avg_contrib_mlp"].clip(lower=0) / (df["total_pos"] + eps)

    plt.figure(figsize=(12, 6))
    plt.bar(df["layer"], df["attn_frac"], label="Attention", color="#1f77b4")
    plt.bar(df["layer"], df["mlp_frac"],  bottom=df["attn_frac"], label="MLP", color="#ff7f0e")
    plt.xlabel("Layer index")
    plt.ylabel("Fraction of positive contribution")
    plt.title("Relative Attention vs MLP Contributions per Layer")
    plt.legend()
    plt.tight_layout()
    stack_path = os.path.join(out_dir, "layer_component_roles_stacked.png")
    plt.savefig(stack_path, dpi=200)
    plt.close()
    print("Saved stacked bar plot to:", stack_path)

# ============================================================
# 8. main
# ============================================================

if __name__ == "__main__":
    run_roles_of_layer_components()

[Dataset] Loaded 10 samples
          answer_key sample : 'From anxiety to excitement' → E
          hard_split sample : True

=== Sanity Check ===
  [0] E → "From anxiety to excitement"
       original: "From anxiety to excitement"  match=True
  [1] E → "Suggests next steps"
       original: "Suggests next steps"  match=True
  [2] D → "The rough foliage"
       original: "The rough foliage"  match=True



ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
# ============================================================
# 诊断代码：直接粘贴到 Colab 运行
# ============================================================
from datasets import load_dataset

dataset = load_dataset('tomg-group-umd/cinepile', split='test')

# Step 1: 查看原始字段
sample = dataset[0]
print("=== 所有字段名 ===")
print(list(sample.keys()))

print("\n=== answer_key 值和类型 ===")
print(repr(sample['answer_key']))
print(type(sample['answer_key']))

print("\n=== choices 值和类型 ===")
print(sample['choices'])
print(type(sample['choices'][0]))

print("\n=== question_category 示例 ===")
print(repr(sample['question_category']))

print("\n=== hard_split 值和类型 ===")
print(repr(sample['hard_split']))
print(type(sample['hard_split']))

# Step 2: 查看前5个样本的 answer_key
print("\n=== 前5个样本的 answer_key ===")
for i in range(5):
    print(f"  [{i}] answer_key = {repr(dataset[i]['answer_key'])}")

# Step 3: 查看所有 answer_key 的唯一值
print("\n=== answer_key 的所有唯一值（前100个样本）===")
unique_keys = set(dataset[i]['answer_key'] for i in range(100))
print(unique_keys)

# Step 4: 查看 question_category 的唯一值
print("\n=== question_category 的唯一值（前100个样本）===")
unique_cats = set(dataset[i]['question_category'] for i in range(100))
for c in sorted(unique_cats):
    print(f"  {repr(c)}")

# Step 5: 手动测试 DeBERTa predict 一次
print("\n=== DeBERTa 手动预测测试 ===")
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base',
    num_labels=1
)
model.eval()

test_sample = dataset[0]
scores = []
scene_q = f"{test_sample['question']} {test_sample['movie_scene'][:512]}"

for i, choice in enumerate(test_sample['choices']):
    inputs = tokenizer(
        scene_q, choice,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True
    )
    with torch.no_grad():
        logit = model(**inputs).logits.squeeze().item()
    scores.append(logit)
    print(f"  Choice {chr(65+i)}: logit = {logit:.4f}")

pred = chr(65 + scores.index(max(scores)))
label = test_sample['answer_key']
print(f"\n  Predicted: {pred}")
print(f"  Label:     {repr(label)}")
print(f"  Match:     {pred == label}")


=== 所有字段名 ===
['movie_name', 'year', 'genre', 'yt_clip_title', 'yt_clip_link', 'movie_scene', 'subtitles', 'question', 'choices', 'answer_key', 'answer_key_position', 'question_category', 'hard_split', 'visual_reliance', 'videoID']

=== answer_key 值和类型 ===
'From anxiety to excitement'
<class 'str'>

=== choices 值和类型 ===
['From despair to hope', 'From fear to acceptance', 'From confusion to understanding', 'From tension to panic', 'From anxiety to excitement']
<class 'str'>

=== question_category 示例 ===
'Theme Exploration'

=== hard_split 值和类型 ===
'True'
<class 'str'>

=== 前5个样本的 answer_key ===
  [0] answer_key = 'From anxiety to excitement'
  [1] answer_key = 'Suggests next steps'
  [2] answer_key = 'The rough foliage'
  [3] answer_key = 'She closes her eyes and focuses on staying calm without moving at all.'
  [4] answer_key = 'Rough and dense'

=== answer_key 的所有唯一值（前100个样本）===
{'Directly underneath', 'Learning about his son', 'In an adjacent room', 'Possibility of meeting his son', 

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight        

  Choice A: logit = -0.1938
  Choice B: logit = -0.1793
  Choice C: logit = -0.1859
  Choice D: logit = -0.1870
  Choice E: logit = -0.1700

  Predicted: E
  Label:     'From anxiety to excitement'
  Match:     False


In [ ]:
from datasets import load_dataset

dataset = load_dataset('tomg-group-umd/cinepile', split='test')

# 查看前20个 answer_key 的原始值
print("=== 前20个 answer_key ===")
for i in range(20):
    ak = dataset[i]['answer_key']
    print(f"  [{i}] type={type(ak).__name__}, value={repr(ak)}")

# 查看所有唯一值
print("\n=== 所有唯一 answer_key 值（完整测试集）===")
from collections import Counter
counter = Counter(str(dataset[i]['answer_key']) for i in range(len(dataset)))
for val, cnt in sorted(counter.items()):
    print(f"  {repr(val):10s} → {cnt} 条")


=== 前20个 answer_key ===
  [0] type=str, value='From anxiety to excitement'
  [1] type=str, value='Suggests next steps'
  [2] type=str, value='The rough foliage'
  [3] type=str, value='She closes her eyes and focuses on staying calm without moving at all.'
  [4] type=str, value='Rough and dense'
  [5] type=str, value='They hear an aircraft approaching.'
  [6] type=str, value='To reach a specific location'
  [7] type=str, value='They become more confident'
  [8] type=str, value='The sound of a helicopter approaching'
  [9] type=str, value='Reed'
  [10] type=str, value='The sound of a helicopter'
  [11] type=str, value='In a rocky plateau'
  [12] type=str, value='It is a valley with spots of light'
  [13] type=str, value='They use a thermal suit'
  [14] type=str, value='They are trying to remain undetected by the helicopter using thermal suits.'
  [15] type=str, value='It illuminates the surrounding foliage.'
  [16] type=str, value='Directly underneath'
  [17] type=str, value='Colleagues'

In [ ]:
# 运行这段诊断，查看实际的 question_category 值分布
from datasets import load_dataset
from collections import Counter

# dataset = load_dataset('tomg-group-umd/cinepile', split='test')

counter = Counter(dataset[i]['question_category'] for i in range(200))
print("=== question_category 实际值分布 ===")
for val, cnt in sorted(counter.items(), key=lambda x: -x[1]):
    print(f"  {cnt:4d}  {repr(val)}")


=== question_category 实际值分布 ===
    88  'Character and\nRelationship Dynamics'
    63  'Setting and\nTechnical Analysis'
    24  'Temporal'
    13  'Narrative and\nPlot Analysis'
    12  'Theme Exploration'


In [ ]:
# 快速验证（无需重新加载数据集）
test_categories = [
    'Character and\nRelationship Dynamics',
    'Setting and\nTechnical Analysis',
    'Temporal',
    'Narrative and\nPlot Analysis',
    'Theme Exploration',
]

category_map = {
    'TEMP': 'Temporal',
    'CRD':  'Character and',
    'NPA':  'Narrative and',
    'STA':  'Setting and',
    'TH':   'Theme Exploration',
}

import pandas as pd
df_test = pd.DataFrame({'question_category': test_categories})

print("=== 映射验证 ===")
for abbr, keyword in category_map.items():
    mask = df_test['question_category'].str.contains(keyword, case=False, na=False)
    matched = df_test.loc[mask, 'question_category'].tolist()
    print(f"  {abbr} ({keyword!r:20s}) → {matched}")


=== 映射验证 ===
  TEMP ('Temporal'          ) → ['Temporal']
  CRD ('Character and'     ) → ['Character and\nRelationship Dynamics']
  NPA ('Narrative and'     ) → ['Narrative and\nPlot Analysis']
  STA ('Setting and'       ) → ['Setting and\nTechnical Analysis']
  TH ('Theme Exploration' ) → ['Theme Exploration']
